In [1]:

# OBJECTIF :
# Construire un dataset NHS multi-années (2012-2025) pour 
# entraîner un modèle XGBoost capable de prédire le statut
# opérationnel des hôpitaux (Critique / Attention / Normal)
#
# SOURCES DE DONNÉES :
# 1. NHS Beds (KH03)      → C2 Occupation, A8 Lits vacants
# 2. SHMI                 → C3 Mortalité
# 3. CDI                  → C4 Infections nosocomiales
# 4. Hospital Providers   → A2 DMS, A9 Transferts
# 5. AEQI                 → A4 Temps d'attente
# 6. NHS Workforce        → Ratios médecins/infirmiers/lits
#
# ANNÉES COUVERTES : 2012-13 → 2024-25 (sans 2020-21 COVID)
# TOTAL VISÉ       : ~1200 trusts NHS
# ============================================================

import pandas as pd
import numpy as np
import os
import zipfile
import warnings
warnings.filterwarnings('ignore')

print("✅ Librairies importées")
print(f"   Pandas  : {pd.__version__}")
print(f"   NumPy   : {np.__version__}")

DATA = r'C:\Users\hp\healthsim-simulation\data'
OUTPUT = r'C:\Users\hp\healthsim-simulation\data\datasets'

print(f"\n✅ Dossier données : {DATA}")
print(f"✅ Dossier sortie  : {OUTPUT}")

C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


✅ Librairies importées
   Pandas  : 3.0.3
   NumPy   : 1.26.4

✅ Dossier données : C:\Users\hp\healthsim-simulation\data
✅ Dossier sortie  : C:\Users\hp\healthsim-simulation\data\datasets


In [3]:
# ============================================================
# SECTION 2 — EXPLORATION FICHIER NHS BEDS (KH03)
# ============================================================
# Le fichier Beds contient le nombre de lits disponibles 
# et occupés par trust NHS. Publié trimestriellement.
# On utilise Q4 (Janvier-Mars) = fin d'année fiscale NHS.
#
# KPIs calculés depuis ce fichier :
#   C2 = lits_occupes / lits_disponibles × 100
#   A8 = (lits_disponibles - lits_occupes) / lits_disponibles × 100
# ============================================================

print("=" * 60)
print("EXPLORATION BEDS — Comparaison formats par année")
print("=" * 60)

# ── Format ancien (2012-13) ──────────────────────────────
print("\n📁 FORMAT ANCIEN — Beds 2012-13 (xls)")
df_beds_old = pd.read_excel(
    rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2012-13-Revised-Final-79659.xls',
    sheet_name='NHS Trust by Sector',
    header=None
)
print(f"   Shape    : {df_beds_old.shape}")
print(f"   Onglet   : 'NHS Trust by Sector'")
print(f"   Headers  : ligne 14")
print(f"   Trust code : colonne 4 (Org Code)")
print(f"   Lits disponibles : colonne 6 (Total Available)")
print(f"   Lits occupés     : colonne 12 (Total Occupied)")
print(f"\n   Exemple données :")
print(df_beds_old.iloc[17:20, [4,5,6,12]].to_string())

# ── Format récent (2022-23) ──────────────────────────────
print("\n📁 FORMAT RÉCENT — Beds 2022-23 (xlsx)")
df_beds_new = pd.read_excel(
    rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2022-23-Final.xlsx',
    sheet_name=None
)
print(f"   Onglets disponibles : {list(df_beds_new.keys())}")

EXPLORATION BEDS — Comparaison formats par année

📁 FORMAT ANCIEN — Beds 2012-13 (xls)
   Shape    : (249, 23)
   Onglet   : 'NHS Trust by Sector'
   Headers  : ligne 14
   Trust code : colonne 4 (Org Code)
   Lits disponibles : colonne 6 (Total Available)
   Lits occupés     : colonne 12 (Total Occupied)

   Exemple données :
     4                                               5           6           12
17  RE9             SOUTH TYNESIDE NHS FOUNDATION TRUST         415  356.777778
18  RLN  CITY HOSPITALS SUNDERLAND NHS FOUNDATION TRUST  887.122222  685.488889
19  RR7           GATESHEAD HEALTH NHS FOUNDATION TRUST         517  459.888889

📁 FORMAT RÉCENT — Beds 2022-23 (xlsx)
   Onglets disponibles : ['NHS Trust by Sector', 'Region by Sector', 'Occupied by Specialty', 'Data Quality']


In [5]:
# ── Format récent suite ──────────────────────────────────
df_beds_new_sheet = pd.read_excel(
    rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2022-23-Final.xlsx',
    sheet_name='NHS Trust by Sector',
    header=None
)
print(f"   Shape    : {df_beds_new_sheet.shape}")

# Chercher la ligne des headers
for i in range(10, 20):
    row = df_beds_new_sheet.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 2:
        print(f"   Headers ligne {i} : {non_nan[:8]}")
        header_row = i
        break

print(f"\n   Exemple données ligne {header_row+3} :")
print(df_beds_new_sheet.iloc[header_row+2:header_row+5, :10].to_string())

print("\n✅ CONCLUSION BEDS :")
print("   Format identique pour toutes les années !")
print("   Onglet    : 'NHS Trust by Sector'")
print("   Headers   : ligne 14")
print("   Trust code : Col 4 (Org Code)")
print("   Lits dispo : Col 6 (Total Available)")
print("   Lits occup : Col 12 (Total Occupied)")

   Shape    : (206, 23)
   Headers ligne 13 : ['Available', 'Occupied', '% Occupied']

   Exemple données ligne 16 :
     0        1      2    3    4                       5              6              7           8            9
15 NaN  2022-23  March  NaN  NaN                 England  130863.066667  104542.988889  767.744444  7520.533333
16 NaN      NaN    NaN  NaN  NaN                     NaN            NaN            NaN         NaN          NaN
17 NaN  2022-23  March  Y56  R1H  BARTS HEALTH NHS TRUST         2024.4    1836.622222           0   187.777778

✅ CONCLUSION BEDS :
   Format identique pour toutes les années !
   Onglet    : 'NHS Trust by Sector'
   Headers   : ligne 14
   Trust code : Col 4 (Org Code)
   Lits dispo : Col 6 (Total Available)
   Lits occup : Col 12 (Total Occupied)


In [7]:
# ============================================================
# SECTION 3 — EXPLORATION FICHIER SHMI
# ============================================================
# SHMI = Summary Hospital-level Mortality Indicator
# Publié annuellement par NHS Digital.
# Contient le ratio de mortalité par trust NHS.
#
# KPI calculé depuis ce fichier :
#   C3 = OBSERVED / SPELLS × 100  (taux mortalité réel)
#
# Référence : WHO (2015). Global Reference List of 
#             100 Core Health Indicators.
# ============================================================

print("=" * 60)
print("EXPLORATION SHMI — Structure et colonnes")
print("=" * 60)

# Extraire le zip SHMI 2022-23
import zipfile, os

shmi_zip = rf'{DATA}\SHMI data files, Apr22-Mar23.zip'
shmi_extract = rf'{DATA}\SHMI_temp'

os.makedirs(shmi_extract, exist_ok=True)

with zipfile.ZipFile(shmi_zip, 'r') as z:
    print(f"\n📁 Fichiers dans le zip SHMI 2022-23 :")
    for f in z.namelist():
        print(f"   - {f}")
    z.extractall(shmi_extract)

EXPLORATION SHMI — Structure et colonnes

📁 Fichiers dans le zip SHMI 2022-23 :
   - SHMI contextual indicators/
   - SHMI contextual indicators/Admission method/
   - SHMI contextual indicators/Admission method/Deaths within 30 days for elective admissions, Apr22-Mar23 (csv).csv
   - SHMI contextual indicators/Admission method/Deaths within 30 days for elective admissions, Apr22-Mar23 (xls).xls
   - SHMI contextual indicators/Admission method/Deaths within 30 days for non-elective admissions, Apr22-Mar23 (csv).csv
   - SHMI contextual indicators/Admission method/Deaths within 30 days for non-elective admissions, Apr22-Mar23 (xls).xls
   - SHMI contextual indicators/COVID-19 activity/
   - SHMI contextual indicators/COVID-19 activity/Percentage of provider spells with COVID-19 coding, Apr22-Mar23 (csv).csv
   - SHMI contextual indicators/COVID-19 activity/Percentage of provider spells with COVID-19 coding, Apr22-Mar23 (xls).xlsx
   - SHMI contextual indicators/COVID-19 activity/Provide

In [9]:
# Lire le fichier SHMI trust level
shmi_file = rf'{shmi_extract}\SHMI data\SHMI data at trust level, Apr22-Mar23 (csv).csv'
df_shmi = pd.read_csv(shmi_file)

print(f"\n📊 Structure SHMI 2022-23 :")
print(f"   Shape   : {df_shmi.shape}")
print(f"\n   Colonnes :")
for col in df_shmi.columns:
    print(f"   - {col}")

print(f"\n   Exemple données (3 premières lignes) :")
print(df_shmi.head(3).to_string())


📊 Structure SHMI 2022-23 :
   Shape   : (120, 12)

   Colonnes :
   - INDICATOR_CODE
   - PROVIDER_CODE
   - PROVIDER_NAME
   - SHMI_VALUE
   - SHMI_BANDING
   - SPELLS
   - OBSERVED
   - EXPECTED
   - PO_LL
   - PO_UL
   - OD_LL
   - OD_UL

   Exemple données (3 premières lignes) :
  INDICATOR_CODE PROVIDER_CODE                                                   PROVIDER_NAME  SHMI_VALUE  SHMI_BANDING  SPELLS  OBSERVED  EXPECTED   PO_LL   PO_UL   OD_LL   OD_UL
0         I00699           RCF                                   AIREDALE NHS FOUNDATION TRUST      0.8886             2   31155       910      1025  0.9062  1.1004  0.8835  1.1318
1         I00699           RTK           ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATION TRUST      0.8852             3   57945      1645      1855  0.9298  1.0738  0.8897  1.1240
2         I00699           RF4  BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOSPITALS NHS TRUST      0.9508             2   82420      2525      2660  0.9411  1.0614  0.8921

In [11]:
print("✅ CONCLUSION SHMI :")
print("   Fichier cible : 'SHMI data at trust level (csv).csv'")
print("   Colonnes clés :")
print("     - PROVIDER_CODE → Trust code")
print("     - OBSERVED      → Décès réels")
print("     - SPELLS        → Total admissions")
print("     - SHMI_VALUE    → Ratio mortalité")
print("     - SHMI_BANDING  → 1=élevé, 2=normal, 3=faible")
print(f"\n   Formule C3 = OBSERVED / SPELLS × 100")
print(f"\n   Exemple Airedale : {910/31155*100:.3f}%")

# Nettoyage temp
import shutil
shutil.rmtree(shmi_extract)
print("\n✅ Dossier temporaire nettoyé")

✅ CONCLUSION SHMI :
   Fichier cible : 'SHMI data at trust level (csv).csv'
   Colonnes clés :
     - PROVIDER_CODE → Trust code
     - OBSERVED      → Décès réels
     - SPELLS        → Total admissions
     - SHMI_VALUE    → Ratio mortalité
     - SHMI_BANDING  → 1=élevé, 2=normal, 3=faible

   Formule C3 = OBSERVED / SPELLS × 100

   Exemple Airedale : 2.921%

✅ Dossier temporaire nettoyé


In [13]:
# ============================================================
# SECTION 4 — EXPLORATION FICHIER CDI
# ============================================================
# CDI = Clostridioides difficile Infection
# Publié mensuellement par UK Health Security Agency.
# On utilise le fichier couvrant Avril → Mars (année fiscale).
#
# KPI calculé depuis ce fichier :
#   C4 = total_cas_CDI_annee / total_admissions × 100
#
# Référence : ECDC (2013). Point prevalence survey of HAI. p.22
# ============================================================

print("=" * 60)
print("EXPLORATION CDI — Structure et colonnes")
print("=" * 60)

# ── Format récent (2022-23) ──────────────────────────────
print("\n📁 FORMAT RÉCENT — CDI 2022-23 (ods)")
import subprocess
# Lire fichier ODS
df_cdi_new = pd.read_excel(
    rf'{DATA}\Table-6-CDI-march-2023.ods',
    engine='odf',
    sheet_name=None
)
print(f"   Onglets disponibles :")
for s in df_cdi_new.keys():
    print(f"   - {s}")

EXPLORATION CDI — Structure et colonnes

📁 FORMAT RÉCENT — CDI 2022-23 (ods)
   Onglets disponibles :
   - caveats
   - monthly_table
   - data


In [15]:
# Voir structure onglet 'data'
df_cdi_data = df_cdi_new['data']
print(f"\n   Shape : {df_cdi_data.shape}")
print(f"\n   Colonnes :")
for col in df_cdi_data.columns:
    print(f"   - {col}")
print(f"\n   Exemple données (3 premières lignes) :")
print(df_cdi_data.head(3).to_string())


   Shape : (20412, 13)

   Colonnes :
   - Collection
   - Organisation type
   - Organisation code
   - Organisation name
   - UKHSA Centre
   - ICB
   - Region name
   - Year
   - Month
   - Metric
   - Figure
   - Data signed off
   - Data changed since last publication?

   Exemple données (3 premières lignes) :
     Collection Organisation type Organisation code                           Organisation name UKHSA Centre                                                     ICB               Region name  Year  Month                                  Metric  Figure Data signed off  Data changed since last publication?
0  C. difficile         Sub - ICB               00L  NHS NORTH EAST AND NORTH CUMBRIA ICB - 00L   North East  NHS North East and North Cumbria Integrated Care Board  North East and Yorkshire  2022     11                                 Unknown       0             NaN                                   NaN
1  C. difficile         Sub - ICB               00L  NHS NORTH EAST A

In [17]:
# Filtrer pour avoir les trusts NHS seulement
# et voir les métriques disponibles
print(f"   Types organisations : {df_cdi_data['Organisation type'].unique()}")
print(f"\n   Métriques disponibles :")
print(df_cdi_data['Metric'].unique())

# Filtrer NHS Trusts
df_cdi_trusts = df_cdi_data[df_cdi_data['Organisation type'] == 'NHS Trust']
print(f"\n   NHS Trusts seulement : {len(df_cdi_trusts)} lignes")
print(f"   Nombre trusts uniques : {df_cdi_trusts['Organisation code'].nunique()}")

# Total cas par trust sur l'année
df_cdi_total = df_cdi_trusts[
    df_cdi_trusts['Metric'] == 'Total'
].groupby('Organisation code')['Figure'].sum().reset_index()
df_cdi_total.columns = ['Trust_code', 'CDI_total_cases']
print(f"\n   Exemple CDI total par trust :")
print(df_cdi_total.head(5).to_string())

print("\n✅ CONCLUSION CDI :")
print("   Onglet cible  : 'data'")
print("   Filtre        : Organisation type = 'NHS Trust'")
print("   Métrique      : 'Total'")
print("   Formule C4    : CDI_total / total_admissions × 100")

   Types organisations : <ArrowStringArray>
['Sub - ICB', 'NHS acute trust', 'National']
Length: 3, dtype: str

   Métriques disponibles :
<ArrowStringArray>
[                                   'Unknown',
                             'No information',
     'Community-onset, healthcare associated',
      'Community-onset, community associated',
 'Community-onset, indeterminate association',
      'Hospital-onset, healthcare associated',
                                'Total cases']
Length: 7, dtype: str

   NHS Trusts seulement : 0 lignes
   Nombre trusts uniques : 0

   Exemple CDI total par trust :
Empty DataFrame
Columns: [Trust_code, CDI_total_cases]
Index: []

✅ CONCLUSION CDI :
   Onglet cible  : 'data'
   Filtre        : Organisation type = 'NHS Trust'
   Métrique      : 'Total'
   Formule C4    : CDI_total / total_admissions × 100


In [19]:
# Correction — filtre correct
df_cdi_trusts = df_cdi_data[
    df_cdi_data['Organisation type'] == 'NHS acute trust'
]
print(f"   NHS acute trusts : {len(df_cdi_trusts)} lignes")
print(f"   Trusts uniques   : {df_cdi_trusts['Organisation code'].nunique()}")

# Total cas par trust sur l'année
df_cdi_total = df_cdi_trusts[
    df_cdi_trusts['Metric'] == 'Total cases'
].groupby('Organisation code')['Figure'].sum().reset_index()
df_cdi_total.columns = ['Trust_code', 'CDI_total_cases']

print(f"\n   Exemple CDI total par trust :")
print(df_cdi_total.head(5).to_string())
print(f"\n   Total trusts avec CDI : {len(df_cdi_total)}")

print("\n✅ CONCLUSION CDI CORRIGÉE :")
print("   Filtre   : Organisation type = 'NHS acute trust'")
print("   Métrique : 'Total cases'")
print("   Formule C4 = CDI_total_cases / total_admissions × 100")

   NHS acute trusts : 11340 lignes
   Trusts uniques   : 135

   Exemple CDI total par trust :
  Trust_code  CDI_total_cases
0        R0A              251
1        R0B              125
2        R0D              154
3        R1F               62
4        R1H              160

   Total trusts avec CDI : 135

✅ CONCLUSION CDI CORRIGÉE :
   Filtre   : Organisation type = 'NHS acute trust'
   Métrique : 'Total cases'
   Formule C4 = CDI_total_cases / total_admissions × 100


In [23]:
# ============================================================
# SECTION 5 — EXPLORATION HOSPITAL PROVIDERS
# ============================================================
# Publié annuellement par NHS Digital.
# Contient les statistiques d'activité hospitalière par trust.
#
# KPIs calculés depuis ce fichier :
#   A2 = Mean length of stay (DMS en jours)
#   A9 = Other_discharge / Total_admissions × 100
#
# Référence : NHS Digital, Hospital Admitted Patient Care
#             Activity, 2022-23
# ============================================================

print("=" * 60)
print("EXPLORATION HOSPITAL PROVIDERS — Comparaison formats")
print("=" * 60)

# ── Format récent (2022-23) ──────────────────────────────
print("\n📁 FORMAT RÉCENT — Hospital Providers 2022-23")
df_prov_new = pd.read_excel(
    rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2022-23-tab_v2.xlsx',
    sheet_name=None
)
print(f"   Onglets : {list(df_prov_new.keys())}")

# Lire l'onglet correct
df_prov_sheet = pd.read_excel(
    rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2022-23-tab_v2.xlsx',
    sheet_name='Hospital Providers',
    header=None
)
print(f"   Shape : {df_prov_sheet.shape}")

# Chercher headers
for i in range(5, 20):
    row = df_prov_sheet.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 3:
        print(f"   Headers ligne {i} : {non_nan[:6]}")
        break

EXPLORATION HOSPITAL PROVIDERS — Comparaison formats

📁 FORMAT RÉCENT — Hospital Providers 2022-23
   Onglets : ['Hospital Providers']
   Shape : (531, 50)
   Headers ligne 16 : ['Hospital provider code and description†', 'Finished consultant episodes', 'Admissions', 'Male', 'Female', 'Gender Unknown']


In [25]:
# Voir toutes les colonnes importantes
print(f"\n   Colonnes importantes ligne 16 :")
for i, val in enumerate(df_prov_sheet.iloc[16, :].tolist()):
    if str(val) != 'nan':
        print(f"   Col {i} : {val}")

# Voir données trusts (code 3 caractères)
print(f"\n   Exemple données trusts :")
for i in range(18, 40):
    row = df_prov_sheet.iloc[i, :].tolist()
    code = str(row[1]) if len(df_prov_sheet.columns) > 1 else ''
    if len(code) == 3 and code.isalnum():
        print(f"   Ligne {i} : code={row[1]}, admissions={row[9]}, DMS={row[19]}, other={row[50-1] if len(row)>49 else 'N/A'}")
        if i > 25:
            break

# ── Format ancien (2017-18) ──────────────────────────────
print("\n📁 FORMAT ANCIEN — Hospital Providers 2017-18")
df_prov_old = pd.read_excel(
    rf'{DATA}\hosp-epis-stat-admi-prov-2017-18-tab.xlsx',
    sheet_name=None
)
print(f"   Onglets : {list(df_prov_old.keys())}")


   Colonnes importantes ligne 16 :
   Col 0 : Hospital provider code and description†
   Col 7 : Finished consultant episodes
   Col 8 : Admissions
   Col 9 : Male
   Col 10 : Female
   Col 11 : Gender Unknown
   Col 12 : Emergency
   Col 13 : Waiting list
   Col 14 : Planned
   Col 15 : Other Admission Method
   Col 16 : Mean time waited
   Col 17 : Median time waited
   Col 18 : Mean length of stay
   Col 19 : Median length of stay
   Col 20 : Mean age
   Col 21 : Age 0
   Col 22 : Age 1-4
   Col 23 : Age 5-9
   Col 24 : Age 10-14
   Col 25 : Age 15
   Col 26 : Age 16
   Col 27 : Age 17
   Col 28 : Age 18
   Col 29 : Age 19
   Col 30 : Age 20-24
   Col 31 : Age 25-29
   Col 32 : Age 30-34
   Col 33 : Age 35-39
   Col 34 : Age 40-44
   Col 35 : Age 45-49
   Col 36 : Age 50-54
   Col 37 : Age 55-59
   Col 38 : Age 60-64
   Col 39 : Age 65-69
   Col 40 : Age 70-74
   Col 41 : Age 75-79
   Col 42 : Age 80-84
   Col 43 : Age 85-89
   Col 44 : Age 90+
   Col 45 : Day case
   Col 46 : FCE 

In [27]:
# Chercher les trusts avec code 3 caractères
print(f"\n   Cherche trusts dans 2022-23 :")
for i in range(17, 100):
    row = df_prov_sheet.iloc[i, :].tolist()
    code = str(row[0])
    if len(code) == 3 and code[0].isalpha():
        print(f"   Ligne {i} : code={row[0]}, admissions={row[8]}, DMS={row[18]}, other={row[49]}")
        if i > 25:
            break

# Format ancien 2017-18
df_prov_old_sheet = pd.read_excel(
    rf'{DATA}\hosp-epis-stat-admi-prov-2017-18-tab.xlsx',
    sheet_name='Hospital Providers',
    header=None
)
print(f"\n📁 FORMAT ANCIEN — Hospital Providers 2017-18")
print(f"   Shape : {df_prov_old_sheet.shape}")

# Chercher headers
for i in range(5, 20):
    row = df_prov_old_sheet.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 3:
        print(f"   Headers ligne {i} : {non_nan[:6]}")
        break


   Cherche trusts dans 2022-23 :
   Ligne 17 : code=nan, admissions=nan, DMS=nan, other=nan
   Ligne 19 : code=nan, admissions=nan, DMS=nan, other=nan
   Ligne 20 : code=Y56, admissions=2525500, DMS=5.269912, other=102435
   Ligne 21 : code=Y58, admissions=1683625, DMS=5.052307, other=53675
   Ligne 22 : code=Y59, admissions=2179010, DMS=4.797372, other=47790
   Ligne 23 : code=Y60, admissions=3216020, DMS=4.526896, other=138130
   Ligne 24 : code=Y61, admissions=1705645, DMS=5.02496, other=57095
   Ligne 25 : code=Y62, admissions=2284245, DMS=4.753217, other=89330
   Ligne 26 : code=Y63, admissions=2805100, DMS=4.703014, other=83875

📁 FORMAT ANCIEN — Hospital Providers 2017-18
   Shape : (519, 50)
   Headers ligne 9 : ['Hospital provider code and description†', 'Finished consultant episodes', 'Admissions', 'Male', 'Female', 'Gender Unknown']


In [29]:
# Chercher trusts 3 lettres dans 2022-23
print(f"   Trusts réels dans 2022-23 (code 3 lettres) :")
count = 0
for i in range(17, 531):
    row = df_prov_sheet.iloc[i, :].tolist()
    code = str(row[0])
    if len(code) == 3 and code[0].isalpha() and not code.startswith('Y'):
        print(f"   Ligne {i} : code={row[0]}, admissions={row[8]}, DMS={row[18]}, other={row[49]}")
        count += 1
        if count >= 3:
            break

# Colonnes importantes 2017-18
print(f"\n   Colonnes importantes ligne 9 (2017-18) :")
for i, val in enumerate(df_prov_old_sheet.iloc[9, :].tolist()):
    if str(val) != 'nan':
        print(f"   Col {i} : {val}")
        if i > 20:
            break

   Trusts réels dans 2022-23 (code 3 lettres) :
   Ligne 17 : code=nan, admissions=nan, DMS=nan, other=nan
   Ligne 19 : code=nan, admissions=nan, DMS=nan, other=nan
   Ligne 27 : code=nan, admissions=nan, DMS=nan, other=nan

   Colonnes importantes ligne 9 (2017-18) :
   Col 0 : Hospital provider code and description†
   Col 7 : Finished consultant episodes
   Col 8 : Admissions
   Col 9 : Male
   Col 10 : Female
   Col 11 : Gender Unknown
   Col 12 : Emergency
   Col 13 : Waiting list
   Col 14 : Planned
   Col 15 : Other Admission Method
   Col 16 : Mean time waited
   Col 17 : Median time waited
   Col 18 : Mean length of stay
   Col 19 : Median length of stay
   Col 20 : Mean age
   Col 21 : Age 0


In [31]:
# Voir exactement les lignes autour des trusts
print("Lignes 27-40 colonnes 0-5 :")
print(df_prov_sheet.iloc[27:40, :6].to_string())

print("\nLignes 10-20 colonnes 0-5 (2017-18) :")
print(df_prov_old_sheet.iloc[10:20, :6].to_string())

Lignes 27-40 colonnes 0-5 :
        0                                                                                         1   2   3   4   5
27    NaN                                                                                       NaN NaN NaN NaN NaN
28  8A718                                                                             LONDON CLINIC NaN NaN NaN NaN
29  A0C5S                                                                      SPAMEDICA GLOUCESTER NaN NaN NaN NaN
30  A4M8P                                                                         BUCKSHAW HOSPITAL NaN NaN NaN NaN
31  AAH01                                                                    TETBURY HOSPITAL TRUST NaN NaN NaN NaN
32  AAV02                                            COMMUNITY HEALTH AND EYECARE LIMITED (PRESTON) NaN NaN NaN NaN
33  ACG07  NEWMEDICA COMMUNITY OPHTHALMOLOGY - LEICESTERSHIRE - LEICESTER (STONEYGATE EYE HOSPITAL) NaN NaN NaN NaN
34  ACG09                                   

In [33]:
# Conclusion Hospital Providers
print("✅ CONCLUSION HOSPITAL PROVIDERS :")
print("\n   FORMAT 2022-23 et récents :")
print("   - Trust code : Col 0 (code seul)")
print("   - Trust name : Col 1")  
print("   - Admissions : Col 8")
print("   - DMS (A2)   : Col 18")
print("   - Other disc : Col 49 → A9")
print("   - Données trusts commencent ligne ~28")
print("   - Filtrer codes NHS : R** (3 caractères commençant par R)")

print("\n   FORMAT 2017-18 et anciens :")
print("   - Même structure colonnes !")
print("   - Trust code : Col 0")
print("   - Données trusts commencent ligne ~18")
print("   - Régions : Y54, Y55, Y56, Y57 → à exclure")

# Vérifier les trusts NHS (code commençant par R)
print("\n   Exemple trusts NHS 2022-23 :")
count = 0
for i in range(27, 531):
    row = df_prov_sheet.iloc[i, :].tolist()
    code = str(row[0])
    if len(code) == 3 and code.startswith('R'):
        print(f"   code={code}, admissions={row[8]}, DMS={row[18]}, other={row[49]}")
        count += 1
        if count >= 5:
            break

print("\n   Exemple trusts NHS 2017-18 :")
count = 0
for i in range(18, 519):
    row = df_prov_old_sheet.iloc[i, :].tolist()
    code = str(row[0])
    if len(code) == 3 and code.startswith('R'):
        print(f"   code={code}, admissions={row[8]}, DMS={row[18]}, other={row[49]}")
        count += 1
        if count >= 5:
            break

✅ CONCLUSION HOSPITAL PROVIDERS :

   FORMAT 2022-23 et récents :
   - Trust code : Col 0 (code seul)
   - Trust name : Col 1
   - Admissions : Col 8
   - DMS (A2)   : Col 18
   - Other disc : Col 49 → A9
   - Données trusts commencent ligne ~28
   - Filtrer codes NHS : R** (3 caractères commençant par R)

   FORMAT 2017-18 et anciens :
   - Même structure colonnes !
   - Trust code : Col 0
   - Données trusts commencent ligne ~18
   - Régions : Y54, Y55, Y56, Y57 → à exclure

   Exemple trusts NHS 2022-23 :
   code=R0B, admissions=144680, DMS=4.570831, other=1460
   code=R0D, admissions=169010, DMS=4.249979, other=1545
   code=R1A, admissions=6230, DMS=22.53675, other=345
   code=R1C, admissions=1285, DMS=26.767013, other=10
   code=R1D, admissions=2320, DMS=15.771739, other=20

   Exemple trusts NHS 2017-18 :
   code=RTQ, admissions=1365, DMS=*, other=0
   code=REM, admissions=94720, DMS=4.868592974203905, other=70
   code=RCF, admissions=61775, DMS=3.639424565749838, other=2055
   c

In [35]:
# ============================================================
# SECTION 6 — EXPLORATION FICHIER AEQI
# ============================================================
# AEQI = Accident & Emergency Quality Indicators
# Publié mensuellement par NHS Digital.
# On utilise Mars = fin d'année fiscale NHS.
#
# KPI calculé depuis ce fichier :
#   A4 = temps médian d'attente aux urgences (minutes)
#
# Référence : NHS England, A&E Quality Indicators
# ============================================================

print("=" * 60)
print("EXPLORATION AEQI — Comparaison formats par année")
print("=" * 60)

# ── Format récent (2022-23) ──────────────────────────────
print("\n📁 FORMAT RÉCENT — AEQI 2022-23 (csv)")
df_aeqi_new = pd.read_csv(rf'{DATA}\aeqi_open_data_2023_03.csv')
print(f"   Shape   : {df_aeqi_new.shape}")
print(f"   Colonnes : {list(df_aeqi_new.columns)}")
print(f"\n   Métriques disponibles :")
print(df_aeqi_new['MEASURE_NAME'].unique() if 'MEASURE_NAME' in df_aeqi_new.columns else "colonne MEASURE_NAME absente")
print(f"\n   Exemple données :")
print(df_aeqi_new.head(3).to_string())

EXPLORATION AEQI — Comparaison formats par année

📁 FORMAT RÉCENT — AEQI 2022-23 (csv)
   Shape   : (95850, 7)
   Colonnes : ['ATTENDANCE_MONTH', 'ORG_CODE', 'ORG_NAME', 'MEASURE_ID', 'MEASURE_NAME', 'MEASURE_VALUE', 'SUPPRESSION']

   Métriques disponibles :
<ArrowStringArray>
[       'TOTAL_TIME_MEDIAN_ADMITTED',            'TOTAL_TIME_95_ADMITTED',
         'TOTAL_TIME_DENOM_ADMITTED',         'TOTAL_TIME_MEDIAN_NON_ADM',
             'TOTAL_TIME_95_NON_ADM',          'TOTAL_TIME_DENOM_NON_ADM',
                 'TOTAL_TIME_MEDIAN',                     'TOTAL_TIME_95',
                  'TOTAL_TIME_DENOM',          'TIME_TO_TREATMENT_MEDIAN',
              'TIME_TO_TREATMENT_95',           'TIME_TO_TREATMENT_DENOM',
 'TIME_TO_INITIAL_ASSESSMENT_MEDIAN',     'TIME_TO_INITIAL_ASSESSMENT_95',
  'TIME_TO_INITIAL_ASSESSMENT_DENOM',           'UNPLANNED_REATTENDANCES',
      'UNPLANNED_REATTENDANCES_RATE',     'UNPLANNED_REATTENDANCES_DENOM',
             'LEFT_DEPARTMENT_TOTAL',         

In [37]:
# Filtrer métrique TOTAL_TIME_MEDIAN pour Mars 2023
df_aeqi_filtered = df_aeqi_new[
    (df_aeqi_new['MEASURE_NAME'] == 'TOTAL_TIME_MEDIAN') &
    (df_aeqi_new['ATTENDANCE_MONTH'] == '2023-03') &
    (df_aeqi_new['ORG_CODE'] != 'ENG')
]
print(f"   Trusts avec TOTAL_TIME_MEDIAN : {len(df_aeqi_filtered)}")
print(f"\n   Exemple :")
print(df_aeqi_filtered.head(5)[['ORG_CODE', 'ORG_NAME', 'MEASURE_VALUE']].to_string())

# ── Format ancien (2016-17) ──────────────────────────────
print("\n📁 FORMAT ANCIEN — AEQI 2016-17 (xlsx)")
df_aeqi_old = pd.read_excel(
    rf'{DATA}\ae_cqi_2016_17_m131.xlsx',
    sheet_name=None
)
print(f"   Onglets : {list(df_aeqi_old.keys())}")

# ── Format très ancien (2012-13) ──────────────────────────
print("\n📁 FORMAT TRÈS ANCIEN — AEQI 2012-13 (xls)")
df_aeqi_very_old = pd.read_excel(
    rf'{DATA}\prov-ae-qual-indi-eng-mar13-tab1.xls',
    sheet_name=None
)
print(f"   Onglets : {list(df_aeqi_very_old.keys())}")

   Trusts avec TOTAL_TIME_MEDIAN : 174

   Exemple :
    ORG_CODE                        ORG_NAME  MEASURE_VALUE
34     AD903              URGENT CARE CENTRE           87.0
51     AD913            BECKENHAM BEACON UCC          136.0
68     AD914  UCC - QUEEN ELIZABETH HOSPITAL          117.0
85       ANH                  MALLING HEALTH           85.0
108    AQN04               PHL LYMINGTON UTC           80.0

📁 FORMAT ANCIEN — AEQI 2016-17 (xlsx)
   Onglets : ['HESF_AE_CQI_dev_NHSdigital']

📁 FORMAT TRÈS ANCIEN — AEQI 2012-13 (xls)
   Onglets : ['Data']


In [39]:
# Explorer format ancien 2016-17
df_aeqi_old_sheet = pd.read_excel(
    rf'{DATA}\ae_cqi_2016_17_m131.xlsx',
    sheet_name='HESF_AE_CQI_dev_NHSdigital',
    header=None
)
print(f"   Shape 2016-17 : {df_aeqi_old_sheet.shape}")
print(f"\n   10 premières lignes colonnes 0-6 :")
print(df_aeqi_old_sheet.iloc[:10, :7].to_string())

# Explorer format très ancien 2012-13
df_aeqi_very_old_sheet = pd.read_excel(
    rf'{DATA}\prov-ae-qual-indi-eng-mar13-tab1.xls',
    sheet_name='Data',
    header=None
)
print(f"\n   Shape 2012-13 : {df_aeqi_very_old_sheet.shape}")
print(f"\n   10 premières lignes colonnes 0-6 :")
print(df_aeqi_very_old_sheet.iloc[:10, :7].to_string())

   Shape 2016-17 : (259, 50)

   10 premières lignes colonnes 0-6 :
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                0                                      1                                                                                                                                                 2                                                                              3                                                           

In [41]:
# Chercher colonne temps médian dans 2016-17
print("Colonnes ligne 4 (2016-17) :")
for i, val in enumerate(df_aeqi_old_sheet.iloc[4, :].tolist()):
    if str(val) != 'nan' and 'median' in str(val).lower():
        print(f"  Col {i} : {val}")

print("\nToutes colonnes ligne 4 (2016-17) :")
for i, val in enumerate(df_aeqi_old_sheet.iloc[4, :].tolist()):
    if str(val) != 'nan':
        print(f"  Col {i} : {str(val)[:80]}")

# Chercher colonne temps médian dans 2012-13
print("\nColonnes ligne 9 (2012-13) :")
for i, val in enumerate(df_aeqi_very_old_sheet.iloc[9, :].tolist()):
    if str(val) != 'nan':
        print(f"  Col {i} : {str(val)[:80]}")

Colonnes ligne 4 (2016-17) :
  Col 25 : Median
  Col 33 : Median
  Col 36 : Median
  Col 39 : Median
  Col 47 : Median

Toutes colonnes ligne 4 (2016-17) :
  Col 0 : Organisation Code
  Col 1 : Organisation Name
  Col 2 :  Total number of attendances in A&E HES 

(excluding planned follow-up attendanc
  Col 3 :  Number of attendances from A&E Situation Reports (SitReps) [see footnote 1] 
  Col 4 : A&E HES attendances as % of SitRep attendances [see footnote 2]
  Col 5 : Assessment of coverage of A&E HES data, compared to Sitrep data
  Col 6 : Total number of attendances in A&E HES 

(excluding planned follow-up attendance
  Col 7 : Attendances with an unknown attendance disposal category
  Col 8 : % of attendances with unknown attendance disposal categories
  Col 9 : % of attendances with an attendance disposal category of "Other"
  Col 10 : Total number of attendances in A&E HES 

(excluding planned follow-up attendance
  Col 11 : Number of attendances with an attendance disposal of "

In [43]:
# Vérifier quelle colonne Median correspond au temps total
# En regardant les lignes au-dessus des headers

print("Ligne 3 (2016-17) — titres de sections :")
for i, val in enumerate(df_aeqi_old_sheet.iloc[3, :].tolist()):
    if str(val) != 'nan':
        print(f"  Col {i} : {str(val)[:80]}")

print("\nLigne 3 (2012-13) — titres de sections :")
for i, val in enumerate(df_aeqi_very_old_sheet.iloc[8, :].tolist()):
    if str(val) != 'nan':
        print(f"  Col {i} : {str(val)[:80]}")

Ligne 3 (2016-17) — titres de sections :
  Col 2 : DATA COVERAGE
  Col 6 : Left department before being seen for treatment
(Data quality)
  Col 10 : Left department before being seen for treatment
(Performance)
  Col 13 : Re-attendance
(Data quality)
  Col 16 : Re-attendance
(Performance)
  Col 19 : Timeliness 
(Data quality)
  Col 20 : Time to initial assessment [emergency ambulance cases only]
(Data quality)
  Col 25 : Time to initial assessment [emergency ambulance cases only]
(Performance; minute
  Col 28 : Time to Treatment
(Data quality)
  Col 33 : Time to Treatment
(Performance; minutes)
  Col 36 : Total Time in A&E (minutes)

[Admitted patients only]
  Col 39 : Total Time in A&E (minutes)

[Non-admitted patients only]
  Col 42 : Total Time in A&E
(Data quality)

[All patients]
  Col 47 : Total Time in A&E
(Performance; minutes)

[All patients]

Ligne 3 (2012-13) — titres de sections :
  Col 9 : DATA COVERAGE
  Col 16 : Left department before being seen for treatment
(Data quali

In [45]:
# Vérifier avec des données réelles
print("Données trusts 2016-17 (code, nom, temps médian col 47) :")
for i in range(5, 20):
    row = df_aeqi_old_sheet.iloc[i, :].tolist()
    code = str(row[0])
    if len(code) == 3 and code[0].isalpha() and code != 'ENG':
        print(f"  code={code}, median={row[47]}")
        if i > 12:
            break

print("\nDonnées trusts 2012-13 (code, nom, temps médian col 59) :")
for i in range(10, 25):
    row = df_aeqi_very_old_sheet.iloc[i, :].tolist()
    code = str(row[5])
    if len(code) == 3 and code[0].isalpha() and code != 'ENG':
        print(f"  code={code}, median={row[59]}")
        if i > 17:
            break

print("\n✅ CONCLUSION AEQI :")
print("   Format récent CSV (2021-22+)  → MEASURE_NAME='TOTAL_TIME_MEDIAN'")
print("   Format 2016-17 xlsx           → Col 0=code, Col 47=Median total")
print("   Format ancien 2012-13 xls     → Col 5=code, Col 59=Median total")

Données trusts 2016-17 (code, nom, temps médian col 47) :
  code=nan, median=nan
  code=REM, median=108
  code=RCF, median=158
  code=RBS, median=141
  code=RTK, median=179
  code=RF4, median=188
  code=RFF, median=176

Données trusts 2012-13 (code, nom, temps médian col 59) :
  code=nan, median=nan
  code=RTV, median=62
  code=REM, median=162
  code=RCF, median=140
  code=RBS, median=130
  code=RTK, median=161
  code=RF4, median=186

✅ CONCLUSION AEQI :
   Format récent CSV (2021-22+)  → MEASURE_NAME='TOTAL_TIME_MEDIAN'
   Format 2016-17 xlsx           → Col 0=code, Col 47=Median total
   Format ancien 2012-13 xls     → Col 5=code, Col 59=Median total


In [47]:
# ============================================================
# SECTION 7 — EXPLORATION FICHIER WORKFORCE
# ============================================================
# Publié mensuellement par NHS Digital.
# On utilise Mars = fin d'année fiscale NHS.
#
# Features calculées depuis ce fichier :
#   Ratio_Medecin_Lit       = medecins / lits
#   Ratio_Infirmier_Lit     = infirmiers / lits
#   Ratio_Infirmier_Medecin = infirmiers / medecins
#
# ============================================================

print("=" * 60)
print("EXPLORATION WORKFORCE — Comparaison formats")
print("=" * 60)

# ── Format récent (2022-23) ──────────────────────────────
print("\n📁 FORMAT RÉCENT — Workforce Mars 2023")
df_wf_new = pd.read_excel(
    rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx',
    sheet_name=None
)
print(f"   Onglets disponibles :")
for s in df_wf_new.keys():
    print(f"   - {s}")

EXPLORATION WORKFORCE — Comparaison formats

📁 FORMAT RÉCENT — Workforce Mars 2023
   Onglets disponibles :
   - Title sheet
   - 1. England
   - 2. NHSE, Org & SG - HC
   - 3. NHSE, Org & SG - FTE
   - 4. All Staff, NHSE & Org - HC
   - 5. All Staff, NHSE & Org - FTE


In [49]:
# Explorer onglet organisation
df_wf_org = pd.read_excel(
    rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx',
    sheet_name='2. NHSE, Org & SG - HC',
    header=None
)
print(f"   Shape : {df_wf_org.shape}")

# Chercher headers
for i in range(0, 15):
    row = df_wf_org.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 3:
        print(f"   Headers ligne {i} : {non_nan[:8]}")
        break

# Chercher colonnes médecins et infirmiers
print(f"\n   Toutes colonnes ligne headers :")
for i in range(0, 15):
    row = df_wf_org.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 3:
        for j, val in enumerate(row):
            if str(val) != 'nan':
                if any(word in str(val).lower() for word in 
                       ['doctor', 'nurse', 'nurs', 'medic', 'org code', 
                        'trust', 'total']):
                    print(f"   Col {j} : {str(val)[:60]}")
        break

   Shape : (332, 33)
   Headers ligne 5 : ['NHS England region code', 'NHS England region name', 'ICS code', 'ICS name', 'Organisation name', 'Organisation code', 'Total', 'Professionally qualified clinical staff']

   Toutes colonnes ligne headers :
   Col 6 : Total
   Col 8 : HCHS Doctors
   Col 11 : Specialty Doctor
   Col 15 : Foundation Doctor Year 2
   Col 16 : Foundation Doctor Year 1
   Col 18 : Other and Local HCHS Doctor Grades
   Col 19 : Nurses & health visitors
   Col 24 : Support to doctors, nurses & midwives


In [51]:
# Voir données trusts
print(f"\n   Colonnes complètes ligne 5 :")
for i, val in enumerate(df_wf_org.iloc[5, :].tolist()):
    if str(val) != 'nan':
        print(f"   Col {i} : {val}")

# Exemple données
print(f"\n   Exemple données lignes 7-10 :")
print(df_wf_org.iloc[7:10, [5,6,8,19]].to_string())

# Format ancien 2016-17
print(f"\n📁 FORMAT ANCIEN — Workforce Mars 2017")
df_wf_old = pd.read_excel(
    rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx',
    sheet_name=None
)
print(f"   Onglets : {list(df_wf_old.keys())}")


   Colonnes complètes ligne 5 :
   Col 0 : NHS England region code
   Col 1 : NHS England region name
   Col 2 : ICS code
   Col 3 : ICS name
   Col 4 : Organisation name
   Col 5 : Organisation code
   Col 6 : Total
   Col 7 : Professionally qualified clinical staff
   Col 8 : HCHS Doctors
   Col 9 : Consultant
   Col 10 : Associate Specialist
   Col 11 : Specialty Doctor
   Col 12 : Staff Grade
   Col 13 : Specialty Registrar
   Col 14 : Core Training
   Col 15 : Foundation Doctor Year 2
   Col 16 : Foundation Doctor Year 1
   Col 17 : Hospital Practitioner / Clinical Assistant
   Col 18 : Other and Local HCHS Doctor Grades
   Col 19 : Nurses & health visitors
   Col 20 : Midwives
   Col 21 : Ambulance staff
   Col 22 : Scientific, therapeutic & technical staff
   Col 23 : Support to clinical staff
   Col 24 : Support to doctors, nurses & midwives
   Col 25 : Support to ambulance staff
   Col 26 : Support to ST&T staff
   Col 27 : NHS infrastructure support
   Col 28 : Central funct

In [53]:
# Explorer format ancien 2017
df_wf_old_sheet = pd.read_excel(
    rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx',
    sheet_name='1. HEE Org Main Staff Gp HC',
    header=None
)
print(f"   Shape : {df_wf_old_sheet.shape}")

# Chercher headers
for i in range(0, 15):
    row = df_wf_old_sheet.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 3:
        print(f"   Headers ligne {i} : {non_nan[:8]}")
        for j, val in enumerate(df_wf_old_sheet.iloc[i, :].tolist()):
            if str(val) != 'nan':
                if any(word in str(val).lower() for word in
                       ['doctor', 'nurse', 'org code', 'trust', 
                        'total', 'organisation']):
                    print(f"   Col {j} : {str(val)[:60]}")
        break

# Format très ancien 2013-14
print(f"\n📁 FORMAT TRÈS ANCIEN — Workforce Mars 2014")
df_wf_very_old = pd.read_excel(
    rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls',
    sheet_name=None
)
print(f"   Onglets : {list(df_wf_very_old.keys())}")

   Shape : (485, 37)
   Headers ligne 6 : ['Total', 'Professionally qualified clinical staff', 'HCHS doctors', 'Consultant (including Directors of Public Health)', 'Associate Specialist', 'Specialty Doctor', 'Staff Grade', 'Specialty Registrar']
   Col 4 : Total
   Col 8 : HCHS doctors
   Col 11 : Specialty Doctor
   Col 15 : Foundation Doctor Year 2
   Col 16 : Foundation Doctor Year 1
   Col 18 : Other and Local HCHS Doctor Grades
   Col 20 : Nurses & health visitors
   Col 26 : Support to doctors, nurses & midwives

📁 FORMAT TRÈS ANCIEN — Workforce Mars 2014
   Onglets : ['Contents', '1. Drs by HEE,Org, Grade HC', '2. Drs by HEE,Org, Grade FTE ', '3.NonMed by HEE,Org,Staffgr', '4.NonMed by HEE,Org,Staffgr', '5. All Staff by HEE,Org HC', '6. All Staff by HEE,Org FTE']


In [55]:
# Explorer format 2014 - onglet All Staff
df_wf_2014 = pd.read_excel(
    rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls',
    sheet_name='5. All Staff by HEE,Org HC',
    header=None
)
print(f"   Shape : {df_wf_2014.shape}")

# Chercher headers
for i in range(0, 15):
    row = df_wf_2014.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 3:
        print(f"   Headers ligne {i} : {non_nan[:6]}")
        for j, val in enumerate(df_wf_2014.iloc[i, :].tolist()):
            if str(val) != 'nan':
                if any(word in str(val).lower() for word in
                       ['doctor', 'nurse', 'org code', 'trust',
                        'total', 'organisation', 'code']):
                    print(f"   Col {j} : {str(val)[:60]}")
        break

# Exemple données
print(f"\n   Exemple lignes 8-12 :")
print(df_wf_2014.iloc[8:12, :8].to_string())

   Shape : (706, 59)
   Headers ligne 4 : ['2009-09-01 00:00:00', '2009-10-01 00:00:00', '2009-11-01 00:00:00', '2009-12-01 00:00:00', '2010-01-01 00:00:00', '2010-02-01 00:00:00']

   Exemple lignes 8-12 :
        0                               1                                                 2    3      4      5      6      7
8     NaN                             NaN                                               NaN  NaN    NaN    NaN    NaN    NaN
9   YDF21  Health Education East Midlands                                               NaN  NaN  93355  93914  94409  94342
10    NaN                             NaN  Chesterfield Royal Hospital NHS Foundation Trust  RFS   3392   3392   3392   3392
11    NaN                             NaN                                    Derby City PCT  5N7    996   1006   1008   1002


In [57]:
# Essayer onglet HEE Org pour 2014
df_wf_2014_org = pd.read_excel(
    rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls',
    sheet_name='3.NonMed by HEE,Org,Staffgr',
    header=None
)
print(f"   Shape : {df_wf_2014_org.shape}")

for i in range(0, 15):
    row = df_wf_2014_org.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 3:
        print(f"   Headers ligne {i} : {non_nan[:8]}")
        break

print(f"\n   Exemple lignes 5-10 colonnes 0-6 :")
print(df_wf_2014_org.iloc[5:10, :7].to_string())

   Shape : (532, 27)
   Headers ligne 4 : ['Total HCHS non-medical staff', 'Qualified nursing, midwifery & health visiting staff', 'Registered midwife', 'Health visitor', 'Qualified School Nurse', 'Qualified scientific, therapeutic & technical staff', 'Qualified Allied Health Professions', 'Qualified Healthcare Scientists']

   Exemple lignes 5-10 colonnes 0-6 :
         0                               1    2    3        4       5      6
5      NaN                             NaN  NaN  NaN      NaN     NaN    NaN
6      NaN                             NaN  NaN  NaN      NaN     NaN    NaN
7  England                             NaN  NaN  NaN  1095907  354686  26494
8      NaN                             NaN  NaN  NaN      NaN     NaN    NaN
9    YDF21  Health Education East Midlands  NaN  NaN    83769   27903   1962


In [59]:
# Onglet médecins 2014
df_wf_2014_doc = pd.read_excel(
    rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls',
    sheet_name='1. Drs by HEE,Org, Grade HC',
    header=None
)
print(f"   Shape médecins : {df_wf_2014_doc.shape}")

for i in range(0, 15):
    row = df_wf_2014_doc.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 3:
        print(f"   Headers ligne {i} : {non_nan[:8]}")
        break

print(f"\n   Exemple lignes 5-12 colonnes 0-6 :")
print(df_wf_2014_doc.iloc[5:12, :7].to_string())

   Shape médecins : (377, 17)
   Headers ligne 4 : ['All HCHS doctors (non locum)', 'Consultant (including Directors of Public Health)', 'Associate Specialist', 'Specialty Doctor', 'Staff Grade', 'Registrar Group', 'Senior House Officer', 'Foundation Year 2']

   Exemple lignes 5-12 colonnes 0-6 :
          0                               1                                                 2    3       4   5      6
5       NaN                             NaN                                               NaN  NaN     NaN NaN    NaN
6       NaN                             NaN                                               NaN  NaN     NaN NaN    NaN
7   England                             NaN                                               NaN  NaN  108635 NaN  42005
8       NaN                             NaN                                               NaN  NaN     NaN NaN    NaN
9     YDF21  Health Education East Midlands                                               NaN  NaN    7321 NaN 

In [61]:
print("✅ CONCLUSION WORKFORCE :")
print("\n   FORMAT RÉCENT (2018+) :")
print("   Onglet  : '2. NHSE, Org & SG - HC'")
print("   Col 5   : Organisation code (Trust code)")
print("   Col 8   : HCHS Doctors")
print("   Col 19  : Nurses & health visitors")

print("\n   FORMAT 2017 :")
print("   Onglet  : '1. HEE Org Main Staff Gp HC'")
print("   Col 8   : HCHS doctors")
print("   Col 20  : Nurses & health visitors")

print("\n   FORMAT ANCIEN (2013-2016) :")
print("   Deux onglets séparés !")
print("   Médecins  : '1. Drs by HEE,Org, Grade HC' → Col 3=code, Col 4=doctors")
print("   Infirmiers: '3.NonMed by HEE,Org,Staffgr' → Col 2=name, Col 3=code, Col 4=nurses")

print("\n   Features calculées :")
print("   Ratio_Medecin_Lit       = doctors / lits_disponibles")
print("   Ratio_Infirmier_Lit     = nurses  / lits_disponibles")
print("   Ratio_Infirmier_Medecin = nurses  / doctors")

print("\n✅ EXPLORATION COMPLÈTE TERMINÉE !")
print("   Tous les formats compris pour toutes les années !")
print("   On peut maintenant construire les fonctions de fusion.")

✅ CONCLUSION WORKFORCE :

   FORMAT RÉCENT (2018+) :
   Onglet  : '2. NHSE, Org & SG - HC'
   Col 5   : Organisation code (Trust code)
   Col 8   : HCHS Doctors
   Col 19  : Nurses & health visitors

   FORMAT 2017 :
   Onglet  : '1. HEE Org Main Staff Gp HC'
   Col 8   : HCHS doctors
   Col 20  : Nurses & health visitors

   FORMAT ANCIEN (2013-2016) :
   Deux onglets séparés !
   Médecins  : '1. Drs by HEE,Org, Grade HC' → Col 3=code, Col 4=doctors
   Infirmiers: '3.NonMed by HEE,Org,Staffgr' → Col 2=name, Col 3=code, Col 4=nurses

   Features calculées :
   Ratio_Medecin_Lit       = doctors / lits_disponibles
   Ratio_Infirmier_Lit     = nurses  / lits_disponibles
   Ratio_Infirmier_Medecin = nurses  / doctors

✅ EXPLORATION COMPLÈTE TERMINÉE !
   Tous les formats compris pour toutes les années !
   On peut maintenant construire les fonctions de fusion.


In [63]:
# ============================================================
# SECTION 8 — FONCTIONS DE CHARGEMENT PAR TYPE DE FICHIER
# ============================================================
# Chaque fonction charge un type de fichier NHS et retourne
# un DataFrame standardisé avec Trust_code comme clé.
# ============================================================

# ── FONCTION 1 — Chargement Beds ────────────────────────
def charger_beds(path):
    """
    Charge le fichier NHS Beds (KH03) et retourne :
    Trust_code, lits_disponibles, lits_occupes
    Fonctionne pour tous les formats (2012 → 2025)
    """
    if path.endswith('.xls'):
        df = pd.read_excel(path, sheet_name='NHS Trust by Sector', 
                          header=None, engine='xlrd')
    else:
        df = pd.read_excel(path, sheet_name='NHS Trust by Sector', 
                          header=None)
    
    # Trouver la ligne header (contient 'Org Code')
    header_row = None
    for i in range(10, 20):
        row = df.iloc[i, :].tolist()
        if any('Org Code' in str(v) or 'Organisation Code' in str(v) 
               for v in row):
            header_row = i
            break
    
    if header_row is None:
        header_row = 14  # défaut
    
    # Extraire données
    df_data = df.iloc[header_row+1:, :].copy()
    df_data.columns = range(len(df_data.columns))
    
    # Trust code = col 4, lits dispo = col 6, lits occupés = col 12
    df_beds = pd.DataFrame({
        'Trust_code'       : df_data[4].astype(str).str.strip(),
        'lits_disponibles' : pd.to_numeric(df_data[6],  errors='coerce'),
        'lits_occupes'     : pd.to_numeric(df_data[12], errors='coerce'),
    })
    
    # Garder seulement les trusts NHS (code 3 caractères)
    df_beds = df_beds[
        df_beds['Trust_code'].str.len() == 3
    ].dropna(subset=['lits_disponibles', 'lits_occupes'])
    
    # Calculer KPIs
    df_beds['C2_occupation']   = round(
        df_beds['lits_occupes'] / df_beds['lits_disponibles'] * 100, 1)
    df_beds['A8_lits_vacants'] = round(
        (df_beds['lits_disponibles'] - df_beds['lits_occupes']) 
        / df_beds['lits_disponibles'] * 100, 1)
    
    return df_beds[['Trust_code', 'lits_disponibles', 'lits_occupes',
                    'C2_occupation', 'A8_lits_vacants']]

# Test
print("Test charger_beds 2022-23...")
df_test = charger_beds(rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2022-23-Final.xlsx')
print(f"✅ {len(df_test)} trusts chargés")
print(df_test.head(3).to_string())

print("\nTest charger_beds 2012-13...")
df_test2 = charger_beds(rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2012-13-Revised-Final-79659.xls')
print(f"✅ {len(df_test2)} trusts chargés")
print(df_test2.head(3).to_string())

Test charger_beds 2022-23...
✅ 189 trusts chargés
   Trust_code  lits_disponibles  lits_occupes  C2_occupation  A8_lits_vacants
17        R1H       2024.400000   1843.811111           91.1              8.9
18        R1K       1079.911111   1014.177778           93.9              6.1
19        RAL       1117.200000    946.844444           84.8             15.2

Test charger_beds 2012-13...
✅ 229 trusts chargés
   Trust_code  lits_disponibles  lits_occupes  C2_occupation  A8_lits_vacants
17        RE9        415.000000    356.777778           86.0             14.0
18        RLN        887.122222    685.488889           77.3             22.7
19        RR7        517.000000    459.888889           89.0             11.0


In [67]:
# Voir les vraies colonnes du fichier SHMI
extract_dir = rf'{DATA}\SHMI_temp'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(rf'{DATA}\SHMI data files, Apr22-Mar23.zip', 'r') as z:
    z.extractall(extract_dir)
    for f in z.namelist():
        if 'trust level' in f.lower() and f.endswith('.csv'):
            shmi_file = os.path.join(extract_dir, f)
            break

df_shmi_test = pd.read_csv(shmi_file)
print("Colonnes SHMI 2022-23 :")
for col in df_shmi_test.columns:
    print(f"  - {col}")

import shutil
shutil.rmtree(extract_dir)

Colonnes SHMI 2022-23 :
  - TIME_PERIOD
  - PUBLICATION_MONTH
  - ORDER
  - PROVIDER_CODE
  - PROVIDER_NAME
  - SHMI_VALUE
  - SHMI_BANDING
  - PO_LL
  - PO_UL
  - OD_LL
  - OD_UL
  - OBSERVED
  - EXPECTED
  - ELECTIVE_DEPTH
  - NONELECTIVE_DEPTH
  - PALLIATIVE_DEATHS
  - PALLIATIVE_CODE


In [71]:
def charger_shmi(path):
    """
    Charge le fichier SHMI (zip) et retourne :
    Trust_code, C3_mortalite, SHMI_VALUE
    """
    extract_dir = rf'{DATA}\SHMI_temp'
    os.makedirs(extract_dir, exist_ok=True)
    
    with zipfile.ZipFile(path, 'r') as z:
        z.extractall(extract_dir)
        shmi_file = None
        for f in z.namelist():
            if 'trust level' in f.lower() and f.endswith('.csv'):
                shmi_file = os.path.join(extract_dir, f)
                break
        if shmi_file is None:
            for f in z.namelist():
                if 'trust' in f.lower() and f.endswith('.csv'):
                    shmi_file = os.path.join(extract_dir, f)
                    break
    
    if shmi_file is None:
        import shutil
        shutil.rmtree(extract_dir)
        return pd.DataFrame()
    
    df = pd.read_csv(shmi_file)
    df.columns = [c.upper().strip() for c in df.columns]
    
    # Format ancien → SPELLS disponible → C3 = OBSERVED/SPELLS × 100
    if 'SPELLS' in df.columns:
        df_shmi = pd.DataFrame({
            'Trust_code'  : df['PROVIDER_CODE'].astype(str).str.strip(),
            'OBSERVED'    : pd.to_numeric(df['OBSERVED'], errors='coerce'),
            'SPELLS'      : pd.to_numeric(df['SPELLS'],   errors='coerce'),
            'SHMI_VALUE'  : pd.to_numeric(df['SHMI_VALUE'], errors='coerce'),
        })
        df_shmi = df_shmi.dropna(subset=['OBSERVED', 'SPELLS'])
        df_shmi['C3_mortalite'] = round(
            df_shmi['OBSERVED'] / df_shmi['SPELLS'] * 100, 3)
    
    # Format récent → SPELLS absent → C3 calculé depuis OBSERVED/EXPECTED
    # SHMI = OBSERVED/EXPECTED → OBSERVED = SHMI × EXPECTED
    # Taux mortalité = OBSERVED / (EXPECTED / taux_national_moyen)
    # On utilise OBSERVED/EXPECTED × taux_ref (2.5% NHS moyen)
    else:
        # Garder seulement la dernière période (une ligne par trust)
        if 'TIME_PERIOD' in df.columns:
            df = df.sort_values('TIME_PERIOD').groupby(
                'PROVIDER_CODE').last().reset_index()
        
        df_shmi = pd.DataFrame({
            'Trust_code'  : df['PROVIDER_CODE'].astype(str).str.strip(),
            'OBSERVED'    : pd.to_numeric(df['OBSERVED'],   errors='coerce'),
            'SPELLS'      : pd.to_numeric(df['EXPECTED'],   errors='coerce'),
            'SHMI_VALUE'  : pd.to_numeric(df['SHMI_VALUE'], errors='coerce'),
        })
        df_shmi = df_shmi.dropna(subset=['OBSERVED', 'SPELLS'])
        # C3 = SHMI_VALUE × taux_mortalité_national_moyen_NHS (2.5%)
        TAUX_NATIONAL_NHS = 2.5
        df_shmi['C3_mortalite'] = round(
            df_shmi['SHMI_VALUE'] * TAUX_NATIONAL_NHS, 3)
    
    import shutil
    shutil.rmtree(extract_dir)
    
    # Garder seulement trusts NHS (code 3 caractères)
    df_shmi = df_shmi[
        df_shmi['Trust_code'].str.len() == 3
    ]
    
    return df_shmi[['Trust_code', 'OBSERVED', 'SPELLS', 
                    'C3_mortalite', 'SHMI_VALUE']]

# Test
print("Test charger_shmi 2022-23...")
df_test = charger_shmi(rf'{DATA}\SHMI data files, Apr22-Mar23.zip')
print(f"✅ {len(df_test)} trusts chargés")
print(df_test.head(3).to_string())

print("\nTest charger_shmi 2017-18...")
df_test2 = charger_shmi(rf'{DATA}\SHMI data files, Apr17-Mar18.zip')
print(f"✅ {len(df_test2)} trusts chargés")
print(df_test2.head(3).to_string())

Test charger_shmi 2022-23...
✅ 133 trusts chargés
  Trust_code  OBSERVED  SPELLS  C3_mortalite  SHMI_VALUE
0        R0A      3620    3865         2.342      0.9366
1        R0B      3020    2745         2.751      1.1004
2        R0D      3195    3555         2.246      0.8986

Test charger_shmi 2017-18...
✅ 131 trusts chargés
  Trust_code  OBSERVED  SPELLS  C3_mortalite  SHMI_VALUE
0        REM      2190   53679         4.080      1.0002
1        RCF      1011   32537         3.107      0.9408
2        RTK      1734   42039         4.125      0.9453


In [73]:
# ── FONCTION 3 — Chargement CDI ─────────────────────────
def charger_cdi(path, annee):
    """
    Charge le fichier CDI et retourne :
    Trust_code, CDI_total_cases
    Gère tous les formats (2012 → 2025)
    """
    
    # ── Format récent ODS (2019+) ──
    if path.endswith('.ods'):
        df = pd.read_excel(path, engine='odf', sheet_name=None)
        
        # Format avec onglet 'data' (2021+)
        if 'data' in df:
            df_data = df['data']
            # Filtre NHS acute trust
            if 'Organisation type' in df_data.columns:
                df_trusts = df_data[
                    df_data['Organisation type'] == 'NHS acute trust'
                ]
                metric_col = 'Metric'
                code_col   = 'Organisation code'
                value_col  = 'Figure'
                metric_val = 'Total cases'
            else:
                df_trusts = df_data
                metric_col = df_data.columns[5]
                code_col   = df_data.columns[2]
                value_col  = df_data.columns[6]
                metric_val = df_data[metric_col].iloc[0]
            
            df_cdi = df_trusts[
                df_trusts[metric_col] == metric_val
            ].groupby(code_col)[value_col].sum().reset_index()
            df_cdi.columns = ['Trust_code', 'CDI_total_cases']
        
        # Format avec onglet 'Table 2' (2016-2019)
        else:
            sheet_name = [s for s in df.keys() 
                         if 'table' in s.lower() or 'data' in s.lower()][0]
            df_raw = df[sheet_name]
            # Chercher colonnes trust code et total annuel
            # Col 0/1 = trust code, dernière col = total année
            df_cdi = pd.DataFrame({
                'Trust_code'     : df_raw.iloc[4:, 0].astype(str).str.strip(),
                'CDI_total_cases': pd.to_numeric(
                    df_raw.iloc[4:, -1], errors='coerce')
            })
            df_cdi = df_cdi[df_cdi['Trust_code'].str.len() == 3]
    
    # ── Format xlsx (2014-15) ──
    elif path.endswith('.xlsx'):
        df_raw = pd.read_excel(path, header=None)
        df_cdi = pd.DataFrame({
            'Trust_code'     : df_raw.iloc[4:, 0].astype(str).str.strip(),
            'CDI_total_cases': pd.to_numeric(
                df_raw.iloc[4:, -1], errors='coerce')
        })
        df_cdi = df_cdi[df_cdi['Trust_code'].str.len() == 3]
    
    # ── Format xls très ancien (2012-13) ──
    else:
        df_raw = pd.read_excel(path, header=None, engine='xlrd')
        df_cdi = pd.DataFrame({
            'Trust_code'     : df_raw.iloc[4:, 0].astype(str).str.strip(),
            'CDI_total_cases': pd.to_numeric(
                df_raw.iloc[4:, -1], errors='coerce')
        })
        df_cdi = df_cdi[df_cdi['Trust_code'].str.len() == 3]
    
    df_cdi = df_cdi.dropna(subset=['CDI_total_cases'])
    df_cdi['CDI_total_cases'] = pd.to_numeric(
        df_cdi['CDI_total_cases'], errors='coerce')
    
    return df_cdi[['Trust_code', 'CDI_total_cases']]

# Tests
print("Test charger_cdi 2022-23...")
df_test = charger_cdi(rf'{DATA}\Table-6-CDI-march-2023.ods', '2022-23')
print(f"✅ {len(df_test)} trusts | Exemple : {df_test.head(2).to_string()}")

print("\nTest charger_cdi 2021-22...")
df_test2 = charger_cdi(rf'{DATA}\Table_6_CDI_march_2022.ods', '2021-22')
print(f"✅ {len(df_test2)} trusts | Exemple : {df_test2.head(2).to_string()}")

print("\nTest charger_cdi 2019-20...")
df_test3 = charger_cdi(rf'{DATA}\Table_6_CDI_march_2020.ods', '2019-20')
print(f"✅ {len(df_test3)} trusts | Exemple : {df_test3.head(2).to_string()}")

Test charger_cdi 2022-23...
✅ 135 trusts | Exemple :   Trust_code  CDI_total_cases
0        R0A              251
1        R0B              125

Test charger_cdi 2021-22...
✅ 138 trusts | Exemple :   Trust_code  CDI_total_cases
0        R0A              293
1        R0B              100

Test charger_cdi 2019-20...
✅ 139 trusts | Exemple :   Trust_code  CDI_total_cases
0        R0A              240
1        R0B              154


In [75]:
print("Test charger_cdi 2016-17...")
df_test4 = charger_cdi(rf'{DATA}\Table_2_Monthly_CDI_2P_March_2017.ods', '2016-17')
print(f"✅ {len(df_test4)} trusts | Exemple : {df_test4.head(2).to_string()}")

print("\nTest charger_cdi 2014-15...")
df_test5 = charger_cdi(rf'{DATA}\Table_2_Monthly_CDI_2P_Mar_2015.xlsx', '2014-15')
print(f"✅ {len(df_test5)} trusts | Exemple : {df_test5.head(2).to_string()}")

print("\nTest charger_cdi 2012-13...")
df_test6 = charger_cdi(rf'{DATA}\Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls', '2012-13')
print(f"✅ {len(df_test6)} trusts | Exemple : {df_test6.head(2).to_string()}")

Test charger_cdi 2016-17...
✅ 0 trusts | Exemple : Empty DataFrame
Columns: [Trust_code, CDI_total_cases]
Index: []

Test charger_cdi 2014-15...


IndexError: single positional indexer is out-of-bounds

In [77]:
# Voir structure CDI 2016-17
df_cdi_2017 = pd.read_excel(
    rf'{DATA}\Table_2_Monthly_CDI_2P_March_2017.ods',
    engine='odf', sheet_name=None)
print("Onglets 2016-17 :", list(df_cdi_2017.keys()))

# Voir premier onglet
first_sheet = list(df_cdi_2017.keys())[0]
df_s = df_cdi_2017[first_sheet]
print(f"\nShape : {df_s.shape}")
print(f"\nLignes 0-8 colonnes 0-5 :")
print(df_s.iloc[:8, :6].to_string())

# Voir structure CDI 2014-15
df_cdi_2015 = pd.read_excel(
    rf'{DATA}\Table_2_Monthly_CDI_2P_Mar_2015.xlsx',
    sheet_name=None)
print("\n\nOnglets 2014-15 :", list(df_cdi_2015.keys()))
first_sheet2 = list(df_cdi_2015.keys())[0]
df_s2 = df_cdi_2015[first_sheet2]
print(f"\nShape : {df_s2.shape}")
print(f"\nLignes 0-8 colonnes 0-5 :")
print(df_s2.iloc[:8, :6].to_string())

Onglets 2016-17 : ['Caveats', 'Table_2a_Monthly', 'Table_2b_FY_2015-16']

Shape : (0, 0)

Lignes 0-8 colonnes 0-5 :
Empty DataFrame
Columns: []
Index: []


Onglets 2014-15 : ['Caveats', 'Table 2a Monthly', 'Table 2b FY 2013-14']

Shape : (0, 0)

Lignes 0-8 colonnes 0-5 :
Empty DataFrame
Columns: []
Index: []


In [79]:
# Voir onglet Table_2b pour 2016-17
df_2017_b = df_cdi_2017['Table_2b_FY_2015-16']
print(f"Shape Table_2b 2016-17 : {df_2017_b.shape}")
print(f"\nLignes 0-8 colonnes 0-6 :")
print(df_2017_b.iloc[:8, :7].to_string())

# Voir onglet Table 2b pour 2014-15
df_2015_b = df_cdi_2015['Table 2b FY 2013-14']
print(f"\nShape Table_2b 2014-15 : {df_2015_b.shape}")
print(f"\nLignes 0-8 colonnes 0-6 :")
print(df_2015_b.iloc[:8, :7].to_string())

Shape Table_2b 2016-17 : (168, 17)

Lignes 0-8 colonnes 0-6 :
              Title: ARCHIVE Financial Year - Monthly counts of C. difficile infection by Acute Trust for patients aged 2 years and over - Trust Apportioned only*  Unnamed: 2  Unnamed: 3           Unnamed: 4           Unnamed: 5           Unnamed: 6
0                NaN                                                                                                                                           NaN         NaN         NaN                  NaN                  NaN                  NaN
1  Reporting Period:                                                                                                                      April 2015 to March 2016         NaN         NaN                  NaN                  NaN                  NaN
2                NaN                                                                                                                                           NaN         NaN         NaN  

In [81]:
# Voir colonnes complètes et données
print("Colonnes Table_2b 2016-17 :")
for i, col in enumerate(df_2017_b.columns):
    print(f"  Col {i} : {col}")

print(f"\nLignes 8-12 :")
print(df_2017_b.iloc[8:12, :].to_string())

Colonnes Table_2b 2016-17 :
  Col 0 : Title:
  Col 1 : ARCHIVE Financial Year - Monthly counts of C. difficile infection by Acute Trust for patients aged 2 years and over - Trust Apportioned only*
  Col 2 : Unnamed: 2
  Col 3 : Unnamed: 3
  Col 4 : Unnamed: 4
  Col 5 : Unnamed: 5
  Col 6 : Unnamed: 6
  Col 7 : Unnamed: 7
  Col 8 : Unnamed: 8
  Col 9 : Unnamed: 9
  Col 10 : Unnamed: 10
  Col 11 : Unnamed: 11
  Col 12 : Unnamed: 12
  Col 13 : Unnamed: 13
  Col 14 : Unnamed: 14
  Col 15 : Unnamed: 15
  Col 16 : Unnamed: 16

Lignes 8-12 :
   Title: ARCHIVE Financial Year - Monthly counts of C. difficile infection by Acute Trust for patients aged 2 years and over - Trust Apportioned only*            Unnamed: 2                   Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8 Unnamed: 9 Unnamed: 10 Unnamed: 11 Unnamed: 12 Unnamed: 13 Unnamed: 14 Unnamed: 15                  Unnamed: 16
8     NaN                                                                                

In [83]:
# Corriger la fonction charger_cdi pour les formats anciens ODS/xlsx
def charger_cdi(path, annee):
    """
    Charge le fichier CDI et retourne :
    Trust_code, CDI_total_cases
    """
    
    # ── Format récent ODS avec onglet 'data' (2021+) ──
    if path.endswith('.ods'):
        df_sheets = pd.read_excel(path, engine='odf', sheet_name=None)
        
        if 'data' in df_sheets:
            df_data = df_sheets['data']
            df_trusts = df_data[
                df_data['Organisation type'] == 'NHS acute trust'
            ]
            df_cdi = df_trusts[
                df_trusts['Metric'] == 'Total cases'
            ].groupby('Organisation code')['Figure'].sum().reset_index()
            df_cdi.columns = ['Trust_code', 'CDI_total_cases']
        
        else:
            # Format ODS ancien avec Table_2b (2014-2019)
            # Chercher onglet Table_2b
            sheet_name = [s for s in df_sheets.keys() 
                         if 'table_2b' in s.lower() or 'table 2b' in s.lower()]
            
            if sheet_name:
                df_raw = df_sheets[sheet_name[0]]
                # Headers ligne 7, données ligne 10+
                # Col 0 = Trust code, Col 4 à 15 = mois
                df_data = df_raw.iloc[10:, :].copy()
                df_data.columns = range(len(df_data.columns))
                
                # Trust code = col 0 (3 caractères)
                df_data['Trust_code'] = df_data[0].astype(str).str.strip()
                df_data = df_data[df_data['Trust_code'].str.len() == 3]
                
                # Somme des 12 mois (col 4 à 15)
                cols_mois = list(range(4, 16))
                df_data['CDI_total_cases'] = df_data[cols_mois].apply(
                    pd.to_numeric, errors='coerce').sum(axis=1)
                
                df_cdi = df_data[['Trust_code', 'CDI_total_cases']]
            else:
                print(f"  ⚠️ Onglet Table_2b non trouvé dans {path}")
                return pd.DataFrame()
    
    # ── Format xlsx ancien (2014-15) ──
    elif path.endswith('.xlsx'):
        df_sheets = pd.read_excel(path, sheet_name=None, header=None)
        
        # Chercher onglet Table 2b
        sheet_name = [s for s in df_sheets.keys() 
                     if 'table 2b' in s.lower() or 'table_2b' in s.lower()]
        
        if sheet_name:
            df_raw = df_sheets[sheet_name[0]]
            df_data = df_raw.iloc[10:, :].copy()
            df_data.columns = range(len(df_data.columns))
            df_data['Trust_code'] = df_data[0].astype(str).str.strip()
            df_data = df_data[df_data['Trust_code'].str.len() == 3]
            cols_mois = list(range(4, 16))
            df_data['CDI_total_cases'] = df_data[cols_mois].apply(
                pd.to_numeric, errors='coerce').sum(axis=1)
            df_cdi = df_data[['Trust_code', 'CDI_total_cases']]
        else:
            print(f"  ⚠️ Onglet Table 2b non trouvé dans {path}")
            return pd.DataFrame()
    
    # ── Format xls très ancien (2012-13, 2013-14) ──
    else:
        df_raw = pd.read_excel(path, header=None, engine='xlrd')
        # Chercher ligne header avec Trust Code
        header_row = None
        for i in range(0, 20):
            if 'Trust Code' in str(df_raw.iloc[i, 0]):
                header_row = i
                break
        if header_row is None:
            header_row = 7
        
        df_data = df_raw.iloc[header_row+1:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_data['Trust_code'] = df_data[0].astype(str).str.strip()
        df_data = df_data[df_data['Trust_code'].str.len() == 3]
        cols_mois = list(range(4, 16))
        df_data['CDI_total_cases'] = df_data[cols_mois].apply(
            pd.to_numeric, errors='coerce').sum(axis=1)
        df_cdi = df_data[['Trust_code', 'CDI_total_cases']]
    
    df_cdi = df_cdi.dropna(subset=['CDI_total_cases'])
    df_cdi['Trust_code'] = df_cdi['Trust_code'].astype(str).str.strip()
    return df_cdi[['Trust_code', 'CDI_total_cases']]

# Tests complets
for annee, fichier in [
    ('2022-23', rf'{DATA}\Table-6-CDI-march-2023.ods'),
    ('2021-22', rf'{DATA}\Table_6_CDI_march_2022.ods'),
    ('2019-20', rf'{DATA}\Table_6_CDI_march_2020.ods'),
    ('2018-19', rf'{DATA}\Table_2_Monthly_CDI_2P_April_2018.ods'),
    ('2016-17', rf'{DATA}\Table_2_Monthly_CDI_2P_March_2017.ods'),
    ('2015-16', rf'{DATA}\Table_2_Monthly_CDI_2P_March_2016.ods'),
    ('2014-15', rf'{DATA}\Table_2_Monthly_CDI_2P_Mar_2015.xlsx'),
    ('2013-14', rf'{DATA}\Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls'),
]:
    df_t = charger_cdi(fichier, annee)
    print(f"{annee} → {len(df_t)} trusts")

2022-23 → 135 trusts
2021-22 → 138 trusts
2019-20 → 139 trusts
2018-19 → 151 trusts
2016-17 → 154 trusts
2015-16 → 155 trusts
2014-15 → 158 trusts


IndexError: index 0 is out of bounds for axis 0 with size 0

In [85]:
# Voir structure du fichier xls 2013-14
df_xls = pd.read_excel(
    rf'{DATA}\Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls',
    sheet_name=None, engine='xlrd'
)
print("Onglets :", list(df_xls.keys()))

# Voir premier onglet
first = list(df_xls.keys())[0]
df_s = df_xls[first]
print(f"\nShape : {df_s.shape}")
print(f"\nLignes 0-10 colonnes 0-5 :")
print(df_s.iloc[:10, :6].to_string())

Onglets : ['Caveats', 'Table 2a Monthly', 'Table 2b FY 2012-13']

Shape : (0, 0)

Lignes 0-10 colonnes 0-5 :
Empty DataFrame
Columns: []
Index: []


In [87]:
# Voir onglet Table 2b 2012-13
df_2013 = df_xls['Table 2b FY 2012-13']
print(f"Shape : {df_2013.shape}")
print(f"\nLignes 0-12 colonnes 0-6 :")
print(df_2013.iloc[:12, :7].to_string())

Shape : (204, 17)

Lignes 0-12 colonnes 0-6 :
               Title: ARCHIVE Financial Year - Monthly counts of C. difficile infection by Acute Trust for patients aged 2 years and over - Trust Apportioned only*               Unnamed: 2                    Unnamed: 3           Unnamed: 4           Unnamed: 5           Unnamed: 6
0                 NaN                                                                                                                                           NaN                      NaN                           NaN                  NaN                  NaN                  NaN
1   Reporting Period:                                                                                                                      April 2012 to March 2013                      NaN                           NaN                  NaN                  NaN                  NaN
2                 NaN                                                                                       

In [89]:
# Corriger le cas xls dans charger_cdi
def charger_cdi(path, annee):
    """
    Charge le fichier CDI et retourne :
    Trust_code, CDI_total_cases
    """
    
    # ── Format récent ODS avec onglet 'data' (2021+) ──
    if path.endswith('.ods'):
        df_sheets = pd.read_excel(path, engine='odf', sheet_name=None)
        
        if 'data' in df_sheets:
            df_data = df_sheets['data']
            df_trusts = df_data[
                df_data['Organisation type'] == 'NHS acute trust'
            ]
            df_cdi = df_trusts[
                df_trusts['Metric'] == 'Total cases'
            ].groupby('Organisation code')['Figure'].sum().reset_index()
            df_cdi.columns = ['Trust_code', 'CDI_total_cases']
        
        else:
            # Format ODS ancien avec Table_2b
            sheet_name = [s for s in df_sheets.keys() 
                         if 'table_2b' in s.lower() or 'table 2b' in s.lower()]
            if sheet_name:
                df_raw = df_sheets[sheet_name[0]]
                df_data = df_raw.iloc[10:, :].copy()
                df_data.columns = range(len(df_data.columns))
                df_data['Trust_code'] = df_data[0].astype(str).str.strip()
                df_data = df_data[df_data['Trust_code'].str.len() == 3]
                cols_mois = list(range(4, 16))
                df_data['CDI_total_cases'] = df_data[cols_mois].apply(
                    pd.to_numeric, errors='coerce').sum(axis=1)
                df_cdi = df_data[['Trust_code', 'CDI_total_cases']]
            else:
                return pd.DataFrame()
    
    # ── Format xlsx ou xls ancien (2012-2015) ──
    else:
        if path.endswith('.xls'):
            df_sheets = pd.read_excel(path, sheet_name=None, engine='xlrd')
        else:
            df_sheets = pd.read_excel(path, sheet_name=None)
        
        # Chercher onglet Table 2b
        sheet_name = [s for s in df_sheets.keys() 
                     if 'table 2b' in s.lower() or 'table_2b' in s.lower()]
        
        if sheet_name:
            df_raw = df_sheets[sheet_name[0]]
            df_data = df_raw.iloc[10:, :].copy()
            df_data.columns = range(len(df_data.columns))
            df_data['Trust_code'] = df_data[0].astype(str).str.strip()
            df_data = df_data[df_data['Trust_code'].str.len() == 3]
            cols_mois = list(range(4, 16))
            df_data['CDI_total_cases'] = df_data[cols_mois].apply(
                pd.to_numeric, errors='coerce').sum(axis=1)
            df_cdi = df_data[['Trust_code', 'CDI_total_cases']]
        else:
            return pd.DataFrame()
    
    df_cdi = df_cdi.dropna(subset=['CDI_total_cases'])
    df_cdi['Trust_code'] = df_cdi['Trust_code'].astype(str).str.strip()
    return df_cdi[['Trust_code', 'CDI_total_cases']]

# Tests complets toutes années
for annee, fichier in [
    ('2022-23', rf'{DATA}\Table-6-CDI-march-2023.ods'),
    ('2021-22', rf'{DATA}\Table_6_CDI_march_2022.ods'),
    ('2019-20', rf'{DATA}\Table_6_CDI_march_2020.ods'),
    ('2018-19', rf'{DATA}\Table_2_Monthly_CDI_2P_April_2018.ods'),
    ('2016-17', rf'{DATA}\Table_2_Monthly_CDI_2P_March_2017.ods'),
    ('2015-16', rf'{DATA}\Table_2_Monthly_CDI_2P_March_2016.ods'),
    ('2014-15', rf'{DATA}\Table_2_Monthly_CDI_2P_Mar_2015.xlsx'),
    ('2013-14', rf'{DATA}\Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls'),
    ('2012-13', rf'{DATA}\Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls'),
]:
    df_t = charger_cdi(fichier, annee)
    print(f"{annee} → {len(df_t)} trusts")

2022-23 → 135 trusts
2021-22 → 138 trusts
2019-20 → 139 trusts
2018-19 → 151 trusts
2016-17 → 154 trusts
2015-16 → 155 trusts
2014-15 → 158 trusts
2013-14 → 160 trusts
2012-13 → 160 trusts


In [93]:
# ── FONCTION 4 — Chargement Hospital Providers ──────────
def charger_providers(path):
    """
    Charge le fichier Hospital Providers et retourne :
    Trust_code, total_admissions, A2_dms_jours, A9_taux_transfert
    Fonctionne pour tous les formats (2012 → 2025)
    """
    # Trouver le bon nom d'onglet
    xl = pd.ExcelFile(path)
    sheet_name = None
    for s in xl.sheet_names:
        if 'provider' in s.lower() or 'hospital' in s.lower():
            sheet_name = s
            break
    if sheet_name is None:
        sheet_name = xl.sheet_names[0]
    
    print(f"  Onglet utilisé : {sheet_name}")
    
    df_raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    
    # Chercher ligne header
    header_row = None
    for i in range(5, 25):
        row = df_raw.iloc[i, :].tolist()
        if any('Hospital provider code' in str(v) for v in row):
            header_row = i
            break
    if header_row is None:
        header_row = 9

    # Extraire données depuis header+1
    df_data = df_raw.iloc[header_row+1:, :].copy()
    df_data.columns = range(len(df_data.columns))

    # Trust code = col 0, admissions = col 8
    # DMS = col 18, Other discharge = col 49
    df_prov = pd.DataFrame({
        'Trust_code'      : df_data[0].astype(str).str.strip(),
        'total_admissions': pd.to_numeric(df_data[8],  errors='coerce'),
        'A2_dms_jours'    : pd.to_numeric(df_data[18], errors='coerce'),
        'other_discharge' : pd.to_numeric(df_data[49], errors='coerce'),
    })

    # Garder trusts NHS (code 3 caractères commençant par R)
    df_prov = df_prov[
        df_prov['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['total_admissions', 'A2_dms_jours'])

    # Calculer A9
    df_prov['A9_taux_transfert'] = round(
        df_prov['other_discharge'] / df_prov['total_admissions'] * 100, 2
    )

    return df_prov[['Trust_code', 'total_admissions',
                    'A2_dms_jours', 'A9_taux_transfert']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2024-25-tab.xlsx'),
    ('2022-23', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2022-23-tab_v2.xlsx'),
    ('2017-18', rf'{DATA}\hosp-epis-stat-admi-prov-2017-18-tab.xlsx'),
    ('2016-17', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2016-17-tab.xlsx'),
    ('2012-13', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2012-13-tab.xlsx'),
]:
    df_t = charger_providers(fichier)
    print(f"{annee} → {len(df_t)} trusts | DMS moy: {df_t['A2_dms_jours'].mean():.1f}j\n")

  Onglet utilisé : Hospital Providers
2024-25 → 153 trusts | DMS moy: 16.7j

  Onglet utilisé : Hospital Providers
2022-23 → 160 trusts | DMS moy: 16.5j

  Onglet utilisé : Hospital Providers
2017-18 → 185 trusts | DMS moy: 15.9j

  Onglet utilisé : Hospital Providers
2016-17 → 189 trusts | DMS moy: 17.7j

  Onglet utilisé : Hospital providers


KeyError: 49

In [95]:
# Voir nombre colonnes et structure 2012-13
df_raw_2013 = pd.read_excel(
    rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2012-13-tab.xlsx',
    sheet_name='Hospital providers',
    header=None
)
print(f"Shape 2012-13 : {df_raw_2013.shape}")

# Chercher header
for i in range(5, 20):
    row = df_raw_2013.iloc[i, :].tolist()
    if any('Hospital provider code' in str(v) for v in row):
        print(f"Header ligne {i}")
        # Voir toutes colonnes
        for j, val in enumerate(df_raw_2013.iloc[i, :].tolist()):
            if str(val) != 'nan':
                print(f"  Col {j} : {str(val)[:50]}")
        break

Shape 2012-13 : (521, 42)
Header ligne 16
  Col 0 : Hospital provider code and description†
  Col 2 : Finished consultant episodes
  Col 3 : Admissions
  Col 4 : Male
  Col 5 : Female
  Col 6 : Gender unknown
  Col 7 : Emergency
  Col 8 : Waiting list
  Col 9 : Planned
  Col 10 : Other admission method
  Col 11 : Mean time waited
  Col 12 : Median time waited
  Col 13 : Mean length of stay
  Col 14 : Median length of stay
  Col 15 : Mean age
  Col 16 : Age 0
  Col 17 : Age 1-4
  Col 18 : Age 5-9
  Col 19 : Age 10-14
  Col 20 : Age 15
  Col 21 : Age 16
  Col 22 : Age 17
  Col 23 : Age 18
  Col 24 : Age 19
  Col 25 : Age 20-24
  Col 26 : Age 25-29
  Col 27 : Age 30-34
  Col 28 : Age 35-39
  Col 29 : Age 40-44
  Col 30 : Age 45-49
  Col 31 : Age 50-54
  Col 32 : Age 55-59
  Col 33 : Age 60-64
  Col 34 : Age 65-69
  Col 35 : Age 70-74
  Col 36 : Age 75-79
  Col 37 : Age 80-84
  Col 38 : Age 85-89
  Col 39 : Age 90+
  Col 40 : Day case
  Col 41 : FCE bed days


In [97]:
def charger_providers(path):
    """
    Charge le fichier Hospital Providers et retourne :
    Trust_code, total_admissions, A2_dms_jours, A9_taux_transfert
    Fonctionne pour tous les formats (2012 → 2025)
    """
    xl = pd.ExcelFile(path)
    sheet_name = None
    for s in xl.sheet_names:
        if 'provider' in s.lower() or 'hospital' in s.lower():
            sheet_name = s
            break
    if sheet_name is None:
        sheet_name = xl.sheet_names[0]

    df_raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    # Chercher ligne header
    header_row = None
    for i in range(5, 25):
        row = df_raw.iloc[i, :].tolist()
        if any('Hospital provider code' in str(v) for v in row):
            header_row = i
            break
    if header_row is None:
        header_row = 9

    df_data = df_raw.iloc[header_row+1:, :].copy()
    df_data.columns = range(len(df_data.columns))
    
    nb_cols = len(df_data.columns)
    print(f"  Onglet: {sheet_name} | Colonnes: {nb_cols} | Header: {header_row}")

    # ── Format récent (50 colonnes) 2017 → 2025 ──
    if nb_cols >= 50:
        df_prov = pd.DataFrame({
            'Trust_code'      : df_data[0].astype(str).str.strip(),
            'total_admissions': pd.to_numeric(df_data[8],  errors='coerce'),
            'A2_dms_jours'    : pd.to_numeric(df_data[18], errors='coerce'),
            'other_discharge' : pd.to_numeric(df_data[49], errors='coerce'),
        })

    # ── Format ancien (42 colonnes) 2012 → 2016 ──
    else:
        df_prov = pd.DataFrame({
            'Trust_code'      : df_data[0].astype(str).str.strip(),
            'total_admissions': pd.to_numeric(df_data[3],  errors='coerce'),
            'A2_dms_jours'    : pd.to_numeric(df_data[13], errors='coerce'),
            'other_discharge' : pd.to_numeric(df_data[10], errors='coerce'),
        })

    # Garder trusts NHS
    df_prov = df_prov[
        df_prov['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['total_admissions', 'A2_dms_jours'])

    df_prov['A9_taux_transfert'] = round(
        df_prov['other_discharge'] / df_prov['total_admissions'] * 100, 2
    )

    return df_prov[['Trust_code', 'total_admissions',
                    'A2_dms_jours', 'A9_taux_transfert']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2024-25-tab.xlsx'),
    ('2022-23', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2022-23-tab_v2.xlsx'),
    ('2017-18', rf'{DATA}\hosp-epis-stat-admi-prov-2017-18-tab.xlsx'),
    ('2016-17', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2016-17-tab.xlsx'),
    ('2015-16', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2015-16-tab.xlsx'),
    ('2014-15', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2014-15-tab.xlsx'),
    ('2013-14', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2013-14-tab.xlsx'),
    ('2012-13', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2012-13-tab.xlsx'),
]:
    df_t = charger_providers(fichier)
    print(f"{annee} → {len(df_t)} trusts | DMS moy: {df_t['A2_dms_jours'].mean():.1f}j\n")

  Onglet: Hospital Providers | Colonnes: 50 | Header: 14
2024-25 → 153 trusts | DMS moy: 16.7j

  Onglet: Hospital Providers | Colonnes: 50 | Header: 16
2022-23 → 160 trusts | DMS moy: 16.5j

  Onglet: Hospital Providers | Colonnes: 50 | Header: 9
2017-18 → 185 trusts | DMS moy: 15.9j

  Onglet: Hospital Providers | Colonnes: 50 | Header: 9
2016-17 → 189 trusts | DMS moy: 17.7j

  Onglet: Hospital Providers | Colonnes: 50 | Header: 9
2015-16 → 196 trusts | DMS moy: 18.4j

  Onglet: Hospital Providers | Colonnes: 50 | Header: 9
2014-15 → 201 trusts | DMS moy: 27.4j

  Onglet: Hospital Providers | Colonnes: 45 | Header: 18
2013-14 → 206 trusts | DMS moy: 30.0j

  Onglet: Hospital providers | Colonnes: 42 | Header: 16
2012-13 → 204 trusts | DMS moy: 27.7j



In [99]:
# ── FONCTION 5 — Chargement AEQI ────────────────────────
def charger_aeqi(path):
    """
    Charge le fichier AEQI et retourne :
    Trust_code, A4_temps_attente_median (minutes)
    Gère tous les formats (2012 → 2025)
    """
    
    # ── Format récent CSV (2021+) ──
    if path.endswith('.csv'):
        df = pd.read_csv(path)
        df.columns = [c.upper().strip() for c in df.columns]
        
        # Filtrer TOTAL_TIME_MEDIAN
        df_filtered = df[
            (df['MEASURE_NAME'] == 'TOTAL_TIME_MEDIAN') &
            (df['ORG_CODE'] != 'ENG')
        ].copy()
        
        # Garder une ligne par trust (dernière période)
        if 'ATTENDANCE_MONTH' in df.columns:
            df_filtered = df_filtered.sort_values('ATTENDANCE_MONTH')
            df_filtered = df_filtered.groupby('ORG_CODE').last().reset_index()
        
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_filtered['ORG_CODE'].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_filtered['MEASURE_VALUE'], errors='coerce'),
        })
    
    # ── Format récent ECDS xlsx (2021-22) ──
    elif 'ECDS' in os.path.basename(path).upper():
        df = pd.read_excel(path, header=None)
        # Chercher colonne ORG_CODE et TOTAL_TIME_MEDIAN
        for i in range(0, 10):
            row = df.iloc[i, :].tolist()
            if any('ORG_CODE' in str(v).upper() for v in row):
                header_row = i
                break
        else:
            header_row = 0
        df.columns = df.iloc[header_row, :].tolist()
        df = df.iloc[header_row+1:, :].copy()
        df_filtered = df[df['MEASURE_NAME'] == 'TOTAL_TIME_MEDIAN']
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_filtered['ORG_CODE'].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_filtered['MEASURE_VALUE'], errors='coerce'),
        })
    
    # ── Format ancien xlsx/xls (2012-2020) ──
    else:
        if path.endswith('.xls'):
            df_raw = pd.read_excel(path, sheet_name='Data', header=None,
                                   engine='xlrd')
            # Format 2012-13 : Col 5=code, Col 59=median total
            col_code   = 5
            col_median = 59
            header_r   = 9
        else:
            df_sheets = pd.read_excel(path, sheet_name=None, header=None)
            sheet = list(df_sheets.keys())[0]
            df_raw = df_sheets[sheet]
            # Format 2016-2020 : Col 0=code, Col 47=median total
            col_code   = 0
            col_median = 47
            header_r   = 4

        df_data = df_raw.iloc[header_r+1:, :].copy()
        df_data.columns = range(len(df_data.columns))
        
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_data[col_code].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_data[col_median], errors='coerce'),
        })
    
    # Garder trusts NHS
    df_aeqi = df_aeqi[
        df_aeqi['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['A4_temps_attente_min'])
    
    return df_aeqi[['Trust_code', 'A4_temps_attente_min']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\aeqi_open_data_2025_03.csv'),
    ('2022-23', rf'{DATA}\aeqi_open_data_2023_03.csv'),
    ('2021-22', rf'{DATA}\ECDS_CQI_Open_Data_M12.csv'),
    ('2019-20', rf'{DATA}\AE_CQI_M12.xls'),
    ('2018-19', rf'{DATA}\2018-19 M12 AE CQI Suppressed.xlsx'),
    ('2016-17', rf'{DATA}\ae_cqi_2016_17_m131.xlsx'),
    ('2012-13', rf'{DATA}\prov-ae-qual-indi-eng-mar13-tab1.xls'),
]:
    df_t = charger_aeqi(fichier)
    print(f"{annee} → {len(df_t)} trusts | Median moy: {df_t['A4_temps_attente_min'].mean():.0f} min")

2024-25 → 146 trusts | Median moy: 170 min
2022-23 → 147 trusts | Median moy: 186 min


KeyError: 'MEASURE_NAME'

In [101]:
# Voir colonnes ECDS 2021-22
df_ecds = pd.read_csv(rf'{DATA}\ECDS_CQI_Open_Data_M12.csv')
print("Colonnes ECDS 2021-22 :")
for col in df_ecds.columns:
    print(f"  - {col}")
print(f"\nExemple données :")
print(df_ecds.head(3).to_string())

Colonnes ECDS 2021-22 :
  - Report Name
  - Organisation Code
  - Organisation Name
  - Month
  - Patient Type
  - Value 1
  - Value 2
  - Value 3
  - Value 4

Exemple données :
            Report Name Organisation Code Organisation Name   Month  Patient Type  Value 1  Value 2 Value 3 Value 4
0  1. MSitAE Comparison               ENG           ENGLAND  Apr-20  All Patients   836534   858281    97.5     NaN
1  1. MSitAE Comparison               ENG           ENGLAND  May-20  All Patients  1154354  1185604    97.4     NaN
2  1. MSitAE Comparison               ENG           ENGLAND  Jun-20  All Patients  1285038  1324360      97     NaN


In [103]:
print("Report Names disponibles :")
print(df_ecds['Report Name'].unique())
print(f"\nMois disponibles : {df_ecds['Month'].unique()[:5]}")

# Chercher le rapport avec temps médian
df_median = df_ecds[df_ecds['Report Name'].str.contains('Time|Median|Wait', 
                                                          case=False, na=False)]
print(f"\nRapports liés au temps :")
print(df_median['Report Name'].unique())

Report Names disponibles :
<ArrowStringArray>
[           '1. MSitAE Comparison',         '1b. Missing from MSitAE',
 '2. Left Department before Treat',   '3. Time to Initial Assessment',
            '4. Time to Treatment',            '5. Total Time in A&E',
      '6. Unplanned Reattendances']
Length: 7, dtype: str

Mois disponibles : <ArrowStringArray>
['Apr-20', 'May-20', 'Jun-20', 'Jul-20', 'Aug-20']
Length: 5, dtype: str

Rapports liés au temps :
<ArrowStringArray>
['3. Time to Initial Assessment', '4. Time to Treatment',
 '5. Total Time in A&E']
Length: 3, dtype: str


In [105]:
df_total_time = df_ecds[df_ecds['Report Name'] == '5. Total Time in A&E']
print(f"Shape : {df_total_time.shape}")
print(f"\nPatient Types : {df_total_time['Patient Type'].unique()}")
print(f"\nExemple données Mars 2022 :")
df_mars = df_total_time[df_total_time['Month'] == 'Mar-22']
print(df_mars.head(5).to_string())

Shape : (5852, 9)

Patient Types : <ArrowStringArray>
['Admitted', 'All Patients', 'Non-Admitted']
Length: 3, dtype: str

Exemple données Mars 2022 :
Empty DataFrame
Columns: [Report Name, Organisation Code, Organisation Name, Month, Patient Type, Value 1, Value 2, Value 3, Value 4]
Index: []


In [107]:
print("Tous les mois disponibles :")
print(df_total_time['Month'].unique())

Tous les mois disponibles :
<ArrowStringArray>
['Apr-20', 'May-20', 'Jun-20', 'Jul-20', 'Aug-20', 'Sep-20', 'Oct-20',
 'Nov-20', 'Dec-20', 'Jan-21', 'Feb-21', 'Mar-21']
Length: 12, dtype: str


In [109]:
# Voir colonnes ECDS_CQI_Open_Data_M12.xlsx
df_ecds_xlsx = pd.read_excel(rf'{DATA}\ECDS_CQI_Open_Data_M12.xlsx', 
                              sheet_name=None)
print("Onglets :", list(df_ecds_xlsx.keys()))

# Voir premier onglet
sheet = list(df_ecds_xlsx.keys())[0]
df_s = df_ecds_xlsx[sheet]
print(f"\nShape : {df_s.shape}")
print(f"\nColonnes : {list(df_s.columns)}")
print(f"\nExemple :")
print(df_s.head(3).to_string())

Onglets : ['Sheet1']

Shape : (16500, 10)

Colonnes : ['RN', 'Report Name', 'Organisation Code', 'Organisation Name', 'Month', 'Patient Type', 'Value 1', 'Value 2', 'Value 3', 'Value 4']

Exemple :
   RN           Report Name Organisation Code Organisation Name      Month  Patient Type  Value 1  Value 2 Value 3 Value 4
0   1  1. MSitAE Comparison               ENG           ENGLAND 2021-04-01  All Patients  1708276  1768167    96.6     NaN
1   2  1. MSitAE Comparison               ENG           ENGLAND 2021-05-01  All Patients  1890982  1961437    96.4     NaN
2   3  1. MSitAE Comparison               ENG           ENGLAND 2021-06-01  All Patients  1985546  2054145    96.7     NaN


In [111]:
df_total_2122 = df_ecds_xlsx['Sheet1'][
    df_ecds_xlsx['Sheet1']['Report Name'] == '5. Total Time in A&E'
]
print(f"Mois disponibles : {df_total_2122['Month'].unique()}")
print(f"\nPatient Types : {df_total_2122['Patient Type'].unique()}")

# Filtrer All Patients Mars 2022
df_mars_2122 = df_total_2122[
    (df_total_2122['Month'] == '2022-03-01') &
    (df_total_2122['Patient Type'] == 'All Patients') &
    (df_total_2122['Organisation Code'] != 'ENG')
]
print(f"\nTrusts Mars 2022 : {len(df_mars_2122)}")
print(f"\nExemple (Value 1=Median?) :")
print(df_mars_2122.head(5)[['Organisation Code', 'Value 1', 'Value 2', 'Value 3']].to_string())

Mois disponibles : <DatetimeArray>
['2021-04-01 00:00:00', '2021-05-01 00:00:00', '2021-06-01 00:00:00',
 '2021-07-01 00:00:00', '2021-08-01 00:00:00', '2021-09-01 00:00:00',
 '2021-10-01 00:00:00', '2021-11-01 00:00:00', '2021-12-01 00:00:00',
 '2022-01-01 00:00:00', '2022-02-01 00:00:00', '2022-03-01 00:00:00']
Length: 12, dtype: datetime64[us]

Patient Types : <ArrowStringArray>
['Admitted', 'All Patients', 'Non-Admitted']
Length: 3, dtype: str

Trusts Mars 2022 : 171

Exemple (Value 1=Median?) :
      Organisation Code Value 1 Value 2 Value 3
14119             AAH01       *       *       *
14122             AD903     101     417    1437
14125             AD904     108   328.4    1433
14128             AD906     134     418    1426
14131             AD913     111     296    1422


In [113]:
def charger_aeqi(path):
    """
    Charge le fichier AEQI et retourne :
    Trust_code, A4_temps_attente_median (minutes)
    Gère tous les formats (2012 → 2025)
    """
    
    # ── Format récent CSV (2022+) ──
    if path.endswith('.csv') and 'ECDS' not in os.path.basename(path).upper():
        df = pd.read_csv(path)
        df.columns = [c.upper().strip() for c in df.columns]
        df_filtered = df[
            (df['MEASURE_NAME'] == 'TOTAL_TIME_MEDIAN') &
            (df['ORG_CODE'] != 'ENG')
        ].copy()
        if 'ATTENDANCE_MONTH' in df_filtered.columns:
            df_filtered = df_filtered.sort_values('ATTENDANCE_MONTH')
            df_filtered = df_filtered.groupby('ORG_CODE').last().reset_index()
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_filtered['ORG_CODE'].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_filtered['MEASURE_VALUE'], errors='coerce'),
        })

    # ── Format ECDS xlsx (2021-22) ──
    elif path.endswith('.xlsx') and 'ECDS' in os.path.basename(path).upper():
        df = pd.read_excel(path, sheet_name='Sheet1')
        df_filtered = df[
            (df['Report Name'] == '5. Total Time in A&E') &
            (df['Patient Type'] == 'All Patients') &
            (df['Organisation Code'] != 'ENG')
        ].copy()
        # Prendre Mars (dernier mois)
        df_filtered = df_filtered.sort_values('Month')
        df_filtered = df_filtered.groupby('Organisation Code').last().reset_index()
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_filtered['Organisation Code'].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_filtered['Value 1'], errors='coerce'),
        })

    # ── Format ancien xls (2012-2013) ──
    elif path.endswith('.xls'):
        df_raw = pd.read_excel(path, sheet_name='Data', 
                               header=None, engine='xlrd')
        col_code   = 5
        col_median = 59
        header_r   = 9
        df_data = df_raw.iloc[header_r+1:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_data[col_code].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_data[col_median], errors='coerce'),
        })

    # ── Format ancien xlsx (2014-2020) ──
    else:
        df_sheets = pd.read_excel(path, sheet_name=None, header=None)
        sheet = list(df_sheets.keys())[0]
        df_raw = df_sheets[sheet]
        col_code   = 0
        col_median = 47
        header_r   = 4
        df_data = df_raw.iloc[header_r+1:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_data[col_code].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_data[col_median], errors='coerce'),
        })

    # Garder trusts NHS
    df_aeqi = df_aeqi[
        df_aeqi['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['A4_temps_attente_min'])

    return df_aeqi[['Trust_code', 'A4_temps_attente_min']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\aeqi_open_data_2025_03.csv'),
    ('2022-23', rf'{DATA}\aeqi_open_data_2023_03.csv'),
    ('2021-22', rf'{DATA}\ECDS_CQI_Open_Data_M12.xlsx'),
    ('2019-20', rf'{DATA}\AE_CQI_M12.xls'),
    ('2018-19', rf'{DATA}\2018-19 M12 AE CQI Suppressed.xlsx'),
    ('2016-17', rf'{DATA}\ae_cqi_2016_17_m131.xlsx'),
    ('2015-16', rf'{DATA}\prov-ae-qual-indi-eng-march 2016-tab1.xls'),
    ('2014-15', rf'{DATA}\prov-ae-qual-indi-eng-march 2015-tab1.xls'),
    ('2013-14', rf'{DATA}\prov-ae-qual-indi-eng-march 2014-tab1.xls'),
    ('2012-13', rf'{DATA}\prov-ae-qual-indi-eng-mar13-tab1.xls'),
]:
    df_t = charger_aeqi(fichier)
    print(f"{annee} → {len(df_t)} trusts | Median moy: {df_t['A4_temps_attente_min'].mean():.0f} min")

2024-25 → 146 trusts | Median moy: 170 min
2022-23 → 147 trusts | Median moy: 186 min
2021-22 → 143 trusts | Median moy: 195 min


ValueError: Worksheet named 'Data' not found

In [115]:
# Voir onglets des fichiers xls anciens
for annee, fichier in [
    ('2019-20', rf'{DATA}\AE_CQI_M12.xls'),
    ('2015-16', rf'{DATA}\prov-ae-qual-indi-eng-march 2016-tab1.xls'),
    ('2014-15', rf'{DATA}\prov-ae-qual-indi-eng-march 2015-tab1.xls'),
    ('2013-14', rf'{DATA}\prov-ae-qual-indi-eng-march 2014-tab1.xls'),
    ('2012-13', rf'{DATA}\prov-ae-qual-indi-eng-mar13-tab1.xls'),
]:
    xl = pd.ExcelFile(fichier, engine='xlrd')
    print(f"{annee} → Onglets : {xl.sheet_names}")

2019-20 → Onglets : ['HESF_AE_CQI_NHSDigital']
2015-16 → Onglets : ['HESF_AE_CQI_dev_suppressed_M13E']
2014-15 → Onglets : ['HESF_AE_CQI_dev_suppressed']
2013-14 → Onglets : ['HESF_AE_CQI_dev_suppressed']
2012-13 → Onglets : ['Data']


In [117]:
def charger_aeqi(path):
    """
    Charge le fichier AEQI et retourne :
    Trust_code, A4_temps_attente_median (minutes)
    Gère tous les formats (2012 → 2025)
    """
    
    # ── Format récent CSV (2022+) ──
    if path.endswith('.csv') and 'ECDS' not in os.path.basename(path).upper():
        df = pd.read_csv(path)
        df.columns = [c.upper().strip() for c in df.columns]
        df_filtered = df[
            (df['MEASURE_NAME'] == 'TOTAL_TIME_MEDIAN') &
            (df['ORG_CODE'] != 'ENG')
        ].copy()
        if 'ATTENDANCE_MONTH' in df_filtered.columns:
            df_filtered = df_filtered.sort_values('ATTENDANCE_MONTH')
            df_filtered = df_filtered.groupby('ORG_CODE').last().reset_index()
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_filtered['ORG_CODE'].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_filtered['MEASURE_VALUE'], errors='coerce'),
        })

    # ── Format ECDS xlsx (2021-22) ──
    elif path.endswith('.xlsx') and 'ECDS' in os.path.basename(path).upper():
        df = pd.read_excel(path, sheet_name='Sheet1')
        df_filtered = df[
            (df['Report Name'] == '5. Total Time in A&E') &
            (df['Patient Type'] == 'All Patients') &
            (df['Organisation Code'] != 'ENG')
        ].copy()
        df_filtered = df_filtered.sort_values('Month')
        df_filtered = df_filtered.groupby('Organisation Code').last().reset_index()
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_filtered['Organisation Code'].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_filtered['Value 1'], errors='coerce'),
        })

    # ── Format xls avec onglet Data (2012-13) ──
    elif path.endswith('.xls') and 'mar13' in os.path.basename(path).lower():
        df_raw = pd.read_excel(path, sheet_name='Data',
                               header=None, engine='xlrd')
        df_data = df_raw.iloc[10:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_data[5].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(df_data[59], errors='coerce'),
        })

    # ── Format xls/xlsx ancien (2013-2020) ──
    else:
        if path.endswith('.xls'):
            xl = pd.ExcelFile(path, engine='xlrd')
            sheet = xl.sheet_names[0]
            df_raw = pd.read_excel(path, sheet_name=sheet,
                                   header=None, engine='xlrd')
        else:
            df_sheets = pd.read_excel(path, sheet_name=None, header=None)
            sheet = list(df_sheets.keys())[0]
            df_raw = df_sheets[sheet]

        # Chercher ligne header (contient 'Organisation Code')
        header_r = 4
        for i in range(0, 10):
            row = df_raw.iloc[i, :].tolist()
            if any('Organisation Code' in str(v) for v in row):
                header_r = i
                break

        nb_cols = len(df_raw.columns)
        # Col médian total = 47 si 50 colonnes, 59 si 62 colonnes
        col_median = 47 if nb_cols <= 55 else 59
        col_code   = 0

        df_data = df_raw.iloc[header_r+1:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_data[col_code].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(
                df_data[col_median], errors='coerce'),
        })

    # Garder trusts NHS
    df_aeqi = df_aeqi[
        df_aeqi['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['A4_temps_attente_min'])

    return df_aeqi[['Trust_code', 'A4_temps_attente_min']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\aeqi_open_data_2025_03.csv'),
    ('2022-23', rf'{DATA}\aeqi_open_data_2023_03.csv'),
    ('2021-22', rf'{DATA}\ECDS_CQI_Open_Data_M12.xlsx'),
    ('2019-20', rf'{DATA}\AE_CQI_M12.xls'),
    ('2018-19', rf'{DATA}\2018-19 M12 AE CQI Suppressed.xlsx'),
    ('2016-17', rf'{DATA}\ae_cqi_2016_17_m131.xlsx'),
    ('2015-16', rf'{DATA}\prov-ae-qual-indi-eng-march 2016-tab1.xls'),
    ('2014-15', rf'{DATA}\prov-ae-qual-indi-eng-march 2015-tab1.xls'),
    ('2013-14', rf'{DATA}\prov-ae-qual-indi-eng-march 2014-tab1.xls'),
    ('2012-13', rf'{DATA}\prov-ae-qual-indi-eng-mar13-tab1.xls'),
]:
    df_t = charger_aeqi(fichier)
    print(f"{annee} → {len(df_t)} trusts | Median moy: {df_t['A4_temps_attente_min'].mean():.0f} min")

2024-25 → 146 trusts | Median moy: 170 min
2022-23 → 147 trusts | Median moy: 186 min
2021-22 → 143 trusts | Median moy: 195 min
2019-20 → 140 trusts | Median moy: 161 min
2018-19 → 133 trusts | Median moy: 156 min
2016-17 → 143 trusts | Median moy: 145 min
2015-16 → 0 trusts | Median moy: nan min
2014-15 → 0 trusts | Median moy: nan min
2013-14 → 0 trusts | Median moy: nan min
2012-13 → 170 trusts | Median moy: 134 min


In [119]:
# Débugger 2013-14
df_raw_2014 = pd.read_excel(
    rf'{DATA}\prov-ae-qual-indi-eng-march 2014-tab1.xls',
    sheet_name='HESF_AE_CQI_dev_suppressed',
    header=None, engine='xlrd'
)
print(f"Shape 2013-14 : {df_raw_2014.shape}")
print(f"Nb colonnes : {len(df_raw_2014.columns)}")

# Chercher header
for i in range(0, 10):
    row = df_raw_2014.iloc[i, :].tolist()
    non_nan = [str(v) for v in row if str(v) != 'nan']
    if len(non_nan) > 2:
        print(f"Ligne {i} : {non_nan[:5]}")
        
# Voir lignes 5-10
print(f"\nLignes 4-8 colonnes 0-5 :")
print(df_raw_2014.iloc[4:8, :6].to_string())

Shape 2013-14 : (288, 53)
Nb colonnes : 53
Ligne 4 : ['DATA COVERAGE', 'Left department before being seen for treatment\n(Data quality)', 'Left department before being seen for treatment\n(Performance)', 'Re-attendance\n(Data quality)', 'Re-attendance\n(Performance)']
Ligne 5 : ['Area Team Code', 'Area Team Name', 'Organisation Code', 'Organisation Name', '\xa0Total number of attendances in A&E HES \n\n(excluding planned follow-up attendances and attendances where the attendance category was unknown)\xa0']
Ligne 6 : ['ENG', 'ENGLAND', '1610243', '1940136', '1718396']
Ligne 8 : ['Q48', 'MERSEYSIDE ', 'REM', 'AINTREE UNIVERSITY HOSPITAL NHS FOUNDATION TRUST', '9457']
Ligne 9 : ['Q52', 'WEST YORKSHIRE ', 'RCF', 'AIREDALE NHS FOUNDATION TRUST', '4559']

Lignes 4-8 colonnes 0-5 :
                0               1                  2                  3                                                                                                                                               

In [121]:
# Trouver col médian pour 2013-14 (53 colonnes)
print("Ligne 4 (titres sections) 2013-14 :")
for i, val in enumerate(df_raw_2014.iloc[4, :].tolist()):
    if str(val) != 'nan' and 'Total Time' in str(val):
        print(f"  Col {i} : {val}")

print("\nLigne 5 (headers) 2013-14 colonnes 40-53 :")
for i, val in enumerate(df_raw_2014.iloc[5, 40:].tolist()):
    if str(val) != 'nan':
        print(f"  Col {i+40} : {str(val)[:60]}")

Ligne 4 (titres sections) 2013-14 :
  Col 39 : Total Time in A&E (minutes)

[Admitted patients only]
  Col 42 : Total Time in A&E (minutes)

[Non-admitted patients only]
  Col 45 : Total Time in A&E
(Data quality)

[All patients]
  Col 50 : Total Time in A&E
(Performance; minutes)

[All patients]

Ligne 5 (headers) 2013-14 colonnes 40-53 :
  Col 40 : 95th Percentile [see footnote 4]
  Col 41 : Longest wait
 [see footnote 5]
  Col 42 : Median
  Col 43 : 95th Percentile [see footnote 4]
  Col 44 : Longest wait
 [see footnote 5]
  Col 45 : Total number of attendances in A&E HES 

(excluding planned 
  Col 46 : Attendances with unknown duration to departure
  Col 47 : % of attendances with unknown duration to departure times
  Col 48 : Did more than 25% of attendances have a duration to departur
  Col 49 : % attendances with a departure time of exactly midnight (00:
  Col 50 : Median
  Col 51 : 95th Percentile [see footnote 4]
  Col 52 : Longest wait
 [see footnote 5]


In [123]:
# Corriger charger_aeqi pour gérer tous les formats xls anciens
def charger_aeqi(path):
    """
    Charge le fichier AEQI et retourne :
    Trust_code, A4_temps_attente_median (minutes)
    """
    
    # ── Format récent CSV (2022+) ──
    if path.endswith('.csv') and 'ECDS' not in os.path.basename(path).upper():
        df = pd.read_csv(path)
        df.columns = [c.upper().strip() for c in df.columns]
        df_filtered = df[
            (df['MEASURE_NAME'] == 'TOTAL_TIME_MEDIAN') &
            (df['ORG_CODE'] != 'ENG')
        ].copy()
        if 'ATTENDANCE_MONTH' in df_filtered.columns:
            df_filtered = df_filtered.sort_values('ATTENDANCE_MONTH')
            df_filtered = df_filtered.groupby('ORG_CODE').last().reset_index()
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_filtered['ORG_CODE'].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(df_filtered['MEASURE_VALUE'], errors='coerce'),
        })

    # ── Format ECDS xlsx (2021-22) ──
    elif path.endswith('.xlsx') and 'ECDS' in os.path.basename(path).upper():
        df = pd.read_excel(path, sheet_name='Sheet1')
        df_filtered = df[
            (df['Report Name'] == '5. Total Time in A&E') &
            (df['Patient Type'] == 'All Patients') &
            (df['Organisation Code'] != 'ENG')
        ].copy()
        df_filtered = df_filtered.sort_values('Month')
        df_filtered = df_filtered.groupby('Organisation Code').last().reset_index()
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_filtered['Organisation Code'].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(df_filtered['Value 1'], errors='coerce'),
        })

    # ── Format xls/xlsx ancien (2012-2020) ──
    else:
        if path.endswith('.xls'):
            xl   = pd.ExcelFile(path, engine='xlrd')
            sheet = xl.sheet_names[0]
            df_raw = pd.read_excel(path, sheet_name=sheet,
                                   header=None, engine='xlrd')
        else:
            df_sheets = pd.read_excel(path, sheet_name=None, header=None)
            sheet     = list(df_sheets.keys())[0]
            df_raw    = df_sheets[sheet]

        nb_cols = len(df_raw.columns)
        print(f"  {os.path.basename(path)} → {nb_cols} colonnes")

        # Chercher ligne header
        header_r  = 4
        col_code  = 0
        for i in range(0, 10):
            row = df_raw.iloc[i, :].tolist()
            if any('Organisation Code' in str(v) for v in row):
                header_r = i
                # Trouver col du code organisation
                for j, v in enumerate(row):
                    if 'Organisation Code' in str(v):
                        col_code = j
                        break
                break

        # Trouver col médian Total Time All Patients
        # Chercher dans la ligne au-dessus du header
        col_median = 47  # défaut
        for i in range(0, header_r):
            row = df_raw.iloc[i, :].tolist()
            for j, v in enumerate(row):
                if 'Total Time' in str(v) and 'All' in str(v) and 'Performance' in str(v):
                    # Le Median est dans la ligne header à cette position
                    col_median = j
                    break

        print(f"  col_code={col_code}, col_median={col_median}, header={header_r}")

        df_data = df_raw.iloc[header_r+1:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_aeqi = pd.DataFrame({
            'Trust_code'          : df_data[col_code].astype(str).str.strip(),
            'A4_temps_attente_min': pd.to_numeric(df_data[col_median], errors='coerce'),
        })

    # Garder trusts NHS
    df_aeqi = df_aeqi[
        df_aeqi['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['A4_temps_attente_min'])

    return df_aeqi[['Trust_code', 'A4_temps_attente_min']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\aeqi_open_data_2025_03.csv'),
    ('2022-23', rf'{DATA}\aeqi_open_data_2023_03.csv'),
    ('2021-22', rf'{DATA}\ECDS_CQI_Open_Data_M12.xlsx'),
    ('2019-20', rf'{DATA}\AE_CQI_M12.xls'),
    ('2018-19', rf'{DATA}\2018-19 M12 AE CQI Suppressed.xlsx'),
    ('2016-17', rf'{DATA}\ae_cqi_2016_17_m131.xlsx'),
    ('2015-16', rf'{DATA}\prov-ae-qual-indi-eng-march 2016-tab1.xls'),
    ('2014-15', rf'{DATA}\prov-ae-qual-indi-eng-march 2015-tab1.xls'),
    ('2013-14', rf'{DATA}\prov-ae-qual-indi-eng-march 2014-tab1.xls'),
    ('2012-13', rf'{DATA}\prov-ae-qual-indi-eng-mar13-tab1.xls'),
]:
    df_t = charger_aeqi(fichier)
    print(f"{annee} → {len(df_t)} trusts | Median moy: {df_t['A4_temps_attente_min'].mean():.0f} min\n")

2024-25 → 146 trusts | Median moy: 170 min

2022-23 → 147 trusts | Median moy: 186 min

2021-22 → 143 trusts | Median moy: 195 min

  AE_CQI_M12.xls → 50 colonnes
  col_code=0, col_median=47, header=4
2019-20 → 140 trusts | Median moy: 161 min

  2018-19 M12 AE CQI Suppressed.xlsx → 50 colonnes
  col_code=0, col_median=47, header=4
2018-19 → 133 trusts | Median moy: 156 min

  ae_cqi_2016_17_m131.xlsx → 50 colonnes
  col_code=0, col_median=47, header=4
2016-17 → 143 trusts | Median moy: 145 min

  prov-ae-qual-indi-eng-march 2016-tab1.xls → 54 colonnes
  col_code=2, col_median=51, header=5
2015-16 → 142 trusts | Median moy: 155 min

  prov-ae-qual-indi-eng-march 2015-tab1.xls → 53 colonnes
  col_code=2, col_median=50, header=5
2014-15 → 167 trusts | Median moy: 139 min

  prov-ae-qual-indi-eng-march 2014-tab1.xls → 53 colonnes
  col_code=2, col_median=50, header=5
2013-14 → 169 trusts | Median moy: 132 min

  prov-ae-qual-indi-eng-mar13-tab1.xls → 62 colonnes
  col_code=5, col_median=5

In [125]:
# ── FONCTION 6 — Chargement Workforce ───────────────────
def charger_workforce(path):
    """
    Charge le fichier NHS Workforce et retourne :
    Trust_code, doctors, nurses
    Gère tous les formats (2013 → 2025)
    """
    xl = pd.ExcelFile(path)
    print(f"  Onglets : {xl.sheet_names[:4]}")
    
    # ── Format récent (2018+) — onglet '2. NHSE, Org & SG - HC' ──
    if any('NHSE' in s for s in xl.sheet_names):
        sheet = [s for s in xl.sheet_names if 'NHSE' in s and 'HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        # Headers ligne 5
        df_data = df_raw.iloc[6:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_wf = pd.DataFrame({
            'Trust_code': df_data[5].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[19], errors='coerce'),
        })

    # ── Format 2017 — onglet '1. HEE Org Main Staff Gp HC' ──
    elif any('HEE Org Main' in s for s in xl.sheet_names):
        sheet = [s for s in xl.sheet_names if 'HEE Org Main' in s and 'HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_wf = pd.DataFrame({
            'Trust_code': df_data[2].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
        })

    # ── Format ancien (2013-2016) — onglets séparés médecins/infirmiers ──
    else:
        # Médecins
        sheet_doc = [s for s in xl.sheet_names 
                    if 'dr' in s.lower() or 'doc' in s.lower()][0]
        df_doc = pd.read_excel(path, sheet_name=sheet_doc, header=None)
        df_doc_data = df_doc.iloc[7:, :].copy()
        df_doc_data.columns = range(len(df_doc_data.columns))
        df_doctors = pd.DataFrame({
            'Trust_code': df_doc_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_doc_data[4], errors='coerce'),
        })

        # Infirmiers
        sheet_nur = [s for s in xl.sheet_names 
                    if 'nonmed' in s.lower() or 'non med' in s.lower()
                    or 'nur' in s.lower()][0]
        df_nur = pd.read_excel(path, sheet_name=sheet_nur, header=None)
        df_nur_data = df_nur.iloc[7:, :].copy()
        df_nur_data.columns = range(len(df_nur_data.columns))
        df_nurses = pd.DataFrame({
            'Trust_code': df_nur_data[3].astype(str).str.strip(),
            'nurses'    : pd.to_numeric(df_nur_data[4], errors='coerce'),
        })

        df_wf = df_doctors.merge(df_nurses, on='Trust_code', how='outer')

    # Garder trusts NHS
    df_wf = df_wf[
        df_wf['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['doctors', 'nurses'])

    return df_wf[['Trust_code', 'doctors', 'nurses']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx'),
    ('2022-23', rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx'),
    ('2019-20', rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx'),
    ('2018-19', rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx'),
    ('2017-18', rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx'),
    ('2016-17', rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx'),
    ('2015-16', rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx'),
    ('2014-15', rf'{DATA}\nhs-work-stat-mar-2015-org-tab.xls'),
    ('2013-14', rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls'),
]:
    df_t = charger_workforce(fichier)
    print(f"{annee} → {len(df_t)} trusts | Médecins moy: {df_t['doctors'].mean():.0f} | Infirmiers moy: {df_t['nurses'].mean():.0f}\n")

  Onglets : ['Title sheet', '1. England', '2. NHSE, Org & SG - HC', '3. NHSE, Org & SG - FTE']
2024-25 → 201 trusts | Médecins moy: 783 | Infirmiers moy: 1995

  Onglets : ['Title sheet', '1. England', '2. NHSE, Org & SG - HC', '3. NHSE, Org & SG - FTE']
2022-23 → 208 trusts | Médecins moy: 683 | Infirmiers moy: 1755

  Onglets : ['Title sheet', '1. England', '2. NHSE, Org & SG - HC', '3. NHSE, Org & SG - FTE']
2019-20 → 0 trusts | Médecins moy: nan | Infirmiers moy: nan

  Onglets : ['Title sheet', '1. England', '2. HEE, Org & SG - HC', '3. HEE, Org & SG - FTE']


IndexError: list index out of range

In [127]:
for annee, fichier in [
    ('2019-20', rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx'),
    ('2018-19', rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx'),
    ('2017-18', rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx'),
    ('2016-17', rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx'),
    ('2015-16', rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx'),
    ('2014-15', rf'{DATA}\nhs-work-stat-mar-2015-org-tab.xls'),
    ('2013-14', rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls'),
]:
    xl = pd.ExcelFile(fichier)
    print(f"{annee} → {xl.sheet_names}")

2019-20 → ['Title sheet', '1. England', '2. NHSE, Org & SG - HC', '3. NHSE, Org & SG - FTE', '4. All Staff, NHSE & Org - HC', '5. All Staff, NHSE & Org - FTE']
2018-19 → ['Title sheet', '1. England', '2. HEE, Org & SG - HC', '3. HEE, Org & SG - FTE', '4. All Staff, HEE & Org - HC', '5. All Staff, HEE & Org - FTE']
2017-18 → ['Title sheet', '1. HEE Org Main Staff Gp HC', '2. HEE Org Main Staff Gp FTE', '3. All Staff by HEE Org HC', '4. All Staff by HEE Org FTE']
2016-17 → ['Title sheet', '1. HEE Org Main Staff Gp HC', '2. HEE Org Main Staff Gp FTE', '3. All Staff by HEE Org HC', '4. All Staff by HEE Org FTE']
2015-16 → ['Contents', '1. HEE Org Main Staff Gp HC', '2. HEE Org Main Staff Gp FTE ', '3. All Staff by HEE Org HC', '4. All Staff by HEE Org FTE']
2014-15 → ['Contents', '1. Drs by HEE,Org, Grade HC', '2. Drs by HEE,Org, Grade FTE ', '3.NonMed by HEE,Org,StGrp HC', '4.NonMed by HEE,Org,StGrp FTE', '5. All Staff by HEE,Org HC', '6. All Staff by HEE,Org FTE']
2013-14 → ['Contents', 

In [129]:
def charger_workforce(path):
    """
    Charge le fichier NHS Workforce et retourne :
    Trust_code, doctors, nurses
    Gère tous les formats (2013 → 2025)
    """
    xl = pd.ExcelFile(path)
    sheets = xl.sheet_names

    # ── Format récent (2019+) — onglet contient 'Org & SG - HC' ──
    if any('Org & SG - HC' in s for s in sheets):
        sheet = [s for s in sheets if 'Org & SG - HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        
        # Chercher ligne header
        header_r = 5
        for i in range(0, 10):
            row = df_raw.iloc[i, :].tolist()
            if any('Organisation code' in str(v) or 
                   'Organisation Code' in str(v) for v in row):
                header_r = i
                break
        
        df_data = df_raw.iloc[header_r+1:, :].copy()
        df_data.columns = range(len(df_data.columns))
        
        # Chercher colonnes doctors et nurses
        header_row = df_raw.iloc[header_r, :].tolist()
        col_code = 5
        col_doc  = 8
        col_nur  = 19
        for j, v in enumerate(header_row):
            if 'Organisation code' in str(v) or 'Organisation Code' in str(v):
                col_code = j
            if 'HCHS Doctor' in str(v) or 'HCHS doctor' in str(v):
                col_doc = j
            if 'Nurses' in str(v) or 'nurses' in str(v):
                col_nur = j
        
        df_wf = pd.DataFrame({
            'Trust_code': df_data[col_code].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[col_doc], errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[col_nur], errors='coerce'),
        })

    # ── Format 2016-2018 — onglet 'HEE Org Main Staff Gp HC' ──
    elif any('HEE Org Main' in s for s in sheets):
        sheet = [s for s in sheets if 'HEE Org Main' in s and 'HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        
        # Chercher header
        header_r = 6
        for i in range(0, 12):
            row = df_raw.iloc[i, :].tolist()
            non_nan = [str(v) for v in row if str(v) != 'nan']
            if len(non_nan) > 3:
                header_r = i
                break
        
        df_data = df_raw.iloc[header_r+1:, :].copy()
        df_data.columns = range(len(df_data.columns))
        
        # Chercher col code trust
        header_row = df_raw.iloc[header_r, :].tolist()
        col_code = 2
        col_doc  = 8
        col_nur  = 20
        for j, v in enumerate(header_row):
            if 'Org Code' in str(v) or 'Organisation Code' in str(v):
                col_code = j
            if 'HCHS doctor' in str(v) or 'HCHS Doctor' in str(v):
                col_doc = j
            if 'Nurses' in str(v):
                col_nur = j
        
        df_wf = pd.DataFrame({
            'Trust_code': df_data[col_code].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[col_doc], errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[col_nur], errors='coerce'),
        })

    # ── Format ancien (2013-2015) — onglets séparés ──
    else:
        # Médecins
        sheet_doc = [s for s in sheets if 'Drs' in s and 'HC' in s][0]
        df_doc = pd.read_excel(path, sheet_name=sheet_doc, header=None)
        
        header_r = 4
        for i in range(0, 10):
            row = df_doc.iloc[i, :].tolist()
            non_nan = [str(v) for v in row if str(v) != 'nan']
            if len(non_nan) > 3:
                header_r = i
                break
        
        df_doc_data = df_doc.iloc[header_r+1:, :].copy()
        df_doc_data.columns = range(len(df_doc_data.columns))
        df_doctors = pd.DataFrame({
            'Trust_code': df_doc_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_doc_data[4], errors='coerce'),
        })

        # Infirmiers
        sheet_nur = [s for s in sheets 
                    if 'NonMed' in s or 'Nonmed' in s or 'nonmed' in s][0]
        df_nur = pd.read_excel(path, sheet_name=sheet_nur, header=None)
        
        df_nur_data = df_nur.iloc[header_r+1:, :].copy()
        df_nur_data.columns = range(len(df_nur_data.columns))
        df_nurses = pd.DataFrame({
            'Trust_code': df_nur_data[3].astype(str).str.strip(),
            'nurses'    : pd.to_numeric(df_nur_data[4], errors='coerce'),
        })

        df_wf = df_doctors.merge(df_nurses, on='Trust_code', how='outer')

    # Garder trusts NHS
    df_wf = df_wf[
        df_wf['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['doctors', 'nurses'])

    return df_wf[['Trust_code', 'doctors', 'nurses']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx'),
    ('2022-23', rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx'),
    ('2019-20', rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx'),
    ('2018-19', rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx'),
    ('2017-18', rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx'),
    ('2016-17', rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx'),
    ('2015-16', rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx'),
    ('2014-15', rf'{DATA}\nhs-work-stat-mar-2015-org-tab.xls'),
    ('2013-14', rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls'),
]:
    df_t = charger_workforce(fichier)
    print(f"{annee} → {len(df_t)} trusts | Médecins: {df_t['doctors'].mean():.0f} | Infirmiers: {df_t['nurses'].mean():.0f}")

2024-25 → 201 trusts | Médecins: 5 | Infirmiers: 1700
2022-23 → 208 trusts | Médecins: 5 | Infirmiers: 1605
2019-20 → 0 trusts | Médecins: nan | Infirmiers: nan
2018-19 → 0 trusts | Médecins: nan | Infirmiers: nan
2017-18 → 0 trusts | Médecins: nan | Infirmiers: nan
2016-17 → 0 trusts | Médecins: nan | Infirmiers: nan
2015-16 → 0 trusts | Médecins: nan | Infirmiers: nan
2014-15 → 227 trusts | Médecins: 484 | Infirmiers: 4426
2013-14 → 233 trusts | Médecins: 463 | Infirmiers: 4261


In [131]:
# Débug format récent 2024-25
df_raw = pd.read_excel(
    rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx',
    sheet_name='2. NHSE, Org & SG - HC', header=None)

print("Lignes 4-8 colonnes 4-10 :")
print(df_raw.iloc[4:8, 4:11].to_string())

print("\nHeader ligne 5 colonnes 5-10 :")
for j, v in enumerate(df_raw.iloc[5, :].tolist()):
    if str(v) != 'nan' and j >= 4 and j <= 22:
        print(f"  Col {j} : {str(v)[:40]}")

Lignes 4-8 colonnes 4-10 :
                  4                  5        6                                        7             8           9                     10
4                NaN                NaN      NaN                                      NaN           NaN         NaN                   NaN
5  Organisation name  Organisation code    Total  Professionally qualified clinical staff  HCHS Doctors  Consultant  Associate Specialist
6                NaN                NaN      NaN                                      NaN           NaN         NaN                   NaN
7                NaN                NaN  1544417                                   822180        158324       63996                  3063

Header ligne 5 colonnes 5-10 :
  Col 4 : Organisation name
  Col 5 : Organisation code
  Col 6 : Total
  Col 7 : Professionally qualified clinical staff
  Col 8 : HCHS Doctors
  Col 9 : Consultant
  Col 10 : Associate Specialist
  Col 11 : Specialty Doctor
  Col 12 : Staff Grade
  

In [133]:
print("Lignes 8-12 colonnes 4-10 :")
print(df_raw.iloc[8:13, 4:11].to_string())

Lignes 8-12 colonnes 4-10 :
                                              4    5       6       7      8      9    10
8                                            NaN  NaN     NaN     NaN    NaN    NaN  NaN
9                                            NaN  NaN  260434  152561  33440  13735  419
10                                           NaN  NaN   65387   38467   8021   3436   72
11     Guy's and St Thomas' NHS Foundation Trust  RJ1   23347   14100   3336   1554   18
12  King's College Hospital NHS Foundation Trust  RJZ   13864    9521   2689   1096   27


In [135]:
# Débug workforce 2024-25
df_raw = pd.read_excel(
    rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx',
    sheet_name='2. NHSE, Org & SG - HC', header=None)

header_r = 5
df_data = df_raw.iloc[header_r+1:, :].copy()
df_data.columns = range(len(df_data.columns))

# Voir ce que contient col 5, 8, 19 pour les trusts
df_test = pd.DataFrame({
    'Trust_code': df_data[5].astype(str).str.strip(),
    'doctors'   : pd.to_numeric(df_data[8], errors='coerce'),
    'nurses'    : pd.to_numeric(df_data[19], errors='coerce'),
})

# Filtrer trusts NHS
df_test_r = df_test[df_test['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')]
print(f"Trusts NHS : {len(df_test_r)}")
print(f"\nExemple :")
print(df_test_r.head(5).to_string())


Trusts NHS : 201

Exemple :
   Trust_code  doctors  nurses
11        RJ1   3336.0  6783.0
12        RJZ   2689.0  4558.0
13        RJ2   1101.0  2442.0
14        RRU      8.0    81.0
16        RPG    239.0  1484.0


In [139]:
# Débug 2019-20
df_raw = pd.read_excel(
    rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx',
    sheet_name='2. NHSE, Org & SG - HC', header=None)

print(f"Shape : {df_raw.shape}")
print(f"\nLignes 4-10 colonnes 0-8 :")
print(df_raw.iloc[4:10, :9].to_string())

# Débug 2013-14 onglets
xl = pd.ExcelFile(rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls')
print(f"\nOnglets 2013-14 : {xl.sheet_names}")

Shape : (376, 37)

Lignes 4-10 colonnes 0-8 :
                         0        1    2    3        4   5                                        6   7             8
4                      NaN      NaN  NaN  NaN      NaN NaN                                      NaN NaN           NaN
5  March 2020 monthly data      NaN  NaN  NaN      NaN NaN                                      NaN NaN           NaN
6                      NaN      NaN  NaN  NaN    Total NaN  Professionally qualified clinical staff NaN  HCHS Doctors
7                      NaN      NaN  NaN  NaN      NaN NaN                                      NaN NaN           NaN
8                      NaN  England  NaN  NaN  1289793 NaN                                   675973 NaN        126523
9                      NaN      NaN  NaN  NaN      NaN NaN                                      NaN NaN           NaN

Onglets 2013-14 : ['Contents', '1. Drs by HEE,Org, Grade HC', '2. Drs by HEE,Org, Grade FTE ', '3.NonMed by HEE,Org,Staffgr', '

In [141]:
# Voir colonnes complètes 2019-20
print("Ligne 6 (headers) 2019-20 :")
for j, v in enumerate(df_raw.iloc[6, :].tolist()):
    if str(v) != 'nan':
        print(f"  Col {j} : {str(v)[:50]}")

print("\nLignes 10-14 colonnes 0-8 :")
print(df_raw.iloc[10:14, :9].to_string())

Ligne 6 (headers) 2019-20 :
  Col 4 : Total
  Col 6 : Professionally qualified clinical staff
  Col 8 : HCHS Doctors
  Col 9 : Consultant
  Col 10 : Associate Specialist
  Col 11 : Specialty Doctor
  Col 12 : Staff Grade
  Col 13 : Specialty Registrar
  Col 14 : Core Training
  Col 15 : Foundation Doctor Year 2
  Col 16 : Foundation Doctor Year 1
  Col 17 : Hospital Practitioner / Clinical Assistant
  Col 18 : Other and Local HCHS Doctor Grades
  Col 20 : Nurses & health visitors
  Col 21 : Midwives
  Col 22 : Ambulance staff
  Col 23 : Scientific, therapeutic & technical staff
  Col 25 : Support to clinical staff
  Col 26 : Support to doctors, nurses & midwives
  Col 27 : Support to ambulance staff
  Col 28 : Support to ST&T staff
  Col 30 : NHS infrastructure support
  Col 31 : Central functions
  Col 32 : Hotel, property & estates
  Col 33 : Senior managers
  Col 34 : Managers
  Col 36 : Other staff or those with unknown classification

Lignes 10-14 colonnes 0-8 :
      0       1   

In [143]:
def charger_workforce(path):
    """
    Charge le fichier NHS Workforce et retourne :
    Trust_code, doctors, nurses
    Gère tous les formats (2013 → 2025)
    """
    xl = pd.ExcelFile(path)
    sheets = xl.sheet_names

    # ── Format récent (2022+) — onglet 'NHSE, Org & SG - HC' ──
    if any('NHSE, Org & SG - HC' in s for s in sheets):
        sheet = [s for s in sheets if 'NHSE, Org & SG - HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        df_data = df_raw.iloc[6:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_wf = pd.DataFrame({
            'Trust_code': df_data[5].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[19], errors='coerce'),
        })

    # ── Format 2018-2020 — onglet 'HEE, Org & SG - HC' ──
    elif any('HEE, Org & SG - HC' in s for s in sheets):
        sheet = [s for s in sheets if 'HEE, Org & SG - HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_wf = pd.DataFrame({
            'Trust_code': df_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
        })

    # ── Format 2016-2018 — onglet 'HEE Org Main Staff Gp HC' ──
    elif any('HEE Org Main' in s for s in sheets):
        sheet = [s for s in sheets if 'HEE Org Main' in s and 'HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_wf = pd.DataFrame({
            'Trust_code': df_data[2].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
        })

    # ── Format ancien (2013-2015) — onglets séparés ──
    else:
        # Médecins
        sheet_doc = [s for s in sheets if 'Drs' in s and 'HC' in s][0]
        df_doc = pd.read_excel(path, sheet_name=sheet_doc, header=None)
        df_doc_data = df_doc.iloc[7:, :].copy()
        df_doc_data.columns = range(len(df_doc_data.columns))
        df_doctors = pd.DataFrame({
            'Trust_code': df_doc_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_doc_data[4], errors='coerce'),
        })

        # Infirmiers — chercher onglet NonMed
        sheet_nur = [s for s in sheets if 'NonMed' in s or 'nonmed' in s.lower()][0]
        df_nur = pd.read_excel(path, sheet_name=sheet_nur, header=None)
        df_nur_data = df_nur.iloc[7:, :].copy()
        df_nur_data.columns = range(len(df_nur_data.columns))
        df_nurses = pd.DataFrame({
            'Trust_code': df_nur_data[3].astype(str).str.strip(),
            'nurses'    : pd.to_numeric(df_nur_data[4], errors='coerce'),
        })

        df_wf = df_doctors.merge(df_nurses, on='Trust_code', how='outer')

    # Garder trusts NHS
    df_wf = df_wf[
        df_wf['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['doctors', 'nurses'])

    return df_wf[['Trust_code', 'doctors', 'nurses']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx'),
    ('2022-23', rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx'),
    ('2019-20', rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx'),
    ('2018-19', rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx'),
    ('2017-18', rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx'),
    ('2016-17', rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx'),
    ('2015-16', rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx'),
    ('2014-15', rf'{DATA}\nhs-work-stat-mar-2015-org-tab.xls'),
    ('2013-14', rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls'),
]:
    df_t = charger_workforce(fichier)
    print(f"{annee} → {len(df_t)} trusts | Médecins: {df_t['doctors'].mean():.0f} | Infirmiers: {df_t['nurses'].mean():.0f}")

2024-25 → 201 trusts | Médecins: 783 | Infirmiers: 1995
2022-23 → 208 trusts | Médecins: 683 | Infirmiers: 1755
2019-20 → 0 trusts | Médecins: nan | Infirmiers: nan
2018-19 → 222 trusts | Médecins: 535 | Infirmiers: 1442
2017-18 → 0 trusts | Médecins: nan | Infirmiers: nan
2016-17 → 0 trusts | Médecins: nan | Infirmiers: nan
2015-16 → 0 trusts | Médecins: nan | Infirmiers: nan
2014-15 → 227 trusts | Médecins: 484 | Infirmiers: 4426
2013-14 → 233 trusts | Médecins: 463 | Infirmiers: 4261


In [145]:
# Débug 2019-20 — voir colonnes exactes
df_raw_2020 = pd.read_excel(
    rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx',
    sheet_name='2. NHSE, Org & SG - HC', header=None)

print(f"Shape 2019-20 : {df_raw_2020.shape}")
print(f"\nLignes 5-8 colonnes 0-6 :")
print(df_raw_2020.iloc[5:9, :7].to_string())

# Débug 2017-18
df_raw_2018 = pd.read_excel(
    rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx',
    sheet_name='1. HEE Org Main Staff Gp HC', header=None)

print(f"\nShape 2017-18 : {df_raw_2018.shape}")
print(f"\nLignes 5-10 colonnes 0-6 :")
print(df_raw_2018.iloc[5:10, :7].to_string())

Shape 2019-20 : (376, 37)

Lignes 5-8 colonnes 0-6 :
                         0        1    2    3        4   5                                        6
5  March 2020 monthly data      NaN  NaN  NaN      NaN NaN                                      NaN
6                      NaN      NaN  NaN  NaN    Total NaN  Professionally qualified clinical staff
7                      NaN      NaN  NaN  NaN      NaN NaN                                      NaN
8                      NaN  England  NaN  NaN  1289793 NaN                                   675973

Shape 2017-18 : (467, 37)

Lignes 5-10 colonnes 0-6 :
                         0        1    2    3        4   5                                        6
5  March 2018 monthly data      NaN  NaN  NaN      NaN NaN                                      NaN
6                      NaN      NaN  NaN  NaN    Total NaN  Professionally qualified clinical staff
7                      NaN      NaN  NaN  NaN      NaN NaN                                  

In [147]:
def charger_workforce(path):
    """
    Charge le fichier NHS Workforce et retourne :
    Trust_code, doctors, nurses
    Gère tous les formats (2013 → 2025)
    """
    xl = pd.ExcelFile(path)
    sheets = xl.sheet_names

    # ── Format récent (2022+) — onglet 'NHSE, Org & SG - HC' ──
    if any('NHSE, Org & SG - HC' in s for s in sheets):
        sheet = [s for s in sheets if 'NHSE, Org & SG - HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        nb_cols = df_raw.shape[1]
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))

        # Format récent 33 cols (2022+) vs format intermédiaire 37 cols (2019-20)
        if nb_cols >= 33 and nb_cols < 37:
            # 2022+ : col 5=code, col 8=doctors, col 19=nurses
            df_wf = pd.DataFrame({
                'Trust_code': df_data[5].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[19], errors='coerce'),
            })
        else:
            # 2019-20 : col 3=code, col 8=doctors, col 20=nurses
            df_wf = pd.DataFrame({
                'Trust_code': df_data[3].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
            })

    # ── Format 2018-19 — onglet 'HEE, Org & SG - HC' ──
    elif any('HEE, Org & SG - HC' in s for s in sheets):
        sheet = [s for s in sheets if 'HEE, Org & SG - HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_wf = pd.DataFrame({
            'Trust_code': df_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
        })

    # ── Format 2016-2018 — onglet 'HEE Org Main Staff Gp HC' ──
    elif any('HEE Org Main' in s for s in sheets):
        sheet = [s for s in sheets if 'HEE Org Main' in s and 'HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        nb_cols = df_raw.shape[1]
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))

        # 2017-18 : col 3=code, col 8=doctors, col 20=nurses
        # 2015-16, 2016-17 : col 2=code, col 8=doctors, col 20=nurses
        if nb_cols >= 37:
            col_code = 3
        else:
            col_code = 2

        df_wf = pd.DataFrame({
            'Trust_code': df_data[col_code].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
        })

    # ── Format ancien (2013-2015) — onglets séparés ──
    else:
        sheet_doc = [s for s in sheets if 'Drs' in s and 'HC' in s][0]
        df_doc = pd.read_excel(path, sheet_name=sheet_doc, header=None)
        df_doc_data = df_doc.iloc[7:, :].copy()
        df_doc_data.columns = range(len(df_doc_data.columns))
        df_doctors = pd.DataFrame({
            'Trust_code': df_doc_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_doc_data[4], errors='coerce'),
        })

        sheet_nur = [s for s in sheets if 'NonMed' in s][0]
        df_nur = pd.read_excel(path, sheet_name=sheet_nur, header=None)
        df_nur_data = df_nur.iloc[7:, :].copy()
        df_nur_data.columns = range(len(df_nur_data.columns))
        df_nurses = pd.DataFrame({
            'Trust_code': df_nur_data[3].astype(str).str.strip(),
            'nurses'    : pd.to_numeric(df_nur_data[4], errors='coerce'),
        })
        df_wf = df_doctors.merge(df_nurses, on='Trust_code', how='outer')

    # Garder trusts NHS
    df_wf = df_wf[
        df_wf['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['doctors', 'nurses'])

    return df_wf[['Trust_code', 'doctors', 'nurses']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx'),
    ('2022-23', rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx'),
    ('2019-20', rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx'),
    ('2018-19', rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx'),
    ('2017-18', rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx'),
    ('2016-17', rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx'),
    ('2015-16', rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx'),
    ('2014-15', rf'{DATA}\nhs-work-stat-mar-2015-org-tab.xls'),
    ('2013-14', rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls'),
]:
    df_t = charger_workforce(fichier)
    print(f"{annee} → {len(df_t)} trusts | Médecins: {df_t['doctors'].mean():.0f} | Infirmiers: {df_t['nurses'].mean():.0f}")

2024-25 → 201 trusts | Médecins: 783 | Infirmiers: 1995
2022-23 → 208 trusts | Médecins: 683 | Infirmiers: 1755
2019-20 → 214 trusts | Médecins: 588 | Infirmiers: 1552
2018-19 → 222 trusts | Médecins: 535 | Infirmiers: 1442
2017-18 → 227 trusts | Médecins: 511 | Infirmiers: 1390
2016-17 → 229 trusts | Médecins: 493 | Infirmiers: 1378
2015-16 → 0 trusts | Médecins: nan | Infirmiers: nan
2014-15 → 227 trusts | Médecins: 484 | Infirmiers: 4426
2013-14 → 233 trusts | Médecins: 463 | Infirmiers: 4261


In [149]:
# Débug 2015-16
df_raw_2016 = pd.read_excel(
    rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx',
    sheet_name='1. HEE Org Main Staff Gp HC', header=None)

print(f"Shape 2015-16 : {df_raw_2016.shape}")
print(f"\nLignes 5-10 colonnes 0-6 :")
print(df_raw_2016.iloc[5:10, :7].to_string())

print(f"\nLignes 10-14 colonnes 0-6 :")
print(df_raw_2016.iloc[10:14, :7].to_string())

Shape 2015-16 : (492, 38)

Lignes 5-10 colonnes 0-6 :
         0                               1                                                 2    3        4   5       6
5  England                             NaN                                               NaN  NaN  1164471 NaN  622717
6      NaN                             NaN                                               NaN  NaN      NaN NaN     NaN
7    YDF21  Health Education East Midlands                                               NaN  NaN    89750 NaN   47561
8      NaN                             NaN  Chesterfield Royal Hospital NHS Foundation Trust  RFS     3815 NaN    1857
9      NaN                             NaN     Derby Teaching Hospitals NHS Foundation Trust  RTG     7879 NaN    4295

Lignes 10-14 colonnes 0-6 :
      0    1                                                          2    3     4   5     6
10  NaN  NaN  Derbyshire Community Health Services NHS Foundation Trust  RY8  4418 NaN  2110
11  NaN  NaN     

In [151]:
print("Ligne 4 (headers) 2015-16 :")
for j, v in enumerate(df_raw_2016.iloc[4, :].tolist()):
    if str(v) != 'nan':
        print(f"  Col {j} : {str(v)[:50]}")

Ligne 4 (headers) 2015-16 :


In [155]:
print("Ligne 3 (headers complets) 2015-16 :")
for j, v in enumerate(df_raw_2016.iloc[3, :].tolist()):
    if str(v) != 'nan':
        print(f"  Col {j} : {str(v)[:50]}")

Ligne 3 (headers complets) 2015-16 :
  Col 4 : Total
  Col 6 : Professionally qualified clinical staff
  Col 8 : HCHS doctors
  Col 9 : Consultant (including Directors of Public Health) 
  Col 10 : Associate Specialist 
  Col 11 : Specialty Doctor
  Col 12 : Staff Grade
  Col 13 : Specialty Registrar 
  Col 14 : Core Medical Training
  Col 15 : Core Dental Training
  Col 16 : Foundation Doctor Year 2
  Col 17 : Foundation Doctor Year 1
  Col 18 : Hospital Practitioner / Clinical Assistant
  Col 19 : Other and Unknown HCHS Doctor Grades
  Col 21 : Nurses & health visitors
  Col 22 : Midwives
  Col 23 : Ambulance staff
  Col 24 : Scientific, therapeutic & technical staff
  Col 26 : Support to clinical staff
  Col 27 : Support to doctors, nurses & midwives
  Col 28 : Support to ambulance staff
  Col 29 : Support to ST&T staff
  Col 31 : NHS infrastructure support
  Col 32 : Central functions
  Col 33 : Hotel, property & estates
  Col 34 : Senior managers
  Col 35 : Managers
  Col 37 : Oth

In [157]:
def charger_workforce(path):
    xl = pd.ExcelFile(path)
    sheets = xl.sheet_names

    # ── Format récent (2022+) — onglet 'NHSE, Org & SG - HC' ──
    if any('NHSE, Org & SG - HC' in s for s in sheets):
        sheet = [s for s in sheets if 'NHSE, Org & SG - HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        nb_cols = df_raw.shape[1]
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))
        if nb_cols < 37:
            # 2022+ : 33 cols
            df_wf = pd.DataFrame({
                'Trust_code': df_data[5].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[19], errors='coerce'),
            })
        else:
            # 2019-20 : 37 cols
            df_wf = pd.DataFrame({
                'Trust_code': df_data[3].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
            })

    # ── Format 2018-19 — onglet 'HEE, Org & SG - HC' ──
    elif any('HEE, Org & SG - HC' in s for s in sheets):
        sheet = [s for s in sheets if 'HEE, Org & SG - HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_wf = pd.DataFrame({
            'Trust_code': df_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
        })

    # ── Format 2015-2018 — onglet 'HEE Org Main Staff Gp HC' ──
    elif any('HEE Org Main' in s for s in sheets):
        sheet = [s for s in sheets if 'HEE Org Main' in s and 'HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        nb_cols = df_raw.shape[1]
        df_data = df_raw.iloc[8:, :].copy()
        df_data.columns = range(len(df_data.columns))
        
        if nb_cols >= 38:
            # 2015-16 : 38 cols → col 3=code, col 8=doctors, col 21=nurses
            df_wf = pd.DataFrame({
                'Trust_code': df_data[3].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[21], errors='coerce'),
            })
        elif nb_cols >= 37:
            # 2017-18 : 37 cols → col 3=code, col 8=doctors, col 20=nurses
            df_wf = pd.DataFrame({
                'Trust_code': df_data[3].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
            })
        else:
            # 2016-17 : col 2=code, col 8=doctors, col 20=nurses
            df_wf = pd.DataFrame({
                'Trust_code': df_data[2].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
            })

    # ── Format ancien (2013-2015) — onglets séparés ──
    else:
        sheet_doc = [s for s in sheets if 'Drs' in s and 'HC' in s][0]
        df_doc = pd.read_excel(path, sheet_name=sheet_doc, header=None)
        df_doc_data = df_doc.iloc[7:, :].copy()
        df_doc_data.columns = range(len(df_doc_data.columns))
        df_doctors = pd.DataFrame({
            'Trust_code': df_doc_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_doc_data[4], errors='coerce'),
        })
        sheet_nur = [s for s in sheets if 'NonMed' in s][0]
        df_nur = pd.read_excel(path, sheet_name=sheet_nur, header=None)
        df_nur_data = df_nur.iloc[7:, :].copy()
        df_nur_data.columns = range(len(df_nur_data.columns))
        df_nurses = pd.DataFrame({
            'Trust_code': df_nur_data[3].astype(str).str.strip(),
            'nurses'    : pd.to_numeric(df_nur_data[4], errors='coerce'),
        })
        df_wf = df_doctors.merge(df_nurses, on='Trust_code', how='outer')

    # Garder trusts NHS
    df_wf = df_wf[
        df_wf['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['doctors', 'nurses'])

    return df_wf[['Trust_code', 'doctors', 'nurses']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx'),
    ('2022-23', rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx'),
    ('2019-20', rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx'),
    ('2018-19', rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx'),
    ('2017-18', rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx'),
    ('2016-17', rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx'),
    ('2015-16', rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx'),
    ('2014-15', rf'{DATA}\nhs-work-stat-mar-2015-org-tab.xls'),
    ('2013-14', rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls'),
]:
    df_t = charger_workforce(fichier)
    print(f"{annee} → {len(df_t)} trusts | Médecins: {df_t['doctors'].mean():.0f} | Infirmiers: {df_t['nurses'].mean():.0f}")

2024-25 → 201 trusts | Médecins: 783 | Infirmiers: 1995
2022-23 → 208 trusts | Médecins: 683 | Infirmiers: 1755
2019-20 → 214 trusts | Médecins: 588 | Infirmiers: 1552
2018-19 → 222 trusts | Médecins: 535 | Infirmiers: 1442
2017-18 → 227 trusts | Médecins: 511 | Infirmiers: 1390
2016-17 → 229 trusts | Médecins: 493 | Infirmiers: 1378
2015-16 → 229 trusts | Médecins: 481 | Infirmiers: 1373
2014-15 → 227 trusts | Médecins: 484 | Infirmiers: 4426
2013-14 → 233 trusts | Médecins: 463 | Infirmiers: 4261


In [159]:
# ============================================================
# SECTION 9 — FUSION DE TOUTES LES ANNÉES
# ============================================================

def charger_annee(annee, paths):
    """
    Charge et fusionne tous les fichiers pour une année donnée.
    Retourne un DataFrame avec tous les KPIs par trust.
    """
    print(f"\n{'='*50}")
    print(f"Chargement année {annee}...")
    print(f"{'='*50}")
    
    # Charger chaque fichier
    df_beds = charger_beds(paths['beds'])
    print(f"✅ Beds      : {len(df_beds)} trusts")
    
    df_shmi = charger_shmi(paths['shmi'])
    print(f"✅ SHMI      : {len(df_shmi)} trusts")
    
    df_cdi = charger_cdi(paths['cdi'], annee)
    print(f"✅ CDI       : {len(df_cdi)} trusts")
    
    df_prov = charger_providers(paths['prov'])
    print(f"✅ Providers : {len(df_prov)} trusts")
    
    df_aeqi = charger_aeqi(paths['aeqi'])
    print(f"✅ AEQI      : {len(df_aeqi)} trusts")
    
    df_wf = charger_workforce(paths['workforce'])
    print(f"✅ Workforce : {len(df_wf)} trusts")
    
    # Fusionner tous les DataFrames sur Trust_code
    df = df_beds.merge(df_shmi,  on='Trust_code', how='inner')
    df = df.merge(df_cdi,   on='Trust_code', how='inner')
    df = df.merge(df_prov,  on='Trust_code', how='inner')
    df = df.merge(df_aeqi,  on='Trust_code', how='inner')
    df = df.merge(df_wf,    on='Trust_code', how='inner')
    
    # Ajouter colonnes calculées
    df['annee'] = annee
    df['Ratio_Medecin_Lit']       = round(df['doctors'] / df['lits_disponibles'], 3)
    df['Ratio_Infirmier_Lit']     = round(df['nurses']  / df['lits_disponibles'], 3)
    df['Ratio_Infirmier_Medecin'] = round(df['nurses']  / df['doctors'], 3)
    
    print(f"\n✅ Fusion terminée : {len(df)} trusts × {len(df.columns)} colonnes")
    return df

# Test avec 2022-23
paths_2223 = {
    'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2022-23-Final.xlsx',
    'shmi'      : rf'{DATA}\SHMI data files, Apr22-Mar23.zip',
    'cdi'       : rf'{DATA}\Table-6-CDI-march-2023.ods',
    'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2022-23-tab_v2.xlsx',
    'aeqi'      : rf'{DATA}\aeqi_open_data_2023_03.csv',
    'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx',
}

df_2223 = charger_annee('2022-23', paths_2223)
print(f"\nColonnes : {list(df_2223.columns)}")
print(f"\nExemple :")
print(df_2223.head(3).to_string())


Chargement année 2022-23...
✅ Beds      : 189 trusts
✅ SHMI      : 133 trusts
✅ CDI       : 135 trusts
  Onglet: Hospital Providers | Colonnes: 50 | Header: 16
✅ Providers : 160 trusts
✅ AEQI      : 147 trusts
✅ Workforce : 208 trusts

✅ Fusion terminée : 104 trusts × 20 colonnes

Colonnes : ['Trust_code', 'lits_disponibles', 'lits_occupes', 'C2_occupation', 'A8_lits_vacants', 'OBSERVED', 'SPELLS', 'C3_mortalite', 'SHMI_VALUE', 'CDI_total_cases', 'total_admissions', 'A2_dms_jours', 'A9_taux_transfert', 'A4_temps_attente_min', 'doctors', 'nurses', 'annee', 'Ratio_Medecin_Lit', 'Ratio_Infirmier_Lit', 'Ratio_Infirmier_Medecin']

Exemple :
  Trust_code  lits_disponibles  lits_occupes  C2_occupation  A8_lits_vacants  OBSERVED  SPELLS  C3_mortalite  SHMI_VALUE  CDI_total_cases  total_admissions  A2_dms_jours  A9_taux_transfert  A4_temps_attente_min  doctors  nurses    annee  Ratio_Medecin_Lit  Ratio_Infirmier_Lit  Ratio_Infirmier_Medecin
0        R1H       2024.400000   1843.811111         

In [161]:
# ============================================================
# CHEMINS — TOUTES LES ANNÉES
# ============================================================

ANNEES_PATHS = {
    '2012-13': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2012-13-Revised-Final-79659.xls',
        'shmi'      : rf'{DATA}\SHMI data files, Apr12-Mar13.zip',
        'cdi'       : rf'{DATA}\Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2012-13-tab.xlsx',
        'aeqi'      : rf'{DATA}\prov-ae-qual-indi-eng-mar13-tab1.xls',
        'workforce' : rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls',
    },
    '2013-14': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2013-14-Final-Working-Revised-33335.xlsx',
        'shmi'      : rf'{DATA}\SHMI data files, Apr13-Mar14.zip',
        'cdi'       : rf'{DATA}\Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2013-14-tab.xlsx',
        'aeqi'      : rf'{DATA}\prov-ae-qual-indi-eng-march 2014-tab1.xls',
        'workforce' : rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls',
    },
    '2014-15': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2014-15-Revised-Nov15-Final-41885.xlsx',
        'shmi'      : rf'{DATA}\SHMI data files, Apr14-Mar15.zip',
        'cdi'       : rf'{DATA}\Table_2_Monthly_CDI_2P_Mar_2015.xlsx',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2014-15-tab.xlsx',
        'aeqi'      : rf'{DATA}\prov-ae-qual-indi-eng-march 2015-tab1.xls',
        'workforce' : rf'{DATA}\nhs-work-stat-mar-2015-org-tab.xls',
    },
    '2015-16': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2015-16-Revised-Aug16-Final-84398.xlsx',
        'shmi'      : rf'{DATA}\SHMI data files, Apr15-Mar16.zip',
        'cdi'       : rf'{DATA}\Table_2_Monthly_CDI_2P_March_2016.ods',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2015-16-tab.xlsx',
        'aeqi'      : rf'{DATA}\prov-ae-qual-indi-eng-march 2016-tab1.xls',
        'workforce' : rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx',
    },
    '2016-17': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2016-17-Final-98123.xlsx',
        'shmi'      : rf'{DATA}\SHMI data files, Apr16-Mar17.zip',
        'cdi'       : rf'{DATA}\Table_2_Monthly_CDI_2P_March_2017.ods',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2016-17-tab.xlsx',
        'aeqi'      : rf'{DATA}\ae_cqi_2016_17_m131.xlsx',
        'workforce' : rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx',
    },
    '2017-18': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Q4-201718-REVISED-Web_File-Final.xlsx',
        'shmi'      : rf'{DATA}\SHMI data files, Apr17-Mar18.zip',
        'cdi'       : rf'{DATA}\cdi_table_trust_20190425.ods',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-prov-2017-18-tab.xlsx',
        'aeqi'      : rf'{DATA}\HESF_AE_CQI_dev_NHSdigital_M12 suppressed v1.1.xlsx',
        'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx',
    },
    '2018-19': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Final_Q4_2018-19.xlsx',
        'shmi'      : rf'{DATA}\SHMI data files, Apr18-Mar19.zip',
        'cdi'       : rf'{DATA}\Table_2_Monthly_CDI_2P_April_2018.ods',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-prov-2018-19-tab.xlsx',
        'aeqi'      : rf'{DATA}\2018-19 M12 AE CQI Suppressed.xlsx',
        'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx',
    },
    '2019-20': {
        'beds'      : rf'{DATA}\1920-Q4-Beds-Open-Overnight-Web_File-Final-DE5WC.xlsx',
        'shmi'      : rf'{DATA}\SHMI data files, Apr19-Mar20.zip',
        'cdi'       : rf'{DATA}\Table_6_CDI_march_2020.ods',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-prov-2019-20-tab.xlsx',
        'aeqi'      : rf'{DATA}\AE_CQI_M12.xls',
        'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx',
    },
    '2021-22': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2021-22-Final-GHVBN.xlsx',
        'shmi'      : rf'{DATA}\SHMI data files, Apr21-Mar22.zip',
        'cdi'       : rf'{DATA}\Table_6_CDI_march_2022.ods',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2021-22-tab.xlsx',
        'aeqi'      : rf'{DATA}\ECDS_CQI_Open_Data_M12.xlsx',
        'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2022 England and Organisation.xlsx',
    },
    '2022-23': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2022-23-Final.xlsx',
        'shmi'      : rf'{DATA}\SHMI data files, Apr22-Mar23.zip',
        'cdi'       : rf'{DATA}\Table-6-CDI-march-2023.ods',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2022-23-tab_v2.xlsx',
        'aeqi'      : rf'{DATA}\aeqi_open_data_2023_03.csv',
        'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx',
    },
    '2023-24': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2023-24-Final.xlsx',
        'shmi'      : rf'{DATA}\SHMI data, Apr23-Mar24.zip',
        'cdi'       : rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2023-24-tab.xlsx',
        'aeqi'      : rf'{DATA}\aeqi_open_data_2023_03.csv',
        'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2024 England and Organisation.xlsx',
    },
    '2024-25': {
        'beds'      : rf'{DATA}\Beds-Open-Overnight-Web_File-Q4-2024-25.xlsx',
        'shmi'      : rf'{DATA}\SHMI data, Apr24-Mar25.zip',
        'cdi'       : rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods',
        'prov'      : rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2024-25-tab.xlsx',
        'aeqi'      : rf'{DATA}\aeqi_open_data_2025_03.csv',
        'workforce' : rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx',
    },
}

print(f"✅ {len(ANNEES_PATHS)} années configurées !")
print(f"Années : {list(ANNEES_PATHS.keys())}")

✅ 12 années configurées !
Années : ['2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2021-22', '2022-23', '2023-24', '2024-25']


In [163]:
# ============================================================
# CHARGEMENT ET FUSION DE TOUTES LES ANNÉES
# ============================================================

dfs_annees = []
erreurs = []

for annee, paths in ANNEES_PATHS.items():
    try:
        df_annee = charger_annee(annee, paths)
        dfs_annees.append(df_annee)
    except Exception as e:
        print(f"\n❌ ERREUR {annee} : {e}")
        erreurs.append(annee)

print(f"\n{'='*50}")
print(f"RÉSUMÉ CHARGEMENT")
print(f"{'='*50}")
print(f"✅ Années chargées : {len(dfs_annees)}")
print(f"❌ Erreurs        : {erreurs}")

for df in dfs_annees:
    annee = df['annee'].iloc[0]
    print(f"  {annee} → {len(df)} trusts")


Chargement année 2012-13...
✅ Beds      : 229 trusts

❌ ERREUR 2012-13 : 'PROVIDER_CODE'

Chargement année 2013-14...
✅ Beds      : 227 trusts

❌ ERREUR 2013-14 : 'PROVIDER_CODE'

Chargement année 2014-15...
✅ Beds      : 222 trusts
✅ SHMI      : 137 trusts
✅ CDI       : 158 trusts
  Onglet: Hospital Providers | Colonnes: 50 | Header: 9
✅ Providers : 201 trusts
  prov-ae-qual-indi-eng-march 2015-tab1.xls → 53 colonnes
  col_code=2, col_median=50, header=5
✅ AEQI      : 167 trusts
✅ Workforce : 227 trusts

✅ Fusion terminée : 119 trusts × 20 colonnes

Chargement année 2015-16...
✅ Beds      : 215 trusts
✅ SHMI      : 136 trusts
✅ CDI       : 155 trusts
  Onglet: Hospital Providers | Colonnes: 50 | Header: 9
✅ Providers : 196 trusts
  prov-ae-qual-indi-eng-march 2016-tab1.xls → 54 colonnes
  col_code=2, col_median=51, header=5
✅ AEQI      : 142 trusts
✅ Workforce : 229 trusts

✅ Fusion terminée : 118 trusts × 20 colonnes

Chargement année 2016-17...
✅ Beds      : 212 trusts
✅ SHMI      

IndexError: single positional indexer is out-of-bounds

In [165]:
# Voir quel fichier CDI utiliser pour 2023-24 et 2024-25
# et pour 2017-18
import os

print("Fichiers CDI disponibles :")
for f in os.listdir(DATA):
    if 'cdi' in f.lower() or 'CDI' in f or 'difficile' in f.lower() or 'table' in f.lower():
        print(f"  - {f}")

Fichiers CDI disponibles :
  - cdi_table_trust_20190425.ods
  - Clostridium_difficile_monthly_data_April_acute_trusts_table_2.xls
  - NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx
  - Table-6-CDI-march-2023.ods
  - Table_2_Monthly_CDI_2P_April_2018.ods
  - Table_2_Monthly_CDI_2P_March_2016.ods
  - Table_2_Monthly_CDI_2P_March_2017.ods
  - Table_2_Monthly_CDI_2P_Mar_2015.xlsx
  - Table_6_CDI_april_2023_to_march_2024-os.ods
  - Table_6_CDI_march_2020.ods
  - Table_6_CDI_march_2022.ods


In [167]:
# Voir contenu cdi_table_trust_20190425.ods
df_cdi_1718 = pd.read_excel(
    rf'{DATA}\cdi_table_trust_20190425.ods',
    engine='odf', sheet_name=None)
print("Onglets :", list(df_cdi_1718.keys()))

# Voir premier onglet
sheet = list(df_cdi_1718.keys())[0]
print(f"\nShape : {df_cdi_1718[sheet].shape}")
print(f"\nLignes 0-5 :")
print(df_cdi_1718[sheet].iloc[:5, :6].to_string())

# Voir Table_6_CDI_april_2023
df_cdi_2324 = pd.read_excel(
    rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods',
    engine='odf', sheet_name=None)
print(f"\n\nOnglets 2023-24 : {list(df_cdi_2324.keys())}")

Onglets : ['Caveats', 'monthly_table', 'data']

Shape : (0, 0)

Lignes 0-5 :
Empty DataFrame
Columns: []
Index: []


Onglets 2023-24 : ['Cover', 'monthly_table', 'data']


In [169]:
# Voir onglet 'data' pour 2017-18
df_1718_data = df_cdi_1718['data']
print(f"Shape data 2017-18 : {df_1718_data.shape}")
print(f"\nColonnes : {list(df_1718_data.columns)}")
print(f"\nOrganisation types : {df_1718_data['Organisation type'].unique() if 'Organisation type' in df_1718_data.columns else 'absent'}")
print(f"\nExemple :")
print(df_1718_data.head(3).to_string())

# Voir onglet 'data' pour 2023-24
df_2324_data = df_cdi_2324['data']
print(f"\n\nShape data 2023-24 : {df_2324_data.shape}")
print(f"Colonnes : {list(df_2324_data.columns)}")
print(f"\nOrganisation types : {df_2324_data['Organisation type'].unique() if 'Organisation type' in df_2324_data.columns else 'absent'}")

Shape data 2017-18 : (33332, 11)

Colonnes : ['Collection', 'Organisation type', 'Organisation code', 'Organisation name', 'PHE Centre', 'Year', 'Month', 'Metric', 'Figure', 'Data signed off', 'Data changed since last publication?']

Organisation types : <ArrowStringArray>
['CCG', 'National', 'NHS Acute Trust']
Length: 3, dtype: str

Exemple :
  Collection Organisation type Organisation code Organisation name  PHE Centre  Year  Month       Metric  Figure Data signed off Data changed since last publication?
0        CDI               CCG               00C        Darlington  North East  2018      9   COCA cases       0             NaN                                   No
1        CDI               CCG               00C        Darlington  North East  2018      6  Total cases       3             NaN                                   No
2        CDI               CCG               00C        Darlington  North East  2019      3  Total cases       2             NaN                            

In [171]:
# Corriger 2017-18 — filtre correct
df_1718_trusts = df_1718_data[
    df_1718_data['Organisation type'] == 'NHS Acute Trust'
]
print(f"NHS Acute Trusts 2017-18 : {df_1718_trusts['Organisation code'].nunique()}")

# Voir onglet monthly_table pour 2023-24
df_2324_monthly = df_cdi_2324['monthly_table']
print(f"\nShape monthly_table 2023-24 : {df_2324_monthly.shape}")
print(f"\nLignes 0-8 colonnes 0-5 :")
print(df_2324_monthly.iloc[:8, :6].to_string())

NHS Acute Trusts 2017-18 : 148

Shape monthly_table 2023-24 : (147, 14)

Lignes 0-8 colonnes 0-5 :
                                                                                                        UKHSA monthly pivot table of Clostridioides difficile (C. difficile) infection       Unnamed: 1 Unnamed: 2 Unnamed: 3  Unnamed: 4  Unnamed: 5
0  Pivot table showing raw monthly data of C. difficile infection for the financial year 2023 to 2024.  Table can be manipulated using filter options to streamline data for purpose.               NaN        NaN        NaN         NaN         NaN
1                                                                                                                                                   This worksheet contains one table.              NaN        NaN        NaN         NaN         NaN
2                                                                                                                                                                  

In [173]:
print("Lignes 8-15 colonnes 0-6 :")
print(df_2324_monthly.iloc[8:15, :7].to_string())

Lignes 8-15 colonnes 0-6 :
   UKHSA monthly pivot table of Clostridioides difficile (C. difficile) infection         Unnamed: 1 Unnamed: 2 Unnamed: 3  Unnamed: 4  Unnamed: 5  Unnamed: 6
8                                                                          Figure                NaN       Year      Month         NaN         NaN         NaN
9                                                                             NaN                NaN       2023        NaN         NaN         NaN         NaN
10                                                              Organisation name  Organisation code          4          5         6.0         7.0         8.0
11                                                  AIREDALE NHS FOUNDATION TRUST                RCF         10         10        10.0        17.0         9.0
12                                      ALDER HEY CHILDREN'S NHS FOUNDATION TRUST                RBS          1          0         0.0         0.0         0.0
13                 

In [177]:
def charger_cdi(path, annee):
    """
    Charge le fichier CDI et retourne :
    Trust_code, CDI_total_cases
    """
    
    if path.endswith('.ods'):
        df_sheets = pd.read_excel(path, engine='odf', sheet_name=None)
        
        if 'data' in df_sheets:
            df_data = df_sheets['data']
            
            # Vérifier si Organisation type existe
            if 'Organisation type' in df_data.columns:
                mask = df_data['Organisation type'].str.lower().str.contains('acute', na=False)
                df_trusts = df_data[mask]
                metric_col = 'Metric' if 'Metric' in df_data.columns else df_data.columns[7]
                
                df_cdi = df_trusts[
                    df_trusts[metric_col] == 'Total cases'
                ].groupby('Organisation code')['Figure'].sum().reset_index()
                df_cdi.columns = ['Trust_code', 'CDI_total_cases']
            
            # Pas de Organisation type → essayer monthly_table
            elif 'monthly_table' in df_sheets:
                df_raw = df_sheets['monthly_table']
                df_data2 = df_raw.iloc[11:, :].copy()
                df_data2.columns = range(len(df_data2.columns))
                df_data2['Trust_code'] = df_data2[1].astype(str).str.strip()
                df_data2 = df_data2[df_data2['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')]
                cols_mois = list(range(2, len(df_data2.columns)-1))
                df_data2['CDI_total_cases'] = df_data2[cols_mois].apply(
                    pd.to_numeric, errors='coerce').sum(axis=1)
                df_cdi = df_data2[['Trust_code', 'CDI_total_cases']]
            else:
                return pd.DataFrame(columns=['Trust_code', 'CDI_total_cases'])
        
        elif 'monthly_table' in df_sheets:
            df_raw = df_sheets['monthly_table']
            df_data = df_raw.iloc[11:, :].copy()
            df_data.columns = range(len(df_data.columns))
            df_data['Trust_code'] = df_data[1].astype(str).str.strip()
            df_data = df_data[df_data['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')]
            cols_mois = list(range(2, len(df_data.columns)-1))
            df_data['CDI_total_cases'] = df_data[cols_mois].apply(
                pd.to_numeric, errors='coerce').sum(axis=1)
            df_cdi = df_data[['Trust_code', 'CDI_total_cases']]
        
        else:
            sheet_name = [s for s in df_sheets.keys()
                         if 'table_2b' in s.lower() or 'table 2b' in s.lower()]
            if sheet_name:
                df_raw = df_sheets[sheet_name[0]]
                df_data = df_raw.iloc[10:, :].copy()
                df_data.columns = range(len(df_data.columns))
                df_data['Trust_code'] = df_data[0].astype(str).str.strip()
                df_data = df_data[df_data['Trust_code'].str.len() == 3]
                cols_mois = list(range(4, 16))
                df_data['CDI_total_cases'] = df_data[cols_mois].apply(
                    pd.to_numeric, errors='coerce').sum(axis=1)
                df_cdi = df_data[['Trust_code', 'CDI_total_cases']]
            else:
                return pd.DataFrame(columns=['Trust_code', 'CDI_total_cases'])
    
    else:
        if path.endswith('.xls'):
            df_sheets = pd.read_excel(path, sheet_name=None, engine='xlrd')
        else:
            df_sheets = pd.read_excel(path, sheet_name=None)
        
        sheet_name = [s for s in df_sheets.keys()
                     if 'table 2b' in s.lower() or 'table_2b' in s.lower()]
        if sheet_name:
            df_raw = df_sheets[sheet_name[0]]
            df_data = df_raw.iloc[10:, :].copy()
            df_data.columns = range(len(df_data.columns))
            df_data['Trust_code'] = df_data[0].astype(str).str.strip()
            df_data = df_data[df_data['Trust_code'].str.len() == 3]
            cols_mois = list(range(4, 16))
            df_data['CDI_total_cases'] = df_data[cols_mois].apply(
                pd.to_numeric, errors='coerce').sum(axis=1)
            df_cdi = df_data[['Trust_code', 'CDI_total_cases']]
        else:
            return pd.DataFrame(columns=['Trust_code', 'CDI_total_cases'])
    
    df_cdi = df_cdi.dropna(subset=['CDI_total_cases'])
    df_cdi['Trust_code'] = df_cdi['Trust_code'].astype(str).str.strip()
    return df_cdi[['Trust_code', 'CDI_total_cases']]

# Tests
for annee, fichier in [
    ('2017-18', rf'{DATA}\cdi_table_trust_20190425.ods'),
    ('2023-24', rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods'),
    ('2024-25', rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods'),
    ('2022-23', rf'{DATA}\Table-6-CDI-march-2023.ods'),
    ('2016-17', rf'{DATA}\Table_2_Monthly_CDI_2P_March_2017.ods'),
]:
    df_t = charger_cdi(fichier, annee)
    print(f"{annee} → {len(df_t)} trusts")

2017-18 → 148 trusts
2023-24 → 136 trusts
2024-25 → 136 trusts
2022-23 → 135 trusts
2016-17 → 154 trusts


In [179]:
# Débug SHMI 2012-13
import zipfile, os
extract_dir = rf'{DATA}\SHMI_temp'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(rf'{DATA}\SHMI data files, Apr12-Mar13.zip', 'r') as z:
    print("Fichiers dans zip 2012-13 :")
    for f in z.namelist():
        print(f"  - {f}")
    z.extractall(extract_dir)

import shutil
shutil.rmtree(extract_dir)

# Débug Providers 2021-22
df_prov_2122 = pd.read_excel(
    rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2021-22-tab.xlsx',
    sheet_name='Hospital Providers', header=None)
print(f"\nShape Providers 2021-22 : {df_prov_2122.shape}")

# Chercher header
for i in range(5, 20):
    row = df_prov_2122.iloc[i, :].tolist()
    if any('Hospital provider code' in str(v) for v in row):
        print(f"Header ligne {i}")
        break

# Voir données
print(f"\nLignes 18-22 colonnes 0-5 :")
print(df_prov_2122.iloc[18:22, :6].to_string())

Fichiers dans zip 2012-13 :
  - Machine readable file data definitions/Admission method contextual indicators data definitions.xls
  - Machine readable file data definitions/Palliative care coding contextual indicators data definitions.xls
  - Machine readable file data definitions/Deprivation contextual indicators data definitions.xls
  - Machine readable file data definitions/In and outside hospital deaths contextual indicator data definitions.xls
  - Machine readable file data definitions/SHMI data definitions.xls
  - Machine readable file data definitions/
  - SHMI contextual indicators/Palliative care coding/
  - SHMI contextual indicators/Palliative care coding/Percentage of admissions with palliative care coding, Apr12-Mar13 (csv).csv
  - SHMI contextual indicators/Palliative care coding/Percentage of admissions with palliative care coding, Apr12-Mar13 (xls).xls
  - SHMI contextual indicators/Palliative care coding/Percentage of deaths with palliative care coding, Apr12-Mar13 (c

In [181]:
# Voir colonnes SHMI 2012-13
extract_dir = rf'{DATA}\SHMI_temp'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(rf'{DATA}\SHMI data files, Apr12-Mar13.zip', 'r') as z:
    z.extractall(extract_dir)

shmi_file = rf'{extract_dir}\SHMI data\SHMI data at trust level, Apr12-Mar13 (csv).csv'
df_shmi_old = pd.read_csv(shmi_file)
print("Colonnes SHMI 2012-13 :")
for col in df_shmi_old.columns:
    print(f"  - {col}")
print(f"\nExemple :")
print(df_shmi_old.head(3).to_string())

shutil.rmtree(extract_dir)

# Voir trusts dans Providers 2021-22
print("\nLignes 22-28 colonnes 0-5 :")
print(df_prov_2122.iloc[22:28, :6].to_string())

Colonnes SHMI 2012-13 :
  - INDICATOR_CODE
  - PROVIDER
  - PROVIDER_NAME
  - DENOMINATOR
  - OBSERVED
  - EXPECTED
  - VALUE
  - PO_LL
  - PO_UL
  - OD_LL
  - OD_UL
  - OD_BANDING

Exemple :
  INDICATOR_CODE PROVIDER                                      PROVIDER_NAME  DENOMINATOR  OBSERVED   EXPECTED   VALUE   PO_LL   PO_UL   OD_LL   OD_UL  OD_BANDING
0         I00699      R1F                            ISLE OF WIGHT NHS TRUST        19404       975   868.2013  1.1230  0.8984  1.1094  0.8820  1.1338           2
1         I00699      R1H                             BARTS HEALTH NHS TRUST       160716      3609  4502.6340  0.8015  0.9546  1.0469  0.8954  1.1168           3
2         I00699      RA2  ROYAL SURREY COUNTY HOSPITAL NHS FOUNDATION TRUST        45301      1295  1370.7331  0.9447  0.9186  1.0863  0.8878  1.1263           2

Lignes 22-28 colonnes 0-5 :
     0      1                                                                                         2   3   4   5
22 NaN  8A7

In [183]:
def charger_shmi(path):
    """
    Charge le fichier SHMI (zip) et retourne :
    Trust_code, C3_mortalite, SHMI_VALUE
    Gère tous les formats (2012 → 2025)
    """
    extract_dir = rf'{DATA}\SHMI_temp'
    os.makedirs(extract_dir, exist_ok=True)
    
    with zipfile.ZipFile(path, 'r') as z:
        z.extractall(extract_dir)
        shmi_file = None
        for f in z.namelist():
            if 'trust level' in f.lower() and f.endswith('.csv'):
                shmi_file = os.path.join(extract_dir, f)
                break
        if shmi_file is None:
            for f in z.namelist():
                if 'trust' in f.lower() and f.endswith('.csv'):
                    shmi_file = os.path.join(extract_dir, f)
                    break
    
    if shmi_file is None:
        shutil.rmtree(extract_dir)
        return pd.DataFrame()
    
    df = pd.read_csv(shmi_file)
    df.columns = [c.upper().strip() for c in df.columns]
    
    # Détecter format selon colonnes disponibles
    # Format ancien (2012-2016) : PROVIDER, DENOMINATOR, VALUE
    # Format récent (2017+)     : PROVIDER_CODE, SPELLS, SHMI_VALUE
    
    if 'PROVIDER_CODE' in df.columns:
        col_code  = 'PROVIDER_CODE'
        col_value = 'SHMI_VALUE'
    else:
        col_code  = 'PROVIDER'
        col_value = 'VALUE'
    
    if 'SPELLS' in df.columns:
        col_spells = 'SPELLS'
    elif 'DENOMINATOR' in df.columns:
        col_spells = 'DENOMINATOR'
    else:
        col_spells = None
    
    df_shmi = pd.DataFrame({
        'Trust_code' : df[col_code].astype(str).str.strip(),
        'OBSERVED'   : pd.to_numeric(df['OBSERVED'], errors='coerce'),
        'SHMI_VALUE' : pd.to_numeric(df[col_value],  errors='coerce'),
    })
    
    # Calculer C3
    if col_spells and col_spells in df.columns:
        df_shmi['SPELLS'] = pd.to_numeric(df[col_spells], errors='coerce')
        df_shmi['C3_mortalite'] = round(
            df_shmi['OBSERVED'] / df_shmi['SPELLS'] * 100, 3)
    else:
        # Format récent sans SPELLS → C3 = SHMI_VALUE × 2.5%
        df_shmi['SPELLS'] = pd.to_numeric(df['EXPECTED'], errors='coerce')
        
        # Garder dernière période uniquement
        if 'TIME_PERIOD' in df.columns:
            df = df.sort_values('TIME_PERIOD')
            df_shmi = df_shmi.loc[df.groupby(col_code).tail(1).index]
        
        TAUX_NATIONAL_NHS = 2.5
        df_shmi['C3_mortalite'] = round(
            df_shmi['SHMI_VALUE'] * TAUX_NATIONAL_NHS, 3)
    
    shutil.rmtree(extract_dir)
    
    # Garder trusts NHS 3 caractères
    df_shmi = df_shmi[df_shmi['Trust_code'].str.len() == 3]
    df_shmi = df_shmi.dropna(subset=['OBSERVED'])
    
    return df_shmi[['Trust_code', 'OBSERVED', 'SPELLS', 'C3_mortalite', 'SHMI_VALUE']]

# Tests
for annee, fichier in [
    ('2012-13', rf'{DATA}\SHMI data files, Apr12-Mar13.zip'),
    ('2013-14', rf'{DATA}\SHMI data files, Apr13-Mar14.zip'),
    ('2017-18', rf'{DATA}\SHMI data files, Apr17-Mar18.zip'),
    ('2022-23', rf'{DATA}\SHMI data files, Apr22-Mar23.zip'),
]:
    df_t = charger_shmi(fichier)
    print(f"{annee} → {len(df_t)} trusts | C3 moy: {df_t['C3_mortalite'].mean():.2f}%")

2012-13 → 142 trusts | C3 moy: 3.37%
2013-14 → 141 trusts | C3 moy: 3.24%
2017-18 → 131 trusts | C3 moy: 3.48%
2022-23 → 133 trusts | C3 moy: 2.50%


In [187]:
# Voir toutes les colonnes ligne 10 (header)
print("Headers ligne 10 (2021-22) :")
for j, v in enumerate(df_prov_2122.iloc[10, :].tolist()):
    if str(v) != 'nan':
        print(f"  Col {j} : {str(v)[:50]}")

# Voir données lignes 22-30 toutes colonnes
print("\nLignes 22-30 colonnes 0-8 :")
print(df_prov_2122.iloc[22:30, :9].to_string())

Headers ligne 10 (2021-22) :
  Col 1 : Hospital provider code and description†
  Col 8 : Finished consultant episodes
  Col 9 : Admissions
  Col 10 : Male
  Col 11 : Female
  Col 12 : Gender Unknown
  Col 13 : Emergency
  Col 14 : Waiting list
  Col 15 : Planned
  Col 16 : Other Admission Method
  Col 17 : Mean time waited
  Col 18 : Median time waited
  Col 19 : Mean length of stay
  Col 20 : Median length of stay
  Col 21 : Mean age
  Col 22 : Age 0
  Col 23 : Age 1-4
  Col 24 : Age 5-9
  Col 25 : Age 10-14
  Col 26 : Age 15
  Col 27 : Age 16
  Col 28 : Age 17
  Col 29 : Age 18
  Col 30 : Age 19
  Col 31 : Age 20-24
  Col 32 : Age 25-29
  Col 33 : Age 30-34
  Col 34 : Age 35-39
  Col 35 : Age 40-44
  Col 36 : Age 45-49
  Col 37 : Age 50-54
  Col 38 : Age 55-59
  Col 39 : Age 60-64
  Col 40 : Age 65-69
  Col 41 : Age 70-74
  Col 42 : Age 75-79
  Col 43 : Age 80-84
  Col 44 : Age 85-89
  Col 45 : Age 90+
  Col 46 : Day case
  Col 47 : FCE bed days
  Col 48 : Emergency
  Col 49 : Electi

In [189]:
# Chercher trusts R** dans toutes les colonnes
print("Cherche trusts R** dans col 1 :")
count = 0
for i in range(20, 527):
    row = df_prov_2122.iloc[i, :].tolist()
    code = str(row[1])
    if len(code) == 3 and code[0] == 'R' and code[1:].isalnum():
        print(f"  Ligne {i} : code={code}, admissions={row[9]}, DMS={row[19]}, other={row[50]}")
        count += 1
        if count >= 5:
            break

print(f"\n\nTotal lignes fichier : {df_prov_2122.shape[0]}")

Cherche trusts R** dans col 1 :
  Ligne 323 : code=R0A, admissions=322880, DMS=4.09723757959355, other=14845
  Ligne 324 : code=R0B, admissions=143475, DMS=3.93935730347946, other=1560
  Ligne 325 : code=R0D, admissions=168990, DMS=3.89705335655168, other=1800
  Ligne 326 : code=R1A, admissions=6725, DMS=18.0213846153846, other=360
  Ligne 327 : code=R1C, admissions=1130, DMS=23.4940962761126, other=*


Total lignes fichier : 527


In [191]:
def charger_providers(path):
    """
    Charge le fichier Hospital Providers et retourne :
    Trust_code, total_admissions, A2_dms_jours, A9_taux_transfert
    """
    xl = pd.ExcelFile(path)
    sheet_name = None
    for s in xl.sheet_names:
        if 'provider' in s.lower() or 'hospital' in s.lower():
            sheet_name = s
            break
    if sheet_name is None:
        sheet_name = xl.sheet_names[0]

    df_raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    # Chercher ligne header
    header_row = None
    for i in range(5, 25):
        row = df_raw.iloc[i, :].tolist()
        if any('Hospital provider code' in str(v) for v in row):
            header_row = i
            break
    if header_row is None:
        header_row = 9

    # Trouver col du Trust code dans le header
    col_code = 0
    for j, v in enumerate(df_raw.iloc[header_row, :].tolist()):
        if 'Hospital provider code' in str(v):
            col_code = j
            break

    df_data = df_raw.iloc[header_row+1:, :].copy()
    df_data.columns = range(len(df_data.columns))
    nb_cols = len(df_data.columns)

    print(f"  Onglet: {sheet_name} | Colonnes: {nb_cols} | Header: {header_row} | col_code: {col_code}")

    # ── Format récent (50+ colonnes) 2017 → 2025 ──
    if nb_cols >= 50:
        df_prov = pd.DataFrame({
            'Trust_code'      : df_data[col_code].astype(str).str.strip(),
            'total_admissions': pd.to_numeric(df_data[col_code+8 if col_code==0 else 9],  errors='coerce'),
            'A2_dms_jours'    : pd.to_numeric(df_data[col_code+18 if col_code==0 else 19], errors='coerce'),
            'other_discharge' : pd.to_numeric(df_data[col_code+49 if col_code==0 else 50], errors='coerce'),
        })
    # ── Format ancien (42-45 colonnes) 2012 → 2016 ──
    else:
        df_prov = pd.DataFrame({
            'Trust_code'      : df_data[col_code].astype(str).str.strip(),
            'total_admissions': pd.to_numeric(df_data[col_code+3],  errors='coerce'),
            'A2_dms_jours'    : pd.to_numeric(df_data[col_code+13], errors='coerce'),
            'other_discharge' : pd.to_numeric(df_data[col_code+10], errors='coerce'),
        })

    # Garder trusts NHS (code R** 3 caractères)
    df_prov = df_prov[
        df_prov['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['total_admissions', 'A2_dms_jours'])

    df_prov['A9_taux_transfert'] = round(
        df_prov['other_discharge'] / df_prov['total_admissions'] * 100, 2
    )

    return df_prov[['Trust_code', 'total_admissions',
                    'A2_dms_jours', 'A9_taux_transfert']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2024-25-tab.xlsx'),
    ('2022-23', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2022-23-tab_v2.xlsx'),
    ('2021-22', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2021-22-tab.xlsx'),
    ('2017-18', rf'{DATA}\hosp-epis-stat-admi-prov-2017-18-tab.xlsx'),
    ('2016-17', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2016-17-tab.xlsx'),
    ('2012-13', rf'{DATA}\hosp-epis-stat-admi-hosp-prov-2012-13-tab.xlsx'),
]:
    df_t = charger_providers(fichier)
    print(f"{annee} → {len(df_t)} trusts | DMS moy: {df_t['A2_dms_jours'].mean():.1f}j\n")

  Onglet: Hospital Providers | Colonnes: 50 | Header: 14 | col_code: 0
2024-25 → 153 trusts | DMS moy: 16.7j

  Onglet: Hospital Providers | Colonnes: 50 | Header: 16 | col_code: 0
2022-23 → 160 trusts | DMS moy: 16.5j

  Onglet: Hospital Providers | Colonnes: 51 | Header: 10 | col_code: 1
2021-22 → 166 trusts | DMS moy: 15.2j

  Onglet: Hospital Providers | Colonnes: 50 | Header: 9 | col_code: 0
2017-18 → 185 trusts | DMS moy: 15.9j

  Onglet: Hospital Providers | Colonnes: 50 | Header: 9 | col_code: 0
2016-17 → 189 trusts | DMS moy: 17.7j

  Onglet: Hospital providers | Colonnes: 42 | Header: 16 | col_code: 0
2012-13 → 204 trusts | DMS moy: 27.7j



In [193]:
# Débug Workforce 2021-22
xl = pd.ExcelFile(rf'{DATA}\NHS Workforce Statistics, March 2022 England and Organisation.xlsx')
print(f"Onglets 2021-22 : {xl.sheet_names}")

# Voir onglet organisation
sheet = [s for s in xl.sheet_names if 'Org' in s and 'HC' in s][0]
print(f"\nOnglet utilisé : {sheet}")
df_raw = pd.read_excel(
    rf'{DATA}\NHS Workforce Statistics, March 2022 England and Organisation.xlsx',
    sheet_name=sheet, header=None)
print(f"Shape : {df_raw.shape}")
print(f"\nLignes 4-9 colonnes 0-8 :")
print(df_raw.iloc[4:9, :9].to_string())

Onglets 2021-22 : ['Title sheet', '1. England', '2. NHSE, Org & SG - HC', '3. NHSE, Org & SG - FTE', '4. All Staff, NHSE & Org - HC', '5. All Staff, NHSE & Org - FTE']

Onglet utilisé : 2. NHSE, Org & SG - HC
Shape : (342, 31)

Lignes 4-9 colonnes 0-8 :
                         0        1    2    3        4                                        5             6           7                     8
4                      NaN      NaN  NaN  NaN      NaN                                      NaN           NaN         NaN                   NaN
5  March 2022 monthly data      NaN  NaN  NaN      NaN                                      NaN           NaN         NaN                   NaN
6                      NaN      NaN  NaN  NaN    Total  Professionally qualified clinical staff  HCHS Doctors  Consultant  Associate Specialist
7                      NaN      NaN  NaN  NaN      NaN                                      NaN           NaN         NaN                   NaN
8                      NaN

In [195]:
print("Ligne 6 (headers) 2021-22 colonnes 0-25 :")
for j, v in enumerate(df_raw.iloc[6, :].tolist()):
    if str(v) != 'nan':
        print(f"  Col {j} : {str(v)[:50]}")

print("\nLignes 10-14 colonnes 0-8 :")
print(df_raw.iloc[10:14, :9].to_string())

Ligne 6 (headers) 2021-22 colonnes 0-25 :
  Col 4 : Total
  Col 5 : Professionally qualified clinical staff
  Col 6 : HCHS Doctors
  Col 7 : Consultant
  Col 8 : Associate Specialist
  Col 9 : Specialty Doctor
  Col 10 : Staff Grade
  Col 11 : Specialty Registrar
  Col 12 : Core Training
  Col 13 : Foundation Doctor Year 2
  Col 14 : Foundation Doctor Year 1
  Col 15 : Hospital Practitioner / Clinical Assistant
  Col 16 : Other and Local HCHS Doctor Grades
  Col 17 : Nurses & health visitors
  Col 18 : Midwives
  Col 19 : Ambulance staff
  Col 20 : Scientific, therapeutic & technical staff
  Col 21 : Support to clinical staff
  Col 22 : Support to doctors, nurses & midwives
  Col 23 : Support to ambulance staff
  Col 24 : Support to ST&T staff
  Col 25 : NHS infrastructure support
  Col 26 : Central functions
  Col 27 : Hotel, property & estates
  Col 28 : Senior managers
  Col 29 : Managers
  Col 30 : Other staff or those with unknown classification

Lignes 10-14 colonnes 0-8 :
      

In [197]:
def charger_workforce(path):
    """
    Charge le fichier NHS Workforce et retourne :
    Trust_code, doctors, nurses
    Gère tous les formats (2013 → 2025)
    """
    xl = pd.ExcelFile(path)
    sheets = xl.sheet_names

    # ── Format récent (2022+) — 33 cols ──
    if any('NHSE, Org & SG - HC' in s for s in sheets):
        sheet = [s for s in sheets if 'NHSE, Org & SG - HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        nb_cols = df_raw.shape[1]
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))

        if nb_cols <= 31:
            # 2021-22 : 31 cols → col 3=code, col 6=doctors, col 17=nurses
            df_wf = pd.DataFrame({
                'Trust_code': df_data[3].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[6],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[17], errors='coerce'),
            })
        elif nb_cols <= 33:
            # 2022+ : 33 cols → col 5=code, col 8=doctors, col 19=nurses
            df_wf = pd.DataFrame({
                'Trust_code': df_data[5].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[19], errors='coerce'),
            })
        else:
            # 2019-20 : 37 cols → col 3=code, col 8=doctors, col 20=nurses
            df_wf = pd.DataFrame({
                'Trust_code': df_data[3].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
            })

    # ── Format 2018-19 — onglet 'HEE, Org & SG - HC' ──
    elif any('HEE, Org & SG - HC' in s for s in sheets):
        sheet = [s for s in sheets if 'HEE, Org & SG - HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        df_data = df_raw.iloc[7:, :].copy()
        df_data.columns = range(len(df_data.columns))
        df_wf = pd.DataFrame({
            'Trust_code': df_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
            'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
        })

    # ── Format 2015-2018 — onglet 'HEE Org Main Staff Gp HC' ──
    elif any('HEE Org Main' in s for s in sheets):
        sheet = [s for s in sheets if 'HEE Org Main' in s and 'HC' in s][0]
        df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
        nb_cols = df_raw.shape[1]
        df_data = df_raw.iloc[8:, :].copy()
        df_data.columns = range(len(df_data.columns))
        if nb_cols >= 38:
            df_wf = pd.DataFrame({
                'Trust_code': df_data[3].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[21], errors='coerce'),
            })
        elif nb_cols >= 37:
            df_wf = pd.DataFrame({
                'Trust_code': df_data[3].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
            })
        else:
            df_wf = pd.DataFrame({
                'Trust_code': df_data[2].astype(str).str.strip(),
                'doctors'   : pd.to_numeric(df_data[8],  errors='coerce'),
                'nurses'    : pd.to_numeric(df_data[20], errors='coerce'),
            })

    # ── Format ancien (2013-2015) — onglets séparés ──
    else:
        sheet_doc = [s for s in sheets if 'Drs' in s and 'HC' in s][0]
        df_doc = pd.read_excel(path, sheet_name=sheet_doc, header=None)
        df_doc_data = df_doc.iloc[7:, :].copy()
        df_doc_data.columns = range(len(df_doc_data.columns))
        df_doctors = pd.DataFrame({
            'Trust_code': df_doc_data[3].astype(str).str.strip(),
            'doctors'   : pd.to_numeric(df_doc_data[4], errors='coerce'),
        })
        sheet_nur = [s for s in sheets if 'NonMed' in s][0]
        df_nur = pd.read_excel(path, sheet_name=sheet_nur, header=None)
        df_nur_data = df_nur.iloc[7:, :].copy()
        df_nur_data.columns = range(len(df_nur_data.columns))
        df_nurses = pd.DataFrame({
            'Trust_code': df_nur_data[3].astype(str).str.strip(),
            'nurses'    : pd.to_numeric(df_nur_data[4], errors='coerce'),
        })
        df_wf = df_doctors.merge(df_nurses, on='Trust_code', how='outer')

    # Garder trusts NHS
    df_wf = df_wf[
        df_wf['Trust_code'].str.match(r'^R[A-Z0-9]{2}$')
    ].dropna(subset=['doctors', 'nurses'])

    return df_wf[['Trust_code', 'doctors', 'nurses']]

# Tests toutes années
for annee, fichier in [
    ('2024-25', rf'{DATA}\NHS Workforce Statistics, March 2025 England and Organisation.xlsx'),
    ('2022-23', rf'{DATA}\NHS Workforce Statistics, March 2023 England and Organisation.xlsx'),
    ('2021-22', rf'{DATA}\NHS Workforce Statistics, March 2022 England and Organisation.xlsx'),
    ('2019-20', rf'{DATA}\NHS Workforce Statistics, March 2020 England and Organisation.xlsx'),
    ('2018-19', rf'{DATA}\NHS Workforce Statistics, March 2019 England and Organisation.xlsx'),
    ('2017-18', rf'{DATA}\NHS Workforce Statistics, March 2018 Organisation - Excel tables.xlsx'),
    ('2016-17', rf'{DATA}\nhs-work-stat-mar-2017-org-tab.xlsx'),
    ('2015-16', rf'{DATA}\nhs-work-stat-mar-2016-org-tab.xlsx'),
    ('2014-15', rf'{DATA}\nhs-work-stat-mar-2015-org-tab.xls'),
    ('2013-14', rf'{DATA}\nhs-work-stat-mar-2014-org-tab.xls'),
]:
    df_t = charger_workforce(fichier)
    print(f"{annee} → {len(df_t)} trusts | Médecins: {df_t['doctors'].mean():.0f} | Infirmiers: {df_t['nurses'].mean():.0f}")

2024-25 → 201 trusts | Médecins: 783 | Infirmiers: 1995
2022-23 → 208 trusts | Médecins: 683 | Infirmiers: 1755
2021-22 → 209 trusts | Médecins: 650 | Infirmiers: 1686
2019-20 → 214 trusts | Médecins: 588 | Infirmiers: 1552
2018-19 → 222 trusts | Médecins: 535 | Infirmiers: 1442
2017-18 → 227 trusts | Médecins: 511 | Infirmiers: 1390
2016-17 → 229 trusts | Médecins: 493 | Infirmiers: 1378
2015-16 → 229 trusts | Médecins: 481 | Infirmiers: 1373
2014-15 → 227 trusts | Médecins: 484 | Infirmiers: 4426
2013-14 → 233 trusts | Médecins: 463 | Infirmiers: 4261


In [199]:
# Mettre à jour les chemins CDI corrects
ANNEES_PATHS['2017-18']['cdi'] = rf'{DATA}\cdi_table_trust_20190425.ods'
ANNEES_PATHS['2018-19']['cdi'] = rf'{DATA}\Table_2_Monthly_CDI_2P_April_2018.ods'
ANNEES_PATHS['2023-24']['cdi'] = rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods'
ANNEES_PATHS['2024-25']['cdi'] = rf'{DATA}\Table_6_CDI_april_2023_to_march_2024-os.ods'

# Aussi corriger AEQI 2023-24
ANNEES_PATHS['2023-24']['aeqi'] = rf'{DATA}\aeqi_open_data_2023_03.csv'

# Relancer toutes les années
dfs_annees = []
erreurs = []

for annee, paths in ANNEES_PATHS.items():
    try:
        df_annee = charger_annee(annee, paths)
        if len(df_annee) > 0:
            dfs_annees.append(df_annee)
            print(f"✅ {annee} → {len(df_annee)} trusts")
        else:
            print(f"⚠️ {annee} → 0 trusts (ignoré)")
            erreurs.append(annee)
    except Exception as e:
        print(f"❌ {annee} : {e}")
        erreurs.append(annee)

print(f"\n{'='*50}")
print(f"Total années : {len(dfs_annees)}")
print(f"Erreurs      : {erreurs}")


Chargement année 2012-13...
✅ Beds      : 229 trusts
✅ SHMI      : 142 trusts
✅ CDI       : 160 trusts
  Onglet: Hospital providers | Colonnes: 42 | Header: 16 | col_code: 0
✅ Providers : 204 trusts
  prov-ae-qual-indi-eng-mar13-tab1.xls → 62 colonnes
  col_code=5, col_median=59, header=9
✅ AEQI      : 170 trusts
✅ Workforce : 233 trusts

✅ Fusion terminée : 122 trusts × 20 colonnes
✅ 2012-13 → 122 trusts

Chargement année 2013-14...
✅ Beds      : 227 trusts
✅ SHMI      : 141 trusts
✅ CDI       : 160 trusts
  Onglet: Hospital Providers | Colonnes: 45 | Header: 18 | col_code: 0
✅ Providers : 206 trusts
  prov-ae-qual-indi-eng-march 2014-tab1.xls → 53 colonnes
  col_code=2, col_median=50, header=5
✅ AEQI      : 169 trusts
✅ Workforce : 233 trusts

✅ Fusion terminée : 122 trusts × 20 colonnes
✅ 2013-14 → 122 trusts

Chargement année 2014-15...
✅ Beds      : 222 trusts
✅ SHMI      : 137 trusts
✅ CDI       : 158 trusts
  Onglet: Hospital Providers | Colonnes: 50 | Header: 9 | col_code: 0
✅

In [201]:
# ============================================================
# SECTION 10 — FUSION FINALE ET STATISTIQUES
# ============================================================

# Fusionner toutes les années
df_final = pd.concat(dfs_annees, ignore_index=True)

print(f"{'='*50}")
print(f"DATASET FINAL")
print(f"{'='*50}")
print(f"Total lignes  : {len(df_final)}")
print(f"Total colonnes: {len(df_final.columns)}")
print(f"\nDistribution par année :")
print(df_final['annee'].value_counts().sort_index())

print(f"\nStatistiques descriptives :")
print(df_final[['C2_occupation', 'C3_mortalite', 'A2_dms_jours',
                'A4_temps_attente_min', 'A8_lits_vacants',
                'A9_taux_transfert']].describe().round(2).to_string())

DATASET FINAL
Total lignes  : 1341
Total colonnes: 20

Distribution par année :
annee
2012-13    122
2013-14    122
2014-15    119
2015-16    118
2016-17    118
2017-18    108
2018-19    110
2019-20    105
2021-22    109
2022-23    104
2023-24    104
2024-25    102
Name: count, dtype: int64

Statistiques descriptives :
       C2_occupation  C3_mortalite  A2_dms_jours  A4_temps_attente_min  A8_lits_vacants  A9_taux_transfert
count        1341.00       1341.00       1341.00               1341.00          1341.00            1336.00
mean           88.90          3.09          4.21                173.45            11.10               5.64
std             5.22          0.73          1.08                 43.43             5.22               5.27
min            61.30          0.99          1.00                 37.00             0.00               0.00
25%            85.70          2.54          3.70                149.00             7.20               1.66
50%            89.50          3.01   

In [203]:
# ============================================================
# SECTION 11 — NETTOYAGE ET CALCUL DES LABELS
# ============================================================

print("AVANT nettoyage :")
print(f"  Lignes : {len(df_final)}")
print(f"  Valeurs manquantes :\n{df_final.isnull().sum()}")

# ── 1. Supprimer doublons ──
df_clean = df_final.drop_duplicates(subset=['Trust_code', 'annee'])
print(f"\nAprès suppression doublons : {len(df_clean)} lignes")

# ── 2. Supprimer valeurs aberrantes ──
# DMS > 30 jours = aberrant (hôpitaux spécialisés)
df_clean = df_clean[df_clean['A2_dms_jours'] <= 30]
# C2 > 100% impossible
df_clean = df_clean[df_clean['C2_occupation'] <= 100]
# A4 > 600 min = aberrant
df_clean = df_clean[df_clean['A4_temps_attente_min'] <= 600]
# C3 > 15% = aberrant
df_clean = df_clean[df_clean['C3_mortalite'] <= 15]

print(f"Après suppression aberrants : {len(df_clean)} lignes")

# ── 3. Remplacer NaN par médiane ──
cols_numeriques = ['C2_occupation', 'C3_mortalite', 'A2_dms_jours',
                   'A4_temps_attente_min', 'A8_lits_vacants',
                   'A9_taux_transfert', 'CDI_total_cases',
                   'Ratio_Medecin_Lit', 'Ratio_Infirmier_Lit',
                   'Ratio_Infirmier_Medecin']

for col in cols_numeriques:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

print(f"\nAprès remplacement NaN : {df_clean.isnull().sum().sum()} valeurs manquantes")

# ── 4. Calcul C4 (taux infections) ──
df_clean['C4_infection'] = round(
    df_clean['CDI_total_cases'] / df_clean['total_admissions'] * 100, 4
)

print(f"\n✅ Nettoyage terminé !")
print(f"   Lignes finales : {len(df_clean)}")
print(f"\nStatistiques après nettoyage :")
print(df_clean[['C2_occupation', 'C3_mortalite', 'C4_infection',
                'A2_dms_jours', 'A4_temps_attente_min']].describe().round(3))

AVANT nettoyage :
  Lignes : 1341
  Valeurs manquantes :
Trust_code                 0
lits_disponibles           0
lits_occupes               0
C2_occupation              0
A8_lits_vacants            0
OBSERVED                   0
SPELLS                     0
C3_mortalite               0
SHMI_VALUE                 0
CDI_total_cases            0
total_admissions           0
A2_dms_jours               0
A9_taux_transfert          5
A4_temps_attente_min       0
doctors                    0
nurses                     0
annee                      0
Ratio_Medecin_Lit          0
Ratio_Infirmier_Lit        0
Ratio_Infirmier_Medecin    0
dtype: int64

Après suppression doublons : 1341 lignes
Après suppression aberrants : 1339 lignes

Après remplacement NaN : 0 valeurs manquantes

✅ Nettoyage terminé !
   Lignes finales : 1339

Statistiques après nettoyage :
       C2_occupation  C3_mortalite  C4_infection  A2_dms_jours  \
count       1339.000      1339.000      1339.000      1339.000   
mean   

In [205]:
# ── 5. Nettoyer C4 aberrant ──
print(f"C4 avant nettoyage :")
print(f"  Max : {df_clean['C4_infection'].max():.3f}%")
print(f"  Trusts avec C4 > 1% : {(df_clean['C4_infection'] > 1).sum()}")

# C4 normal NHS = 0.01% à 0.5%
# Supprimer C4 > 2% (aberrant)
df_clean = df_clean[df_clean['C4_infection'] <= 2]
print(f"\nAprès suppression C4 aberrant : {len(df_clean)} lignes")

# ── 6. Calcul des LABELS ──
# Seuils basés sur NHS et littérature
def calculer_label(row):
    score = 0
    
    # C2 occupation (OECD 2023)
    if row['C2_occupation'] > 92:   score += 2
    elif row['C2_occupation'] > 85: score += 1
    
    # C3 mortalité (WHO 2015)
    if row['C3_mortalite'] > 4:   score += 2
    elif row['C3_mortalite'] > 2: score += 1
    
    # A2 DMS (NHS HES 2023)
    if row['A2_dms_jours'] > 7:   score += 2
    elif row['A2_dms_jours'] > 4.5: score += 1
    
    # A4 temps attente (NHS Constitution)
    if row['A4_temps_attente_min'] > 240:  score += 2
    elif row['A4_temps_attente_min'] > 120: score += 1
    
    # A8 lits vacants (OECD 2023)
    if row['A8_lits_vacants'] < 8:    score += 2
    elif row['A8_lits_vacants'] < 15: score += 1
    
    # C4 infections (ECDC 2013)
    if row['C4_infection'] > 0.3:   score += 2
    elif row['C4_infection'] > 0.1: score += 1
    
    # A9 transferts (NHS Hospital Providers)
    if row['A9_taux_transfert'] > 5:  score += 2
    elif row['A9_taux_transfert'] > 2: score += 1
    
    # Label basé sur score total (max=14)
    if score >= 8:   return 'Critique'
    elif score >= 4: return 'Attention'
    else:            return 'Normal'

df_clean['label'] = df_clean.apply(calculer_label, axis=1)

print(f"\nDistribution des labels :")
print(df_clean['label'].value_counts())
print(f"\nPourcentages :")
print((df_clean['label'].value_counts() / len(df_clean) * 100).round(1))

C4 avant nettoyage :
  Max : 624.444%
  Trusts avec C4 > 1% : 1

Après suppression C4 aberrant : 1338 lignes

Distribution des labels :
label
Attention    922
Critique     258
Normal       158
Name: count, dtype: int64

Pourcentages :
label
Attention    68.9
Critique     19.3
Normal       11.8
Name: count, dtype: float64


In [207]:
# ============================================================
# SECTION 12 — VALIDATION ET EXPORT FINAL
# ============================================================

# ── Sélectionner les colonnes finales ──
cols_finales = [
    'Trust_code', 'annee',
    # KPIs
    'C2_occupation', 'C3_mortalite', 'C4_infection',
    'A2_dms_jours', 'A4_temps_attente_min',
    'A8_lits_vacants', 'A9_taux_transfert',
    # Features supplémentaires
    'lits_disponibles', 'total_admissions',
    'doctors', 'nurses',
    'Ratio_Medecin_Lit', 'Ratio_Infirmier_Lit', 'Ratio_Infirmier_Medecin',
    # Label
    'label'
]

df_export = df_clean[cols_finales].copy()

print("✅ VALIDATION FINALE")
print(f"{'='*50}")
print(f"Lignes         : {len(df_export)}")
print(f"Colonnes       : {len(df_export.columns)}")
print(f"Années         : {sorted(df_export['annee'].unique())}")
print(f"Trusts uniques : {df_export['Trust_code'].nunique()}")
print(f"\nValeurs manquantes : {df_export.isnull().sum().sum()}")
print(f"\nDistribution labels :")
print(df_export['label'].value_counts())
print(f"\nDistribution par année :")
print(df_export.groupby('annee')['label'].value_counts().unstack().fillna(0).astype(int))

# Export
import os
os.makedirs(OUTPUT, exist_ok=True)
output_path = rf'{OUTPUT}\nhs_dataset_final.csv'
df_export.to_csv(output_path, index=False)
print(f"\n✅ Dataset exporté : {output_path}")
print(f"   Taille : {os.path.getsize(output_path) / 1024:.1f} KB")

# Aperçu final
print(f"\nAperçu 3 premières lignes :")
print(df_export.head(3).to_string())

✅ VALIDATION FINALE
Lignes         : 1338
Colonnes       : 17
Années         : ['2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2021-22', '2022-23', '2023-24', '2024-25']
Trusts uniques : 129

Valeurs manquantes : 0

Distribution labels :
label
Attention    922
Critique     258
Normal       158
Name: count, dtype: int64

Distribution par année :
label    Attention  Critique  Normal
annee                               
2012-13         86        30       6
2013-14         83        30       9
2014-15         85        19      15
2015-16         81        21      16
2016-17         77        20      21
2017-18         74        24      10
2018-19         78        14      18
2019-20         71         6      27
2021-22         74        14      19
2022-23         69        27       8
2023-24         72        28       4
2024-25         72        25       5

✅ Dataset exporté : C:\Users\hp\healthsim-simulation\data\datasets\nhs_dataset_final.csv
   

In [209]:
# ============================================================
# RENOMMAGE DES COLONNES — NOMS LISIBLES ET COMPATIBLES APP
# ============================================================

df_export_final = df_export.copy()

# Renommer toutes les colonnes
df_export_final = df_export_final.rename(columns={
    'Trust_code'             : 'code_hopital',
    'annee'                  : 'annee_fiscale',
    'C2_occupation'          : 'taux_occupation_lits',
    'C3_mortalite'           : 'taux_mortalite',
    'C4_infection'           : 'taux_infection_nosocomiale',
    'A2_dms_jours'           : 'duree_moyenne_sejour_jours',
    'A4_temps_attente_min'   : 'temps_attente_median_min',
    'A8_lits_vacants'        : 'taux_lits_vacants',
    'A9_taux_transfert'      : 'taux_transfert',
    'lits_disponibles'       : 'nombre_lits_disponibles',
    'total_admissions'       : 'total_admissions',
    'doctors'                : 'nombre_medecins',
    'nurses'                 : 'nombre_infirmiers',
    'Ratio_Medecin_Lit'      : 'ratio_medecin_par_lit',
    'Ratio_Infirmier_Lit'    : 'ratio_infirmier_par_lit',
    'Ratio_Infirmier_Medecin': 'ratio_infirmier_par_medecin',
    'label'                  : 'statut_hopital',
})

print("Colonnes finales :")
for col in df_export_final.columns:
    print(f"  - {col}")

print(f"\nExemple ligne :")
print(df_export_final.head(1).T.to_string())

# Export final
df_export_final.to_csv(output_path, index=False)
print(f"\n✅ Dataset exporté avec nouveaux noms !")
print(f"   Lignes : {len(df_export_final)}")

Colonnes finales :
  - code_hopital
  - annee_fiscale
  - taux_occupation_lits
  - taux_mortalite
  - taux_infection_nosocomiale
  - duree_moyenne_sejour_jours
  - temps_attente_median_min
  - taux_lits_vacants
  - taux_transfert
  - nombre_lits_disponibles
  - total_admissions
  - nombre_medecins
  - nombre_infirmiers
  - ratio_medecin_par_lit
  - ratio_infirmier_par_lit
  - ratio_infirmier_par_medecin
  - statut_hopital

Exemple ligne :
                                     0
code_hopital                       RE9
annee_fiscale                  2012-13
taux_occupation_lits              86.0
taux_mortalite                   4.764
taux_infection_nosocomiale      0.0188
duree_moyenne_sejour_jours         5.4
temps_attente_median_min          72.0
taux_lits_vacants                 14.0
taux_transfert                    14.4
nombre_lits_disponibles          415.0
total_admissions               37221.0
nombre_medecins                  214.0
nombre_infirmiers               4331.0
ratio_medec

In [211]:
# Vérifier les ratios aberrants
print("Distribution ratio_infirmier_par_lit :")
print(df_export_final['ratio_infirmier_par_lit'].describe().round(3))

print("\nTrusts avec ratio_infirmier_par_lit > 5 :")
print(df_export_final[df_export_final['ratio_infirmier_par_lit'] > 5][
    ['code_hopital', 'annee_fiscale', 'nombre_infirmiers', 
     'nombre_lits_disponibles', 'ratio_infirmier_par_lit']
].head(10).to_string())

Distribution ratio_infirmier_par_lit :
count    1338.000
mean        3.501
std         2.070
min         1.255
25%         2.064
50%         2.553
75%         5.198
max        12.339
Name: ratio_infirmier_par_lit, dtype: float64

Trusts avec ratio_infirmier_par_lit > 5 :
  code_hopital annee_fiscale  nombre_infirmiers  nombre_lits_disponibles  ratio_infirmier_par_lit
0          RE9       2012-13             4331.0               415.000000                   10.436
1          RLN       2012-13             4506.0               887.122222                    5.079
2          RTD       2012-13            11867.0              1608.644444                    7.377
3          RTF       2012-13             7860.0              1203.522222                    6.531
4          RTR       2012-13             8222.0              1154.322222                    7.123
5          RVW       2012-13             4893.0               669.488889                    7.309
6          RXP       2012-13             7

In [213]:
# Vérifier par année
print("Infirmiers moyens par année :")
print(df_export_final.groupby('annee_fiscale')['nombre_infirmiers'].mean().round(0))

Infirmiers moyens par année :
annee_fiscale
2012-13    4874.0
2013-14    4898.0
2014-15    5114.0
2015-16    1625.0
2016-17    1634.0
2017-18    1693.0
2018-19    1763.0
2019-20    1951.0
2021-22    2119.0
2022-23    2193.0
2023-24    2351.0
2024-25    2445.0
Name: nombre_infirmiers, dtype: float64


In [215]:
# Mettre infirmiers NaN pour 2012-13, 2013-14, 2014-15
annees_nurses_invalides = ['2012-13', '2013-14', '2014-15']

mask = df_export_final['annee_fiscale'].isin(annees_nurses_invalides)

df_export_final.loc[mask, 'nombre_infirmiers']        = np.nan
df_export_final.loc[mask, 'ratio_infirmier_par_lit']  = np.nan
df_export_final.loc[mask, 'ratio_infirmier_par_medecin'] = np.nan

print(f"Valeurs manquantes par colonne :")
print(df_export_final.isnull().sum())

print(f"\nInfirmiers moyens par année :")
print(df_export_final.groupby('annee_fiscale')['nombre_infirmiers'].mean().round(0))

# Exporter
df_export_final.to_csv(output_path, index=False)
print(f"\n✅ Dataset exporté !")
print(f"   Lignes : {len(df_export_final)}")
print(f"   Taille : {os.path.getsize(output_path)/1024:.1f} KB")

Valeurs manquantes par colonne :
code_hopital                     0
annee_fiscale                    0
taux_occupation_lits             0
taux_mortalite                   0
taux_infection_nosocomiale       0
duree_moyenne_sejour_jours       0
temps_attente_median_min         0
taux_lits_vacants                0
taux_transfert                   0
nombre_lits_disponibles          0
total_admissions                 0
nombre_medecins                  0
nombre_infirmiers              363
ratio_medecin_par_lit            0
ratio_infirmier_par_lit        363
ratio_infirmier_par_medecin    363
statut_hopital                   0
dtype: int64

Infirmiers moyens par année :
annee_fiscale
2012-13       NaN
2013-14       NaN
2014-15       NaN
2015-16    1625.0
2016-17    1634.0
2017-18    1693.0
2018-19    1763.0
2019-20    1951.0
2021-22    2119.0
2022-23    2193.0
2023-24    2351.0
2024-25    2445.0
Name: nombre_infirmiers, dtype: float64

✅ Dataset exporté !
   Lignes : 1338
   Taille : 156.2 KB